In [ ]:
import cv2
import imageio.v2 as iio
import numpy as np
import pandas as pd
import json
import math
import subprocess
import warnings
import shutil
from IPython.display import Video, display
from pathlib import Path
from google.colab import files

In [ ]:
from pathlib import Path
setup_dir = Path("/content/seganymo-setup")
setup_dir.mkdir(parents=True, exist_ok=True)
payload = {'setup.sh': '#!/usr/bin/env bash\n# Run on a Google Colab Linux x86_64 GPU runtime.\nset -euo pipefail\ntrap \'echo "ERROR: setup stopped at line $LINENO. Fix the error above, then rerun this cell." >&2\' ERR\n\nSCRIPT_DIR="$(cd -- "$(dirname -- "${BASH_SOURCE[0]}")" && pwd)"\nREPO_DIR="/content/SegAnyMo"\nVENV_DIR="/content/seganymo-venv"\nCOMMIT="6fe8fe2874ccb47169d87d78ea3b2a59e203e5ac"\nPYTHON_VERSION="3.12.11"\nTOOL_DIR="/content/seganymo-tools"\nexport UV_CACHE_DIR="/content/seganymo-uv-cache"\nexport UV_PYTHON_INSTALL_DIR="/content/seganymo-python"\nexport UV_PYTHON_BIN_DIR="/content/seganymo-python-bin"\nexport UV_NO_PROGRESS=1\nexport PYTHONNOUSERSITE=1\n\nif [[ "$(uname -s)" != Linux || "$(uname -m)" != x86_64 || ! -d /content ]]; then\n    echo \'This notebook targets Google Colab Linux x86_64 GPU runtimes.\' >&2\n    exit 1\nfi\nif ! nvidia-smi --query-gpu=name --format=csv,noheader; then\n    echo \'Select Runtime > Change runtime type > GPU, then rerun.\' >&2\n    exit 1\nfi\npython3 - <<\'PY\'\nimport shutil\nfree = shutil.disk_usage(\'/content\').free / 2**30\nprint(f\'Free disk: {free:.1f} GiB\')\nif free < 18:\n    raise SystemExit(\'Need at least 18 GiB free for installation and checkpoints. Start a fresh runtime.\')\nPY\n\n# Apply URL rewriting to every Git invocation, including recursive submodules.\n# Per-command options also recover a clone interrupted by the old SSH failure.\ngit_https() {\n    git -c \'url.https://github.com/.insteadOf=git@github.com:\' \\\n        -c \'url.https://github.com/.insteadOf=ssh://git@github.com/\' "$@"\n}\nif [[ ! -d "$REPO_DIR/.git" ]]; then\n    if [[ -e "$REPO_DIR" ]]; then\n        echo "ERROR: $REPO_DIR exists without .git. Use a fresh runtime or move it aside." >&2\n        exit 1\n    fi\n    git_https clone --no-recurse-submodules https://github.com/nnanhuang/SegAnyMo.git "$REPO_DIR"\nfi\ncd "$REPO_DIR"\nif ! git cat-file -e "${COMMIT}^{commit}" 2>/dev/null; then\n    git_https fetch origin "$COMMIT"\nfi\ngit checkout --detach "$COMMIT"\ngit_https submodule sync --recursive\ngit_https submodule update --init --recursive\ntest "$(git rev-parse HEAD)" = "$COMMIT"\n\n# uv is a small, separately installed bootstrap tool. It obtains the same\n# Python release on each runtime, without depending on Colab\'s apt sources.\nif [[ ! -x "$TOOL_DIR/bin/uv" ]] || [[ "$("$TOOL_DIR/bin/uv" --version)" != \'uv 0.8.15 \'* ]]; then\n    python3 -m pip install --disable-pip-version-check --no-cache-dir \\\n        --no-deps --upgrade --target "$TOOL_DIR" uv==0.8.15\nfi\nUV="$TOOL_DIR/bin/uv"\n"$UV" python install "$PYTHON_VERSION"\nif [[ ! -e "$VENV_DIR" ]]; then\n    "$UV" venv --managed-python --python "$PYTHON_VERSION" "$VENV_DIR"\nfi\nif ! "$VENV_DIR/bin/python" -c \'import sys; assert sys.version_info[:3] == (3, 12, 11)\' ; then\n    echo "ERROR: incompatible or incomplete venv at $VENV_DIR; start a fresh runtime." >&2\n    exit 1\nfi\n\n# Bootstrap build tools, then install the checked-in, hash-locked environment.\n# No build isolation means source-package builds use these exact build tools.\n"$UV" pip install --python "$VENV_DIR/bin/python" --no-deps \\\n    pip==25.0.1 setuptools==75.8.0 wheel==0.45.1 packaging==25.0\n"$UV" pip sync --python "$VENV_DIR/bin/python" --require-hashes \\\n    --no-build-isolation "$SCRIPT_DIR/requirements-linux-py312.lock"\nSAM2_BUILD_CUDA=0 "$VENV_DIR/bin/python" -m pip install \\\n    --no-build-isolation --no-deps -e "$REPO_DIR/sam2"\n"$VENV_DIR/bin/python" -m pip check\n\n"$VENV_DIR/bin/python" "$SCRIPT_DIR/patch_repo.py" "$REPO_DIR"\n"$VENV_DIR/bin/python" "$SCRIPT_DIR/patch_depth.py" "$REPO_DIR"\n"$VENV_DIR/bin/python" "$SCRIPT_DIR/patch_launcher.py" "$REPO_DIR"\n\n# Only use Drive after the user mounted it. Each teammate gets their own cache.\nCACHE_ARGS=()\nif mountpoint -q /content/drive && [[ -d /content/drive/MyDrive ]]; then\n    CACHE_ARGS=(--cache /content/drive/MyDrive/seganymo_checkpoints)\nfi\n"$VENV_DIR/bin/python" "$SCRIPT_DIR/checkpoints.py" "$REPO_DIR" "${CACHE_ARGS[@]}"\n\n# Activation in one %%bash cell does not persist into another notebook cell.\n# Save a small environment file that every inference cell explicitly sources.\ncat > /content/seganymo-env.sh <<\'ENV\'\nexport SEGANYMO_REPO=/content/SegAnyMo\nexport PATH="/content/seganymo-venv/bin:$PATH"\nexport PYTHONNOUSERSITE=1\nexport PYTHONPATH="/content/SegAnyMo/sam2:/content/SegAnyMo/preproc:/content/SegAnyMo/preproc/dinov2:/content/SegAnyMo"\nexport XFORMERS_DISABLED=1\nexport MPLBACKEND=Agg\nexport NVIDIA_TF32_OVERRIDE=0\nENV\nsource /content/seganymo-env.sh\npython "$SCRIPT_DIR/smoke_test.py" "$REPO_DIR"\npython -m pip freeze --all > "$REPO_DIR/environment.txt"\ngit submodule status --recursive > "$REPO_DIR/submodules.txt"\necho \'Setup checks passed. Source /content/seganymo-env.sh in every inference bash cell.\'\n', 'requirements-linux-py312.lock': '# This file was autogenerated by uv via the following command:\n#    uv pip compile colab_team_setup/requirements.in --python-version 3.12.11 --python-platform x86_64-manylinux_2_28 --generate-hashes --no-annotate --exclude-newer 2025-09-04 -o colab_team_setup/requirements-linux-py312.lock --cache-dir /private/tmp/seganymo-team-uv-cache\nabsl-py==2.3.1 \\\n    --hash=sha256:a97820526f7fbfd2ec1bce83f3f25e3a14840dac0d8e02a0b71cd75db3f77fc9 \\\n    --hash=sha256:eeecf07f0c2a93ace0772c92e596ace6d3d3996c042b2128459aaae2a76de11d\nannotated-types==0.7.0 \\\n    --hash=sha256:1f02e8b43a8fbbc3f3e0d4f0f4bfc8131bcb4eebe8849b8e5c773f3a1c582a53 \\\n    --hash=sha256:aff07c09a53a08bc8cfccb9c85b05f1aa9a2a6f23728d790723543408344ce89\nantlr4-python3-runtime==4.9.3 \\\n    --hash=sha256:f224469b4168294902bb1efa80a8bf7855f24c99aef99cbefc1bcd3cce77881b\nasttokens==3.0.0 \\\n    --hash=sha256:0dcd8baa8d62b0c1d118b399b2ddba3c4aff271d0d7a9e0d4c1681c79035bbc7 \\\n    --hash=sha256:e3078351a059199dd5138cb1c706e6430c05eff2ff136af5eb4790f9d28932e2\ncertifi==2025.8.3 \\\n    --hash=sha256:e564105f78ded564e3ae7c923924435e1daa7463faeab5bb932bc53ffae63407 \\\n    --hash=sha256:f6c12493cfb1b06ba2ff328595af9350c65d6644968e5d3a2ffd78699af217a5\ncharset-normalizer==3.4.3 \\\n    --hash=sha256:00237675befef519d9af72169d8604a067d92755e84fe76492fef5441db05b91 \\\n    --hash=sha256:02425242e96bcf29a49711b0ca9f37e451da7c70562bc10e8ed992a5a7a25cc0 \\\n    --hash=sha256:027b776c26d38b7f15b26a5da1044f376455fb3766df8fc38563b4efbc515154 \\\n    --hash=sha256:07a0eae9e2787b586e129fdcbe1af6997f8d0e5abaa0bc98c0e20e124d67e601 \\\n    --hash=sha256:0cacf8f7297b0c4fcb74227692ca46b4a5852f8f4f24b3c766dd94a1075c4884 \\\n    --hash=sha256:0e78314bdc32fa80696f72fa16dc61168fda4d6a0c014e0380f9d02f0e5d8a07 \\\n    --hash=sha256:0f2be7e0cf7754b9a30eb01f4295cc3d4358a479843b31f328afd210e2c7598c \\\n    --hash=sha256:13faeacfe61784e2559e690fc53fa4c5ae97c6fcedb8eb6fb8d0a15b475d2c64 \\\n    --hash=sha256:14c2a87c65b351109f6abfc424cab3927b3bdece6f706e4d12faaf3d52ee5efe \\\n    --hash=sha256:1606f4a55c0fd363d754049cdf400175ee96c992b1f8018b993941f221221c5f \\\n    --hash=sha256:16a8770207946ac75703458e2c743631c79c59c5890c80011d536248f8eaa432 \\\n    --hash=sha256:18343b2d246dc6761a249ba1fb13f9ee9a2bcd95decc767319506056ea4ad4dc \\\n    --hash=sha256:18b97b8404387b96cdbd30ad660f6407799126d26a39ca65729162fd810a99aa \\\n    --hash=sha256:1bb60174149316da1c35fa5233681f7c0f9f514509b8e399ab70fea5f17e45c9 \\\n    --hash=sha256:1e8ac75d72fa3775e0b7cb7e4629cec13b7514d928d15ef8ea06bca03ef01cae \\\n    --hash=sha256:1ef99f0456d3d46a50945c98de1774da86f8e992ab5c77865ea8b8195341fc19 \\\n    --hash=sha256:2001a39612b241dae17b4687898843f254f8748b796a2e16f1051a17078d991d \\\n    --hash=sha256:23b6b24d74478dc833444cbd927c338349d6ae852ba53a0d02a2de1fce45b96e \\\n    --hash=sha256:252098c8c7a873e17dd696ed98bbe91dbacd571da4b87df3736768efa7a792e4 \\\n    --hash=sha256:257f26fed7d7ff59921b78244f3cd93ed2af1800ff048c33f624c87475819dd7 \\\n    --hash=sha256:2c322db9c8c89009a990ef07c3bcc9f011a3269bc06782f916cd3d9eed7c9312 \\\n    --hash=sha256:30a96e1e1f865f78b030d65241c1ee850cdf422d869e9028e2fc1d5e4db73b92 \\\n    --hash=sha256:30d006f98569de3459c2fc1f2acde170b7b2bd265dc1943e87e1a4efe1b67c31 \\\n    --hash=sha256:31a9a6f775f9bcd865d88ee350f0ffb0e25936a7f930ca98995c05abf1faf21c \\\n    --hash=sha256:320e8e66157cc4e247d9ddca8e21f427efc7a04bbd0ac8a9faf56583fa543f9f \\\n    --hash=sha256:34a7f768e3f985abdb42841e20e17b330ad3aaf4bb7e7aeeb73db2e70f077b99 \\\n    --hash=sha256:3653fad4fe3ed447a596ae8638b437f827234f01a8cd801842e43f3d0a6b281b \\\n    --hash=sha256:3cd35b7e8aedeb9e34c41385fda4f73ba609e561faedfae0a9e75e44ac558a15 \\\n    --hash=sha256:3cfb2aad70f2c6debfbcb717f23b7eb55febc0bb23dcffc0f076009da10c6392 \\\n    --hash=sha256:416175faf02e4b0810f1f38bcb54682878a4af94059a1cd63b8747244420801f \\\n    --hash=sha256:41d1fc408ff5fdfb910200ec0e74abc40387bccb3252f3f27c0676731df2b2c8 \\\n    --hash=sha256:42e5088973e56e31e4fa58eb6bd709e42fc03799c11c42929592889a2e54c491 \\\n    --hash=sha256:4ca4c094de7771a98d7fbd67d9e5dbf1eb73efa4f744a730437d8a3a5cf994f0 \\\n    --hash=sha256:511729f456829ef86ac41ca78c63a5cb55240ed23b4b737faca0eb1abb1c41bc \\\n    --hash=sha256:53cd68b185d98dde4ad8990e56a58dea83a4162161b1ea9272e5c9182ce415e0 \\\n    --hash=sha256:585f3b2a80fbd26b048a0be90c5aae8f06605d3c92615911c3a2b03a8a3b796f \\\n    --hash=sha256:5b413b0b1bfd94dbf4023ad6945889f374cd24e3f62de58d6bb102c4d9ae534a \\\n    --hash=sha256:5d8d01eac18c423815ed4f4a2ec3b439d654e55ee4ad610e153cf02faf67ea40 \\\n    --hash=sha256:6aab0f181c486f973bc7262a97f5aca3ee7e1437011ef0c2ec04b5a11d16c927 \\\n    --hash=sha256:6cf8fd4c04756b6b60146d98cd8a77d0cdae0e1ca20329da2ac85eed779b6849 \\\n    --hash=sha256:6fb70de56f1859a3f71261cbe41005f56a7842cc348d3aeb26237560bfa5e0ce \\\n    --hash=sha256:6fce4b8500244f6fcb71465d4a4930d132ba9ab8e71a7859e6a5d59851068d14 \\\n    --hash=sha256:70bfc5f2c318afece2f5838ea5e4c3febada0be750fcf4775641052bbba14d05 \\\n    --hash=sha256:73dc19b562516fc9bcf6e5d6e596df0b4eb98d87e4f79f3ae71840e6ed21361c \\\n    --hash=sha256:74d77e25adda8581ffc1c720f1c81ca082921329452eba58b16233ab1842141c \\\n    --hash=sha256:78deba4d8f9590fe4dae384aeff04082510a709957e968753ff3c48399f6f92a \\\n    --hash=sha256:86df271bf921c2ee3818f0522e9a5b8092ca2ad8b065ece5d7d9d0e9f4849bcc \\\n    --hash=sha256:88ab34806dea0671532d3f82d82b85e8fc23d7b2dd12fa837978dad9bb392a34 \\\n    --hash=sha256:8999f965f922ae054125286faf9f11bc6932184b93011d138925a1773830bbe9 \\\n    --hash=sha256:8dcfc373f888e4fb39a7bc57e93e3b845e7f462dacc008d9749568b1c4ece096 \\\n    --hash=sha256:939578d9d8fd4299220161fdd76e86c6a251987476f5243e8864a7844476ba14 \\\n    --hash=sha256:96b2b3d1a83ad55310de8c7b4a2d04d9277d5591f40761274856635acc5fcb30 \\\n    --hash=sha256:a2d08ac246bb48479170408d6c19f6385fa743e7157d716e144cad849b2dd94b \\\n    --hash=sha256:b256ee2e749283ef3ddcff51a675ff43798d92d746d1a6e4631bf8c707d22d0b \\\n    --hash=sha256:b5e3b2d152e74e100a9e9573837aba24aab611d39428ded46f4e4022ea7d1942 \\\n    --hash=sha256:b89bc04de1d83006373429975f8ef9e7932534b8cc9ca582e4db7d20d91816db \\\n    --hash=sha256:bd28b817ea8c70215401f657edef3a8aa83c29d447fb0b622c35403780ba11d5 \\\n    --hash=sha256:c60e092517a73c632ec38e290eba714e9627abe9d301c8c8a12ec32c314a2a4b \\\n    --hash=sha256:c6dbd0ccdda3a2ba7c2ecd9d77b37f3b5831687d8dc1b6ca5f56a4880cc7b7ce \\\n    --hash=sha256:c6e490913a46fa054e03699c70019ab869e990270597018cef1d8562132c2669 \\\n    --hash=sha256:c6f162aabe9a91a309510d74eeb6507fab5fff92337a15acbe77753d88d9dcf0 \\\n    --hash=sha256:c6fd51128a41297f5409deab284fecbe5305ebd7e5a1f959bee1c054622b7018 \\\n    --hash=sha256:cc34f233c9e71701040d772aa7490318673aa7164a0efe3172b2981218c26d93 \\\n    --hash=sha256:cc9370a2da1ac13f0153780040f465839e6cccb4a1e44810124b4e22483c93fe \\\n    --hash=sha256:ccf600859c183d70eb47e05a44cd80a4ce77394d1ac0f79dbd2dd90a69a3a049 \\\n    --hash=sha256:ce571ab16d890d23b5c278547ba694193a45011ff86a9162a71307ed9f86759a \\\n    --hash=sha256:cf1ebb7d78e1ad8ec2a8c4732c7be2e736f6e5123a4146c5b89c9d1f585f8cef \\\n    --hash=sha256:d0e909868420b7049dafd3a31d45125b31143eec59235311fc4c57ea26a4acd2 \\\n    --hash=sha256:d22dbedd33326a4a5190dd4fe9e9e693ef12160c77382d9e87919bce54f3d4ca \\\n    --hash=sha256:d716a916938e03231e86e43782ca7878fb602a125a91e7acb8b5112e2e96ac16 \\\n    --hash=sha256:d79c198e27580c8e958906f803e63cddb77653731be08851c7df0b1a14a8fc0f \\\n    --hash=sha256:d95bfb53c211b57198bb91c46dd5a2d8018b3af446583aab40074bf7988401cb \\\n    --hash=sha256:e28e334d3ff134e88989d90ba04b47d84382a828c061d0d1027b1b12a62b39b1 \\\n    --hash=sha256:ec557499516fc90fd374bf2e32349a2887a876fbf162c160e3c01b6849eaf557 \\\n    --hash=sha256:fb6fecfd65564f208cbf0fba07f107fb661bcd1a7c389edbced3f7a493f70e37 \\\n    --hash=sha256:fb731e5deb0c7ef82d698b0f4c5bb724633ee2a489401594c5c88b02e6cb15f7 \\\n    --hash=sha256:fb7f67a1bfa6e40b438170ebdc8158b78dc465a5a67b6dde178a46987b244a72 \\\n    --hash=sha256:fd10de089bcdcd1be95a2f73dbe6254798ec1bda9f450d5828c96f93e2536b9c \\\n    --hash=sha256:fdabf8315679312cfa71302f9bd509ded4f2f263fb5b765cf1433b39106c3cc9\nclick==8.2.1 \\\n    --hash=sha256:27c491cc05d968d271d5a1db13e3b5a184636d9d930f148c50b038f0d0646202 \\\n    --hash=sha256:61a3265b914e850b85317d0b3109c7f8cd35a670f963866005d6ef1d5175a12b\ncloudpickle==3.1.1 \\\n    --hash=sha256:b216fa8ae4019d5482a8ac3c95d8f6346115d8835911fd4aefd1a445e4242c64 \\\n    --hash=sha256:c8c5a44295039331ee9dad40ba100a9c7297b6f988e50e87ccdf3765a668350e\ncontourpy==1.3.3 \\\n    --hash=sha256:023b44101dfe49d7d53932be418477dba359649246075c996866106da069af69 \\\n    --hash=sha256:07ce5ed73ecdc4a03ffe3e1b3e3c1166db35ae7584be76f65dbbe28a7791b0cc \\\n    --hash=sha256:083e12155b210502d0bca491432bb04d56dc3432f95a979b429f2848c3dbe880 \\\n    --hash=sha256:0bf67e0e3f482cb69779dd3061b534eb35ac9b17f163d851e2a547d56dba0a3a \\\n    --hash=sha256:0c1fc238306b35f246d61a1d416a627348b5cf0648648a031e14bb8705fcdfe8 \\\n    --hash=sha256:13b68d6a62db8eafaebb8039218921399baf6e47bf85006fd8529f2a08ef33fc \\\n    --hash=sha256:15ff10bfada4bf92ec8b31c62bf7c1834c244019b4a33095a68000d7075df470 \\\n    --hash=sha256:177fb367556747a686509d6fef71d221a4b198a3905fe824430e5ea0fda54eb5 \\\n    --hash=sha256:1cadd8b8969f060ba45ed7c1b714fe69185812ab43bd6b86a9123fe8f99c3263 \\\n    --hash=sha256:1fd43c3be4c8e5fd6e4f2baeae35ae18176cf2e5cced681cca908addf1cdd53b \\\n    --hash=sha256:22e9b1bd7a9b1d652cd77388465dc358dafcd2e217d35552424aa4f996f524f5 \\\n    --hash=sha256:23416f38bfd74d5d28ab8429cc4d63fa67d5068bd711a85edb1c3fb0c3e2f381 \\\n    --hash=sha256:283edd842a01e3dcd435b1c5116798d661378d83d36d337b8dde1d16a5fc9ba3 \\\n    --hash=sha256:2a2a8b627d5cc6b7c41a4beff6c5ad5eb848c88255fda4a8745f7e901b32d8e4 \\\n    --hash=sha256:2b7e9480ffe2b0cd2e787e4df64270e3a0440d9db8dc823312e2c940c167df7e \\\n    --hash=sha256:322ab1c99b008dad206d406bb61d014cf0174df491ae9d9d0fac6a6fda4f977f \\\n    --hash=sha256:33c82d0138c0a062380332c861387650c82e4cf1747aaa6938b9b6516762e772 \\\n    --hash=sha256:348ac1f5d4f1d66d3322420f01d42e43122f43616e0f194fc1c9f5d830c5b286 \\\n    --hash=sha256:3519428f6be58431c56581f1694ba8e50626f2dd550af225f82fb5f5814d2a42 \\\n    --hash=sha256:3c30273eb2a55024ff31ba7d052dde990d7d8e5450f4bbb6e913558b3d6c2301 \\\n    --hash=sha256:3d1a3799d62d45c18bafd41c5fa05120b96a28079f2393af559b843d1a966a77 \\\n    --hash=sha256:451e71b5a7d597379ef572de31eeb909a87246974d960049a9848c3bc6c41bf7 \\\n    --hash=sha256:459c1f020cd59fcfe6650180678a9993932d80d44ccde1fa1868977438f0b411 \\\n    --hash=sha256:4d00e655fcef08aba35ec9610536bfe90267d7ab5ba944f7032549c55a146da1 \\\n    --hash=sha256:4debd64f124ca62069f313a9cb86656ff087786016d76927ae2cf37846b006c9 \\\n    --hash=sha256:4feffb6537d64b84877da813a5c30f1422ea5739566abf0bd18065ac040e120a \\\n    --hash=sha256:50ed930df7289ff2a8d7afeb9603f8289e5704755c7e5c3bbd929c90c817164b \\\n    --hash=sha256:51e79c1f7470158e838808d4a996fa9bac72c498e93d8ebe5119bc1e6becb0db \\\n    --hash=sha256:556dba8fb6f5d8742f2923fe9457dbdd51e1049c4a43fd3986a0b14a1d815fc6 \\\n    --hash=sha256:598c3aaece21c503615fd59c92a3598b428b2f01bfb4b8ca9c4edeecc2438620 \\\n    --hash=sha256:5ed3657edf08512fc3fe81b510e35c2012fbd3081d2e26160f27ca28affec989 \\\n    --hash=sha256:626d60935cf668e70a5ce6ff184fd713e9683fb458898e4249b63be9e28286ea \\\n    --hash=sha256:644a6853d15b2512d67881586bd03f462c7ab755db95f16f14d7e238f2852c67 \\\n    --hash=sha256:655456777ff65c2c548b7c454af9c6f33f16c8884f11083244b5819cc214f1b5 \\\n    --hash=sha256:66c8a43a4f7b8df8b71ee1840e4211a3c8d93b214b213f590e18a1beca458f7d \\\n    --hash=sha256:6afc576f7b33cf00996e5c1102dc2a8f7cc89e39c0b55df93a0b78c1bd992b36 \\\n    --hash=sha256:6c3d53c796f8647d6deb1abe867daeb66dcc8a97e8455efa729516b997b8ed99 \\\n    --hash=sha256:709a48ef9a690e1343202916450bc48b9e51c049b089c7f79a267b46cffcdaa1 \\\n    --hash=sha256:70f9aad7de812d6541d29d2bbf8feb22ff7e1c299523db288004e3157ff4674e \\\n    --hash=sha256:8153b8bfc11e1e4d75bcb0bff1db232f9e10b274e0929de9d608027e0d34ff8b \\\n    --hash=sha256:87acf5963fc2b34825e5b6b048f40e3635dd547f590b04d2ab317c2619ef7ae8 \\\n    --hash=sha256:88df9880d507169449d434c293467418b9f6cbe82edd19284aa0409e7fdb933d \\\n    --hash=sha256:929ddf8c4c7f348e4c0a5a3a714b5c8542ffaa8c22954862a46ca1813b667ee7 \\\n    --hash=sha256:92d9abc807cf7d0e047b95ca5d957cf4792fcd04e920ca70d48add15c1a90ea7 \\\n    --hash=sha256:95b181891b4c71de4bb404c6621e7e2390745f887f2a026b2d99e92c17892339 \\\n    --hash=sha256:9e999574eddae35f1312c2b4b717b7885d4edd6cb46700e04f7f02db454e67c1 \\\n    --hash=sha256:a15459b0f4615b00bbd1e91f1b9e19b7e63aea7483d03d804186f278c0af2659 \\\n    --hash=sha256:a22738912262aa3e254e4f3cb079a95a67132fc5a063890e224393596902f5a4 \\\n    --hash=sha256:ab2fd90904c503739a75b7c8c5c01160130ba67944a7b77bbf36ef8054576e7f \\\n    --hash=sha256:ab3074b48c4e2cf1a960e6bbeb7f04566bf36b1861d5c9d4d8ac04b82e38ba20 \\\n    --hash=sha256:afe5a512f31ee6bd7d0dda52ec9864c984ca3d66664444f2d72e0dc4eb832e36 \\\n    --hash=sha256:b08a32ea2f8e42cf1d4be3169a98dd4be32bafe4f22b6c4cb4ba810fa9e5d2cb \\\n    --hash=sha256:b20c7c9a3bf701366556e1b1984ed2d0cedf999903c51311417cf5f591d8c78d \\\n    --hash=sha256:b2e8faa0ed68cb29af51edd8e24798bb661eac3bd9f65420c1887b6ca89987c8 \\\n    --hash=sha256:b7301b89040075c30e5768810bc96a8e8d78085b47d8be6e4c3f5a0b4ed478a0 \\\n    --hash=sha256:b7448cb5a725bb1e35ce88771b86fba35ef418952474492cf7c764059933ff8b \\\n    --hash=sha256:ca0fdcd73925568ca027e0b17ab07aad764be4706d0a925b89227e447d9737b7 \\\n    --hash=sha256:ca658cd1a680a5c9ea96dc61cdbae1e85c8f25849843aa799dfd3cb370ad4fbe \\\n    --hash=sha256:cbedb772ed74ff5be440fa8eee9bd49f64f6e3fc09436d9c7d8f1c287b121d77 \\\n    --hash=sha256:cd5dfcaeb10f7b7f9dc8941717c6c2ade08f587be2226222c12b25f0483ed497 \\\n    --hash=sha256:cf9022ef053f2694e31d630feaacb21ea24224be1c3ad0520b13d844274614fd \\\n    --hash=sha256:d002b6f00d73d69333dac9d0b8d5e84d9724ff9ef044fd63c5986e62b7c9e1b1 \\\n    --hash=sha256:d06bb1f751ba5d417047db62bca3c8fde202b8c11fb50742ab3ab962c81e8216 \\\n    --hash=sha256:d304906ecc71672e9c89e87c4675dc5c2645e1f4269a5063b99b0bb29f232d13 \\\n    --hash=sha256:e4e6b05a45525357e382909a4c1600444e2a45b4795163d3b22669285591c1ae \\\n    --hash=sha256:e74a9a0f5e3fff48fb5a7f2fd2b9b70a3fe014a67522f79b7cca4c0c7e43c9ae \\\n    --hash=sha256:ea37e7b45949df430fe649e5de8351c423430046a2af20b1c1961cae3afcda77 \\\n    --hash=sha256:f64836de09927cba6f79dcd00fdd7d5329f3fccc633468507079c829ca4db4e3 \\\n    --hash=sha256:fd6ec6be509c787f1caf6b247f0b1ca598bef13f4ddeaa126b7658215529ba0f \\\n    --hash=sha256:fd907ae12cd483cd83e414b12941c632a969171bf90fc937d0c9f268a31cafff \\\n    --hash=sha256:fd914713266421b7536de2bfa8181aa8c699432b6763a0ea64195ebe28bff6a9 \\\n    --hash=sha256:fde6c716d51c04b1c25d0b90364d0be954624a0ee9d60e23e850e8d48353d07a\ncycler==0.12.1 \\\n    --hash=sha256:85cef7cff222d8644161529808465972e51340599459b8ac3ccbac5a854e0d30 \\\n    --hash=sha256:88bb128f02ba341da8ef447245a9e138fae777f6a23943da4540077d3601eb1c\ndecorator==5.2.1 \\\n    --hash=sha256:65f266143752f734b0a7cc83c46f4618af75b8c5911b00ccb61d0ac9b6da0360 \\\n    --hash=sha256:d316bb415a2d9e2d2b3abcc4084c6502fc09240e292cd76a76afc106a1c8e04a\ndocker-pycreds==0.4.0 \\\n    --hash=sha256:6ce3270bcaf404cc4c3e27e4b6c70d3521deae82fb508767870fdbf772d584d4 \\\n    --hash=sha256:7266112468627868005106ec19cd0d722702d2b7d5912a28e19b826c3d37af49\neinops==0.8.1 \\\n    --hash=sha256:919387eb55330f5757c6bea9165c5ff5cfe63a642682ea788a6d472576d81737 \\\n    --hash=sha256:de5d960a7a761225532e0f1959e5315ebeafc0cd43394732f103ca44b9837e84\neinshape==1.0 \\\n    --hash=sha256:42da4c2dea3a27f87ee45a7cee5072a636b97cb184bb07bf5d6412ba0ff7b965 \\\n    --hash=sha256:53538d75dd099f4ead4a4f786fafdcb0b729bb587e0b3afeca25ceef18c9ac14\nexecuting==2.2.1 \\\n    --hash=sha256:3632cc370565f6648cc328b32435bd120a1e4ebb20c77e3fdde9a13cd1e533c4 \\\n    --hash=sha256:760643d3452b4d777d295bb167ccc74c64a81df23fb5e08eff250c425a4b2017\nfilelock==3.18.0 \\\n    --hash=sha256:adbc88eabb99d2fec8c9c1b229b171f18afa655400173ddc653d5d01501fb9f2 \\\n    --hash=sha256:c401f4f8377c4464e6db25fff06205fd89bdd83b65eb0488ed1b160f780e21de\nflow-vis==0.1 \\\n    --hash=sha256:0cde94741777953f9de6deb125f34ea11f71ffe435892135a98bd480e22643a4 \\\n    --hash=sha256:73ad53717d0e35de5c501dbd987349092f3e0772c4ecf8b4f37ba0c7cae2e98b\nfonttools==4.59.2 \\\n    --hash=sha256:036cd87a2dbd7ef72f7b68df8314ced00b8d9973aee296f2464d06a836aeb9a9 \\\n    --hash=sha256:0476ea74161322e08c7a982f83558a2b81b491509984523a1a540baf8611cc31 \\\n    --hash=sha256:0ec99f9bdfee9cdb4a9172f9e8fd578cce5feb231f598909e0aecf5418da4f25 \\\n    --hash=sha256:12dc4670e6e6cc4553e8de190f86a549e08ca83a036363115d94a2d67488831e \\\n    --hash=sha256:14870930181493b1d740b6f25483e20185e5aea58aec7d266d16da7be822b4bb \\\n    --hash=sha256:1603b85d5922042563eea518e272b037baf273b9a57d0f190852b0b075079000 \\\n    --hash=sha256:1647201af10993090120da2e66e9526c4e20e88859f3e34aa05b8c24ded2a564 \\\n    --hash=sha256:1a1bfe5378962825dabe741720885e8b9ae9745ec7ecc4a5ec1f1ce59a6062bf \\\n    --hash=sha256:2543b81641ea5b8ddfcae7926e62aafd5abc604320b1b119e5218c014a7a5d3c \\\n    --hash=sha256:2a159e36ae530650acd13604f364b3a2477eff7408dcac6a640d74a3744d2514 \\\n    --hash=sha256:381bde13216ba09489864467f6bc0c57997bd729abfbb1ce6f807ba42c06cceb \\\n    --hash=sha256:39ad9612c6a622726a6a130e8ab15794558591f999673f1ee7d2f3d30f6a3e1c \\\n    --hash=sha256:3cdf9d32690f0e235342055f0a6108eedfccf67b213b033bac747eb809809513 \\\n    --hash=sha256:464d15b58a9fd4304c728735fc1d42cd812fd9ebc27c45b18e78418efd337c28 \\\n    --hash=sha256:47742c33fe65f41eabed36eec2d7313a8082704b7b808752406452f766c573fc \\\n    --hash=sha256:4d974312a9f405628e64f475b1f5015a61fd338f0a1b61d15c4822f97d6b045b \\\n    --hash=sha256:511946e8d7ea5c0d6c7a53c4cb3ee48eda9ab9797cd9bf5d95829a398400354f \\\n    --hash=sha256:53c1a411b7690042535a4f0edf2120096a39a506adeb6c51484a232e59f2aa0c \\\n    --hash=sha256:5729e12a982dba3eeae650de48b06f3b9ddb51e9aee2fcaf195b7d09a96250e2 \\\n    --hash=sha256:594a6fd2f8296583ac7babc4880c8deee7c4f05ab0141addc6bce8b8e367e996 \\\n    --hash=sha256:59d85088e29fa7a8f87d19e97a1beae2a35821ee48d8ef6d2c4f965f26cb9f8a \\\n    --hash=sha256:6235fc06bcbdb40186f483ba9d5d68f888ea68aa3c8dac347e05a7c54346fbc8 \\\n    --hash=sha256:67f9640d6b31d66c0bc54bdbe8ed50983c755521c101576a25e377a8711e8207 \\\n    --hash=sha256:6dee142b8b3096514c96ad9e2106bf039e2fe34a704c587585b569a36df08c3c \\\n    --hash=sha256:738f31f23e0339785fd67652a94bc69ea49e413dfdb14dcb8c8ff383d249464e \\\n    --hash=sha256:7ad5d8d8cc9e43cb438b3eb4a0094dd6d4088daa767b0a24d52529361fd4c199 \\\n    --hash=sha256:7bb32e0e33795e3b7795bb9b88cb6a9d980d3cbe26dd57642471be547708e17a \\\n    --hash=sha256:7ff58ea1eb8fc7e05e9a949419f031890023f8785c925b44d6da17a6a7d6e85d \\\n    --hash=sha256:82906d002c349cad647a7634b004825a7335f8159d0d035ae89253b4abf6f3ea \\\n    --hash=sha256:83ad6e5d06ef3a2884c4fa6384a20d6367b5cfe560e3b53b07c9dc65a7020e73 \\\n    --hash=sha256:8991bdbae39cf78bcc9cd3d81f6528df1f83f2e7c23ccf6f990fa1f0b6e19708 \\\n    --hash=sha256:8bd0f759020e87bb5d323e6283914d9bf4ae35a7307dafb2cbd1e379e720ad37 \\\n    --hash=sha256:8bd733e47bf4c6dee2b2d8af7a1f7b0c091909b22dbb969a29b2b991e61e5ba4 \\\n    --hash=sha256:8e5e2682cf7be766d84f462ba8828d01e00c8751a8e8e7ce12d7784ccb69a30d \\\n    --hash=sha256:92ac2d45794f95d1ad4cb43fa07e7e3776d86c83dc4b9918cf82831518165b4b \\\n    --hash=sha256:95807a3b5e78f2714acaa26a33bc2143005cc05c0217b322361a772e59f32b89 \\\n    --hash=sha256:95922a922daa1f77cc72611747c156cfb38030ead72436a2c551d30ecef519b9 \\\n    --hash=sha256:980fd7388e461b19a881d35013fec32c713ffea1fc37aef2f77d11f332dfd7da \\\n    --hash=sha256:9836394e2f4ce5f9c0a7690ee93bd90aa1adc6b054f1a57b562c5d242c903104 \\\n    --hash=sha256:9cde8b6a6b05f68516573523f2013a3574cb2c75299d7d500f44de82ba947b80 \\\n    --hash=sha256:a039c38d5644c691eb53cd65360921338f54e44c90b4e764605711e046c926ee \\\n    --hash=sha256:a10c1bd7644dc58f8862d8ba0cf9fb7fef0af01ea184ba6ce3f50ab7dfe74d5a \\\n    --hash=sha256:a72155928d7053bbde499d32a9c77d3f0f3d29ae72b5a121752481bcbd71e50f \\\n    --hash=sha256:a8d40594982ed858780e18a7e4c80415af65af0f22efa7de26bdd30bf24e1e14 \\\n    --hash=sha256:af6dbd463a3530256abf21f675ddf87646272bc48901803a185c49d06287fbf1 \\\n    --hash=sha256:b3ebda00c3bb8f32a740b72ec38537d54c7c09f383a4cfefb0b315860f825b08 \\\n    --hash=sha256:c52694eae5d652361d59ecdb5a2246bff7cff13b6367a12da8499e9df56d148d \\\n    --hash=sha256:cdcdf7aad4bab7fd0f2938624a5a84eb4893be269f43a6701b0720b726f24df0 \\\n    --hash=sha256:d029804c70fddf90be46ed5305c136cae15800a2300cb0f6bba96d48e770dde0 \\\n    --hash=sha256:d09e487d6bfbe21195801323ba95c91cb3523f0fcc34016454d4d9ae9eaa57fe \\\n    --hash=sha256:dec2f22486d7781087b173799567cffdcc75e9fb2f1c045f05f8317ccce76a3e \\\n    --hash=sha256:e4f5100e66ec307cce8b52fc03e379b5d1596e9cb8d8b19dfeeccc1e68d86c96 \\\n    --hash=sha256:e72c0749b06113f50bcb80332364c6be83a9582d6e3db3fe0b280f996dc2ef22 \\\n    --hash=sha256:e937790f3c2c18a1cbc7da101550a84319eb48023a715914477d2e7faeaba570 \\\n    --hash=sha256:f1f1bbc23ba1312bd8959896f46f667753b90216852d2a8cfa2d07e0cb234144 \\\n    --hash=sha256:f33839aa091f7eef4e9078f5b7ab1b8ea4b1d8a50aeaef9fdb3611bba80869ec \\\n    --hash=sha256:fa9ecaf2dcef8941fb5719e16322345d730f4c40599bbf47c9753de40eb03882 \\\n    --hash=sha256:fc21c4a05226fd39715f66c1c28214862474db50df9f08fd1aa2f96698887bc3\nfsspec==2025.9.0 \\\n    --hash=sha256:19fd429483d25d28b65ec68f9f4adc16c17ea2c7c7bf54ec61360d478fb19c19 \\\n    --hash=sha256:530dc2a2af60a414a832059574df4a6e10cce927f6f4a78209390fe38955cfb7\ngitdb==4.0.12 \\\n    --hash=sha256:5ef71f855d191a3326fcfbc0d5da835f26b13fbcba60c32c21091c349ffdb571 \\\n    --hash=sha256:67073e15955400952c6565cc3e707c554a4eea2e428946f7a4c162fab9bd9bcf\ngitpython==3.1.45 \\\n    --hash=sha256:85b0ee964ceddf211c41b9f27a49086010a190fd8132a24e21f362a4b36a791c \\\n    --hash=sha256:8908cb2e02fb3b93b7eb0f2827125cb699869470432cc885f019b8fd0fccff77\nh5py==3.12.1 \\\n    --hash=sha256:018a4597f35092ae3fb28ee851fdc756d2b88c96336b8480e124ce1ac6fb9166 \\\n    --hash=sha256:050a4f2c9126054515169c49cb900949814987f0c7ae74c341b0c9f9b5056834 \\\n    --hash=sha256:06a903a4e4e9e3ebbc8b548959c3c2552ca2d70dac14fcfa650d9261c66939ed \\\n    --hash=sha256:1473348139b885393125126258ae2d70753ef7e9cec8e7848434f385ae72069e \\\n    --hash=sha256:2f0f1a382cbf494679c07b4371f90c70391dedb027d517ac94fa2c05299dacda \\\n    --hash=sha256:326d70b53d31baa61f00b8aa5f95c2fcb9621a3ee8365d770c551a13dbbcbfdf \\\n    --hash=sha256:3b15d8dbd912c97541312c0e07438864d27dbca857c5ad634de68110c6beb1c2 \\\n    --hash=sha256:3fdf95092d60e8130ba6ae0ef7a9bd4ade8edbe3569c13ebbaf39baefffc5ba4 \\\n    --hash=sha256:4532c7e97fbef3d029735db8b6f5bf01222d9ece41e309b20d63cfaae2fb5c4d \\\n    --hash=sha256:513171e90ed92236fc2ca363ce7a2fc6f2827375efcbb0cc7fbdd7fe11fecafc \\\n    --hash=sha256:52ab036c6c97055b85b2a242cb540ff9590bacfda0c03dd0cf0661b311f522f8 \\\n    --hash=sha256:577d618d6b6dea3da07d13cc903ef9634cde5596b13e832476dd861aaf651f3e \\\n    --hash=sha256:59400f88343b79655a242068a9c900001a34b63e3afb040bd7cdf717e440f653 \\\n    --hash=sha256:59685fe40d8c1fbbee088c88cd4da415a2f8bee5c270337dc5a1c4aa634e3307 \\\n    --hash=sha256:5c4b41d1019322a5afc5082864dfd6359f8935ecd37c11ac0029be78c5d112c9 \\\n    --hash=sha256:62be1fc0ef195891949b2c627ec06bc8e837ff62d5b911b6e42e38e0f20a897d \\\n    --hash=sha256:6fdf6d7936fa824acfa27305fe2d9f39968e539d831c5bae0e0d83ed521ad1ac \\\n    --hash=sha256:7b3b8f3b48717e46c6a790e3128d39c61ab595ae0a7237f06dfad6a3b51d5351 \\\n    --hash=sha256:84342bffd1f82d4f036433e7039e241a243531a1d3acd7341b35ae58cdab05bf \\\n    --hash=sha256:ad8a76557880aed5234cfe7279805f4ab5ce16b17954606cca90d578d3e713ef \\\n    --hash=sha256:ba51c0c5e029bb5420a343586ff79d56e7455d496d18a30309616fdbeed1068f \\\n    --hash=sha256:cb65f619dfbdd15e662423e8d257780f9a66677eae5b4b3fc9dca70b5fd2d2a3 \\\n    --hash=sha256:ccd9006d92232727d23f784795191bfd02294a4f2ba68708825cb1da39511a93 \\\n    --hash=sha256:d2b8dd64f127d8b324f5d2cd1c0fd6f68af69084e9e47d27efeb9e28e685af3e \\\n    --hash=sha256:d3e465aee0ec353949f0f46bf6c6f9790a2006af896cee7c178a8c3e5090aa32 \\\n    --hash=sha256:e4d51919110a030913201422fb07987db4338eba5ec8c5a15d6fab8e03d443fc\nhuggingface-hub==0.24.5 \\\n    --hash=sha256:7b45d6744dd53ce9cbf9880957de00e9d10a9ae837f1c9b7255fc8fa4e8264f3 \\\n    --hash=sha256:d93fb63b1f1a919a22ce91a14518974e81fc4610bf344dfe7572343ce8d3aced\nhydra-core==1.3.2 \\\n    --hash=sha256:8a878ed67216997c3e9d88a8e72e7b4767e81af37afb4ea3334b269a4390a824 \\\n    --hash=sha256:fa0238a9e31df3373b35b0bfb672c34cc92718d21f81311d8996a16de1141d8b\nidna==3.10 \\\n    --hash=sha256:12f65c9b470abda6dc35cf8e63cc574b1c52b11df2c86030af0ac09b01b13ea9 \\\n    --hash=sha256:946d195a0d259cbba61165e88e65941f16e9b36ea6ddb97f00452bae8b1287d3\nimageio==2.37.0 \\\n    --hash=sha256:11efa15b87bc7871b61590326b2d635439acc321cf7f8ce996f812543ce10eed \\\n    --hash=sha256:71b57b3669666272c818497aebba2b4c5f20d5b37c81720e5e1a56d59c492996\nimageio-ffmpeg==0.5.1 \\\n    --hash=sha256:0ed7a9b31f560b0c9d929c5291cd430edeb9bed3ce9a497480e536dd4326484c \\\n    --hash=sha256:1460e84712b9d06910c1f7bb524096b0341d4b7844cea6c20e099d0a24e795b1 \\\n    --hash=sha256:1521e79e253bedbdd36a547e0cbd94a025ba0b558e17f08fea687d805a0e4698 \\\n    --hash=sha256:5289f75c7f755b499653f3209fea4efd1430cba0e39831c381aad2d458f7a316 \\\n    --hash=sha256:7fa9132a291d5eb28c44553550deb40cbdab831f2a614e55360301a6582eb205 \\\n    --hash=sha256:89efe2c79979d8174ba8476deb7f74d74c331caee3fb2b65ba2883bec0737625\niopath==0.1.10 \\\n    --hash=sha256:3311c16a4d9137223e20f141655759933e1eda24f8bff166af834af3c645ef01\nipython==9.5.0 \\\n    --hash=sha256:129c44b941fe6d9b82d36fc7a7c18127ddb1d6f02f78f867f402e2e3adde3113 \\\n    --hash=sha256:88369ffa1d5817d609120daa523a6da06d02518e582347c29f8451732a9c5e72\nipython-pygments-lexers==1.1.1 \\\n    --hash=sha256:09c0138009e56b6854f9535736f4171d855c8c08a563a0dcd8022f78355c7e81 \\\n    --hash=sha256:a9462224a505ade19a605f71f8fa63c2048833ce50abc86768a0d81d876dc81c\njedi==0.19.2 \\\n    --hash=sha256:4770dc3de41bde3966b02eb84fbcf557fb33cce26ad23da12c742fb50ecb11f0 \\\n    --hash=sha256:a8ef22bde8490f57fe5c7681a3c83cb58874daf72b4784de3cce5b6ef6edb5b9\njinja2==3.1.6 \\\n    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \\\n    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67\njoblib==1.5.2 \\\n    --hash=sha256:3faa5c39054b2f03ca547da9b2f52fde67c06240c31853f306aea97f13647b55 \\\n    --hash=sha256:4e1f0bdbb987e6d843c70cf43714cb276623def372df3c22fe5266b2670bc241\nkiwisolver==1.4.9 \\\n    --hash=sha256:0749fd8f4218ad2e851e11cc4dc05c7cbc0cbc4267bdfdb31782e65aace4ee9c \\\n    --hash=sha256:0763515d4df10edf6d06a3c19734e2566368980d21ebec439f33f9eb936c07b7 \\\n    --hash=sha256:0856e241c2d3df4efef7c04a1e46b1936b6120c9bcf36dd216e3acd84bc4fb21 \\\n    --hash=sha256:0a590506f303f512dff6b7f75fd2fd18e16943efee932008fe7140e5fa91d80e \\\n    --hash=sha256:0ab74e19f6a2b027ea4f845a78827969af45ce790e6cb3e1ebab71bdf9f215ff \\\n    --hash=sha256:0ae37737256ba2de764ddc12aed4956460277f00c4996d51a197e72f62f5eec7 \\\n    --hash=sha256:0e4e2bf29574a6a7b7f6cb5fa69293b9f96c928949ac4a53ba3f525dffb87f9c \\\n    --hash=sha256:15163165efc2f627eb9687ea5f3a28137217d217ac4024893d753f46bce9de26 \\\n    --hash=sha256:17680d737d5335b552994a2008fab4c851bcd7de33094a82067ef3a576ff02fa \\\n    --hash=sha256:1a12cf6398e8a0a001a059747a1cbf24705e18fe413bc22de7b3d15c67cffe3f \\\n    --hash=sha256:1b11d6a633e4ed84fc0ddafd4ebfd8ea49b3f25082c04ad12b8315c11d504dc1 \\\n    --hash=sha256:1fa333e8b2ce4d9660f2cda9c0e1b6bafcfb2457a9d259faa82289e73ec24891 \\\n    --hash=sha256:2327a4a30d3ee07d2fbe2e7933e8a37c591663b96ce42a00bc67461a87d7df77 \\\n    --hash=sha256:2405a7d98604b87f3fc28b1716783534b1b4b8510d8142adca34ee0bc3c87543 \\\n    --hash=sha256:2489e4e5d7ef9a1c300a5e0196e43d9c739f066ef23270607d45aba368b91f2d \\\n    --hash=sha256:24c175051354f4a28c5d6a31c93906dc653e2bf234e8a4bbfb964892078898ce \\\n    --hash=sha256:2635d352d67458b66fd0667c14cb1d4145e9560d503219034a18a87e971ce4f3 \\\n    --hash=sha256:2c1a4f57df73965f3f14df20b80ee29e6a7930a57d2d9e8491a25f676e197c60 \\\n    --hash=sha256:2c93f00dcba2eea70af2be5f11a830a742fe6b579a1d4e00f47760ef13be247a \\\n    --hash=sha256:39a219e1c81ae3b103643d2aedb90f1ef22650deb266ff12a19e7773f3e5f089 \\\n    --hash=sha256:3b3115b2581ea35bb6d1f24a4c90af37e5d9b49dcff267eeed14c3893c5b86ab \\\n    --hash=sha256:40092754720b174e6ccf9e845d0d8c7d8e12c3d71e7fc35f55f3813e96376f78 \\\n    --hash=sha256:412f287c55a6f54b0650bd9b6dce5aceddb95864a1a90c87af16979d37c89771 \\\n    --hash=sha256:464415881e4801295659462c49461a24fb107c140de781d55518c4b80cb6790f \\\n    --hash=sha256:497d05f29a1300d14e02e6441cf0f5ee81c1ff5a304b0d9fb77423974684e08b \\\n    --hash=sha256:4a2899935e724dd1074cb568ce7ac0dce28b2cd6ab539c8e001a8578eb106d14 \\\n    --hash=sha256:4a48a2ce79d65d363597ef7b567ce3d14d68783d2b2263d98db3d9477805ba32 \\\n    --hash=sha256:4d1d9e582ad4d63062d34077a9a1e9f3c34088a2ec5135b1f7190c07cf366527 \\\n    --hash=sha256:52a15b0f35dad39862d376df10c5230155243a2c1a436e39eb55623ccbd68185 \\\n    --hash=sha256:540c7c72324d864406a009d72f5d6856f49693db95d1fbb46cf86febef873634 \\\n    --hash=sha256:5656aa670507437af0207645273ccdfee4f14bacd7f7c67a4306d0dcaeaf6eed \\\n    --hash=sha256:5a0f2724dfd4e3b3ac5a82436a8e6fd16baa7d507117e4279b660fe8ca38a3a1 \\\n    --hash=sha256:60c439763a969a6af93b4881db0eed8fadf93ee98e18cbc35bc8da868d0c4f0c \\\n    --hash=sha256:61874cdb0a36016354853593cffc38e56fc9ca5aa97d2c05d3dcf6922cd55a11 \\\n    --hash=sha256:67bb8b474b4181770f926f7b7d2f8c0248cbcb78b660fdd41a47054b28d2a752 \\\n    --hash=sha256:720e05574713db64c356e86732c0f3c5252818d05f9df320f0ad8380641acea5 \\\n    --hash=sha256:72d0eb9fba308b8311685c2268cf7d0a0639a6cd027d8128659f72bdd8a024b4 \\\n    --hash=sha256:767c23ad1c58c9e827b649a9ab7809fd5fd9db266a9cf02b0e926ddc2c680d58 \\\n    --hash=sha256:77937e5e2a38a7b48eef0585114fe7930346993a88060d0bf886086d2aa49ef5 \\\n    --hash=sha256:7a08b491ec91b1d5053ac177afe5290adacf1f0f6307d771ccac5de30592d198 \\\n    --hash=sha256:7b4da0d01ac866a57dd61ac258c5607b4cd677f63abaec7b148354d2b2cdd536 \\\n    --hash=sha256:7cf974dd4e35fa315563ac99d6287a1024e4dc2077b8a7d7cd3d2fb65d283134 \\\n    --hash=sha256:84fd60810829c27ae375114cd379da1fa65e6918e1da405f356a775d49a62bcf \\\n    --hash=sha256:858e4c22fb075920b96a291928cb7dea5644e94c0ee4fcd5af7e865655e4ccf2 \\\n    --hash=sha256:85b5352f94e490c028926ea567fc569c52ec79ce131dadb968d3853e809518c2 \\\n    --hash=sha256:85bd218b5ecfbee8c8a82e121802dcb519a86044c9c3b2e4aef02fa05c6da370 \\\n    --hash=sha256:8a1f570ce4d62d718dce3f179ee78dac3b545ac16c0c04bb363b7607a949c0d1 \\\n    --hash=sha256:8fdca1def57a2e88ef339de1737a1449d6dbf5fab184c54a1fca01d541317154 \\\n    --hash=sha256:90f47e70293fc3688b71271100a1a5453aa9944a81d27ff779c108372cf5567b \\\n    --hash=sha256:92a2f997387a1b79a75e7803aa7ded2cfbe2823852ccf1ba3bcf613b62ae3197 \\\n    --hash=sha256:9928fe1eb816d11ae170885a74d074f57af3a0d65777ca47e9aeb854a1fba386 \\\n    --hash=sha256:9af39d6551f97d31a4deebeac6f45b156f9755ddc59c07b402c148f5dbb6482a \\\n    --hash=sha256:9cf554f21be770f5111a1690d42313e140355e687e05cf82cb23d0a721a64a48 \\\n    --hash=sha256:a30fd6fdef1430fd9e1ba7b3398b5ee4e2887783917a687d86ba69985fb08748 \\\n    --hash=sha256:a31d512c812daea6d8b3be3b2bfcbeb091dbb09177706569bcfc6240dcf8b41c \\\n    --hash=sha256:a5d0432ccf1c7ab14f9949eec60c5d1f924f17c037e9f8b33352fa05799359b8 \\\n    --hash=sha256:a60ea74330b91bd22a29638940d115df9dc00af5035a9a2a6ad9399ffb4ceca5 \\\n    --hash=sha256:ac5a486ac389dddcc5bef4f365b6ae3ffff2c433324fb38dd35e3fab7c957999 \\\n    --hash=sha256:aedff62918805fb62d43a4aa2ecd4482c380dc76cd31bd7c8878588a61bd0369 \\\n    --hash=sha256:b34e51affded8faee0dfdb705416153819d8ea9250bbbf7ea1b249bdeb5f1122 \\\n    --hash=sha256:b4b4d74bda2b8ebf4da5bd42af11d02d04428b2c32846e4c2c93219df8a7987b \\\n    --hash=sha256:b67e6efbf68e077dd71d1a6b37e43e1a99d0bff1a3d51867d45ee8908b931098 \\\n    --hash=sha256:b78efa4c6e804ecdf727e580dbb9cba85624d2e1c6b5cb059c66290063bd99a9 \\\n    --hash=sha256:bb4ae2b57fc1d8cbd1cf7b1d9913803681ffa903e7488012be5b76dedf49297f \\\n    --hash=sha256:bdd1a81a1860476eb41ac4bc1e07b3f07259e6d55bbf739b79c8aaedcf512799 \\\n    --hash=sha256:bdee92c56a71d2b24c33a7d4c2856bd6419d017e08caa7802d2963870e315028 \\\n    --hash=sha256:be6a04e6c79819c9a8c2373317d19a96048e5a3f90bec587787e86a1153883c2 \\\n    --hash=sha256:bfc08add558155345129c7803b3671cf195e6a56e7a12f3dde7c57d9b417f525 \\\n    --hash=sha256:c3b22c26c6fd6811b0ae8363b95ca8ce4ea3c202d3d0975b2914310ceb1bcc4d \\\n    --hash=sha256:c9e7cdf45d594ee04d5be1b24dd9d49f3d1590959b2271fb30b5ca2b262c00fb \\\n    --hash=sha256:cb27e7b78d716c591e88e0a09a2139c6577865d7f2e152488c2cc6257f460872 \\\n    --hash=sha256:cc9617b46837c6468197b5945e196ee9ca43057bb7d9d1ae688101e4e1dddf64 \\\n    --hash=sha256:ccd09f20ccdbbd341b21a67ab50a119b64a403b09288c27481575105283c1586 \\\n    --hash=sha256:ce6a3a4e106cf35c2d9c4fa17c05ce0b180db622736845d4315519397a77beaf \\\n    --hash=sha256:d0005b053977e7b43388ddec89fa567f43d4f6d5c2c0affe57de5ebf290dc552 \\\n    --hash=sha256:d4188e73af84ca82468f09cadc5ac4db578109e52acb4518d8154698d3a87ca2 \\\n    --hash=sha256:d4efec7bcf21671db6a3294ff301d2fc861c31faa3c8740d1a94689234d1b415 \\\n    --hash=sha256:d75aa530ccfaa593da12834b86a0724f58bff12706659baa9227c2ccaa06264c \\\n    --hash=sha256:d84cd4061ae292d8ac367b2c3fa3aad11cb8625a95d135fe93f286f914f3f5a6 \\\n    --hash=sha256:d8aacd3d4b33b772542b2e01beb50187536967b514b00003bdda7589722d2a64 \\\n    --hash=sha256:d8fc5c867c22b828001b6a38d2eaeb88160bf5783c6cb4a5e440efc981ce286d \\\n    --hash=sha256:d976bbb382b202f71c67f77b0ac11244021cfa3f7dfd9e562eefcea2df711548 \\\n    --hash=sha256:dba5ee5d3981160c28d5490f0d1b7ed730c22470ff7f6cc26cfcfaacb9896a07 \\\n    --hash=sha256:dc1ae486f9abcef254b5618dfb4113dd49f94c68e3e027d03cf0143f3f772b61 \\\n    --hash=sha256:dd0a578400839256df88c16abddf9ba14813ec5f21362e1fe65022e00c883d4d \\\n    --hash=sha256:deed0c7258ceb4c44ad5ec7d9918f9f14fd05b2be86378d86cf50e63d1e7b771 \\\n    --hash=sha256:e09c2279a4d01f099f52d5c4b3d9e208e91edcbd1a175c9662a8b16e000fece9 \\\n    --hash=sha256:e2ea9f7ab7fbf18fffb1b5434ce7c69a07582f7acc7717720f1d69f3e806f90c \\\n    --hash=sha256:e6b93f13371d341afee3be9f7c5964e3fe61d5fa30f6a30eb49856935dfe4fc3 \\\n    --hash=sha256:eb14a5da6dc7642b0f3a18f13654847cd8b7a2550e2645a5bda677862b03ba16 \\\n    --hash=sha256:ed0fecd28cc62c54b262e3736f8bb2512d8dcfdc2bcf08be5f47f96bf405b145 \\\n    --hash=sha256:ede8c6d533bc6601a47ad4046080d36b8fc99f81e6f1c17b0ac3c2dc91ac7611 \\\n    --hash=sha256:efb3a45b35622bb6c16dbfab491a8f5a391fe0e9d45ef32f4df85658232ca0e2 \\\n    --hash=sha256:f117e1a089d9411663a3207ba874f31be9ac8eaa5b533787024dc07aeb74f464 \\\n    --hash=sha256:f2ba92255faa7309d06fe44c3a4a97efe1c8d640c2a79a5ef728b685762a6fd2 \\\n    --hash=sha256:f6008a4919fdbc0b0097089f67a1eb55d950ed7e90ce2cc3e640abadd2757a04 \\\n    --hash=sha256:f68208a520c3d86ea51acf688a3e3002615a7f0238002cccc17affecc86a8a54 \\\n    --hash=sha256:f68e4f3eeca8fb22cc3d731f9715a13b652795ef657a13df1ad0c7dc0e9731df \\\n    --hash=sha256:fb3b8132019ea572f4611d770991000d7f58127560c4889729248eb5852a102f \\\n    --hash=sha256:fb940820c63a9590d31d88b815e7a3aa5915cad3ce735ab45f0c730b39547de1 \\\n    --hash=sha256:fc1795ac5cd0510207482c3d1d3ed781143383b8cfd36f5c645f3897ce066220\nlazy-loader==0.4 \\\n    --hash=sha256:342aa8e14d543a154047afb4ba8ef17f5563baad3fc610d7b15b213b0f119efc \\\n    --hash=sha256:47c75182589b91a4e1a85a136c074285a5ad4d9f39c63e0d7fb76391c4574cd1\nmarkupsafe==3.0.2 \\\n    --hash=sha256:0bff5e0ae4ef2e1ae4fdf2dfd5b76c75e5c2fa4132d05fc1b0dabcd20c7e28c4 \\\n    --hash=sha256:0f4ca02bea9a23221c0182836703cbf8930c5e9454bacce27e767509fa286a30 \\\n    --hash=sha256:1225beacc926f536dc82e45f8a4d68502949dc67eea90eab715dea3a21c1b5f0 \\\n    --hash=sha256:131a3c7689c85f5ad20f9f6fb1b866f402c445b220c19fe4308c0b147ccd2ad9 \\\n    --hash=sha256:15ab75ef81add55874e7ab7055e9c397312385bd9ced94920f2802310c930396 \\\n    --hash=sha256:1a9d3f5f0901fdec14d8d2f66ef7d035f2157240a433441719ac9a3fba440b13 \\\n    --hash=sha256:1c99d261bd2d5f6b59325c92c73df481e05e57f19837bdca8413b9eac4bd8028 \\\n    --hash=sha256:1e084f686b92e5b83186b07e8a17fc09e38fff551f3602b249881fec658d3eca \\\n    --hash=sha256:2181e67807fc2fa785d0592dc2d6206c019b9502410671cc905d132a92866557 \\\n    --hash=sha256:2cb8438c3cbb25e220c2ab33bb226559e7afb3baec11c4f218ffa7308603c832 \\\n    --hash=sha256:3169b1eefae027567d1ce6ee7cae382c57fe26e82775f460f0b2778beaad66c0 \\\n    --hash=sha256:3809ede931876f5b2ec92eef964286840ed3540dadf803dd570c3b7e13141a3b \\\n    --hash=sha256:38a9ef736c01fccdd6600705b09dc574584b89bea478200c5fbf112a6b0d5579 \\\n    --hash=sha256:3d79d162e7be8f996986c064d1c7c817f6df3a77fe3d6859f6f9e7be4b8c213a \\\n    --hash=sha256:444dcda765c8a838eaae23112db52f1efaf750daddb2d9ca300bcae1039adc5c \\\n    --hash=sha256:48032821bbdf20f5799ff537c7ac3d1fba0ba032cfc06194faffa8cda8b560ff \\\n    --hash=sha256:4aa4e5faecf353ed117801a068ebab7b7e09ffb6e1d5e412dc852e0da018126c \\\n    --hash=sha256:52305740fe773d09cffb16f8ed0427942901f00adedac82ec8b67752f58a1b22 \\\n    --hash=sha256:569511d3b58c8791ab4c2e1285575265991e6d8f8700c7be0e88f86cb0672094 \\\n    --hash=sha256:57cb5a3cf367aeb1d316576250f65edec5bb3be939e9247ae594b4bcbc317dfb \\\n    --hash=sha256:5b02fb34468b6aaa40dfc198d813a641e3a63b98c2b05a16b9f80b7ec314185e \\\n    --hash=sha256:6381026f158fdb7c72a168278597a5e3a5222e83ea18f543112b2662a9b699c5 \\\n    --hash=sha256:6af100e168aa82a50e186c82875a5893c5597a0c1ccdb0d8b40240b1f28b969a \\\n    --hash=sha256:6c89876f41da747c8d3677a2b540fb32ef5715f97b66eeb0c6b66f5e3ef6f59d \\\n    --hash=sha256:6e296a513ca3d94054c2c881cc913116e90fd030ad1c656b3869762b754f5f8a \\\n    --hash=sha256:70a87b411535ccad5ef2f1df5136506a10775d267e197e4cf531ced10537bd6b \\\n    --hash=sha256:7e94c425039cde14257288fd61dcfb01963e658efbc0ff54f5306b06054700f8 \\\n    --hash=sha256:846ade7b71e3536c4e56b386c2a47adf5741d2d8b94ec9dc3e92e5e1ee1e2225 \\\n    --hash=sha256:88416bd1e65dcea10bc7569faacb2c20ce071dd1f87539ca2ab364bf6231393c \\\n    --hash=sha256:88b49a3b9ff31e19998750c38e030fc7bb937398b1f78cfa599aaef92d693144 \\\n    --hash=sha256:8c4e8c3ce11e1f92f6536ff07154f9d49677ebaaafc32db9db4620bc11ed480f \\\n    --hash=sha256:8e06879fc22a25ca47312fbe7c8264eb0b662f6db27cb2d3bbbc74b1df4b9b87 \\\n    --hash=sha256:9025b4018f3a1314059769c7bf15441064b2207cb3f065e6ea1e7359cb46db9d \\\n    --hash=sha256:93335ca3812df2f366e80509ae119189886b0f3c2b81325d39efdb84a1e2ae93 \\\n    --hash=sha256:9778bd8ab0a994ebf6f84c2b949e65736d5575320a17ae8984a77fab08db94cf \\\n    --hash=sha256:9e2d922824181480953426608b81967de705c3cef4d1af983af849d7bd619158 \\\n    --hash=sha256:a123e330ef0853c6e822384873bef7507557d8e4a082961e1defa947aa59ba84 \\\n    --hash=sha256:a904af0a6162c73e3edcb969eeeb53a63ceeb5d8cf642fade7d39e7963a22ddb \\\n    --hash=sha256:ad10d3ded218f1039f11a75f8091880239651b52e9bb592ca27de44eed242a48 \\\n    --hash=sha256:b424c77b206d63d500bcb69fa55ed8d0e6a3774056bdc4839fc9298a7edca171 \\\n    --hash=sha256:b5a6b3ada725cea8a5e634536b1b01c30bcdcd7f9c6fff4151548d5bf6b3a36c \\\n    --hash=sha256:ba8062ed2cf21c07a9e295d5b8a2a5ce678b913b45fdf68c32d95d6c1291e0b6 \\\n    --hash=sha256:ba9527cdd4c926ed0760bc301f6728ef34d841f405abf9d4f959c478421e4efd \\\n    --hash=sha256:bbcb445fa71794da8f178f0f6d66789a28d7319071af7a496d4d507ed566270d \\\n    --hash=sha256:bcf3e58998965654fdaff38e58584d8937aa3096ab5354d493c77d1fdd66d7a1 \\\n    --hash=sha256:c0ef13eaeee5b615fb07c9a7dadb38eac06a0608b41570d8ade51c56539e509d \\\n    --hash=sha256:cabc348d87e913db6ab4aa100f01b08f481097838bdddf7c7a84b7575b7309ca \\\n    --hash=sha256:cdb82a876c47801bb54a690c5ae105a46b392ac6099881cdfb9f6e95e4014c6a \\\n    --hash=sha256:cfad01eed2c2e0c01fd0ecd2ef42c492f7f93902e39a42fc9ee1692961443a29 \\\n    --hash=sha256:d16a81a06776313e817c951135cf7340a3e91e8c1ff2fac444cfd75fffa04afe \\\n    --hash=sha256:d8213e09c917a951de9d09ecee036d5c7d36cb6cb7dbaece4c71a60d79fb9798 \\\n    --hash=sha256:e07c3764494e3776c602c1e78e298937c3315ccc9043ead7e685b7f2b8d47b3c \\\n    --hash=sha256:e17c96c14e19278594aa4841ec148115f9c7615a47382ecb6b82bd8fea3ab0c8 \\\n    --hash=sha256:e444a31f8db13eb18ada366ab3cf45fd4b31e4db1236a4448f68778c1d1a5a2f \\\n    --hash=sha256:e6a2a455bd412959b57a172ce6328d2dd1f01cb2135efda2e4576e8a23fa3b0f \\\n    --hash=sha256:eaa0a10b7f72326f1372a713e73c3f739b524b3af41feb43e4921cb529f5929a \\\n    --hash=sha256:eb7972a85c54febfb25b5c4b4f3af4dcc731994c7da0d8a0b4a6eb0640e1d178 \\\n    --hash=sha256:ee55d3edf80167e48ea11a923c7386f4669df67d7994554387f84e7d8b0a2bf0 \\\n    --hash=sha256:f3818cb119498c0678015754eba762e0d61e5b52d34c8b13d770f0719f7b1d79 \\\n    --hash=sha256:f8b3d067f2e40fe93e1ccdd6b2e1d16c43140e76f02fb1319a05cf2b79d99430 \\\n    --hash=sha256:fcabf5ff6eea076f859677f5f0b6b5c1a51e70a376b0579e0eadef8db48c6b50\nmatplotlib==3.10.1 \\\n    --hash=sha256:01e63101ebb3014e6e9f80d9cf9ee361a8599ddca2c3e166c563628b39305dbb \\\n    --hash=sha256:02582304e352f40520727984a5a18f37e8187861f954fea9be7ef06569cf85b4 \\\n    --hash=sha256:057206ff2d6ab82ff3e94ebd94463d084760ca682ed5f150817b859372ec4401 \\\n    --hash=sha256:0721a3fd3d5756ed593220a8b86808a36c5031fce489adb5b31ee6dbb47dd5b2 \\\n    --hash=sha256:0f69dc9713e4ad2fb21a1c30e37bd445d496524257dfda40ff4a8efb3604ab5c \\\n    --hash=sha256:11b65088c6f3dae784bc72e8d039a2580186285f87448babb9ddb2ad0082993a \\\n    --hash=sha256:1985ad3d97f51307a2cbfc801a930f120def19ba22864182dacef55277102ba6 \\\n    --hash=sha256:19b06241ad89c3ae9469e07d77efa87041eac65d78df4fcf9cac318028009b01 \\\n    --hash=sha256:2589659ea30726284c6c91037216f64a506a9822f8e50592d48ac16a2f29e044 \\\n    --hash=sha256:35e87384ee9e488d8dd5a2dd7baf471178d38b90618d8ea147aced4ab59c9bea \\\n    --hash=sha256:3f06bad951eea6422ac4e8bdebcf3a70c59ea0a03338c5d2b109f57b64eb3972 \\\n    --hash=sha256:4c59af3e8aca75d7744b68e8e78a669e91ccbcf1ac35d0102a7b1b46883f1dd7 \\\n    --hash=sha256:4f0647b17b667ae745c13721602b540f7aadb2a32c5b96e924cd4fea5dcb90f1 \\\n    --hash=sha256:56c5d9fcd9879aa8040f196a235e2dcbdf7dd03ab5b07c0696f80bc6cf04bedd \\\n    --hash=sha256:5d45d3f5245be5b469843450617dcad9af75ca50568acf59997bed9311131a0b \\\n    --hash=sha256:648406f1899f9a818cef8c0231b44dcfc4ff36f167101c3fd1c9151f24220fdc \\\n    --hash=sha256:66e907a06e68cb6cfd652c193311d61a12b54f56809cafbed9736ce5ad92f107 \\\n    --hash=sha256:7e496c01441be4c7d5f96d4e40f7fca06e20dcb40e44c8daa2e740e1757ad9e6 \\\n    --hash=sha256:8e875b95ac59a7908978fe307ecdbdd9a26af7fa0f33f474a27fcf8c99f64a19 \\\n    --hash=sha256:8e8e25b1209161d20dfe93037c8a7f7ca796ec9aa326e6e4588d8c4a5dd1e473 \\\n    --hash=sha256:a144867dd6bf8ba8cb5fc81a158b645037e11b3e5cf8a50bd5f9917cb863adfe \\\n    --hash=sha256:a3dfb036f34873b46978f55e240cff7a239f6c4409eac62d8145bad3fc6ba5a3 \\\n    --hash=sha256:a97ff127f295817bc34517255c9db6e71de8eddaab7f837b7d341dee9f2f587f \\\n    --hash=sha256:aa3854b5f9473564ef40a41bc922be978fab217776e9ae1545c9b3a5cf2092a3 \\\n    --hash=sha256:bc411ebd5889a78dabbc457b3fa153203e22248bfa6eedc6797be5df0164dbf9 \\\n    --hash=sha256:c42eee41e1b60fd83ee3292ed83a97a5f2a8239b10c26715d8a6172226988d7b \\\n    --hash=sha256:c96f2c2f825d1257e437a1482c5a2cf4fee15db4261bd6fc0750f81ba2b4ba3d \\\n    --hash=sha256:cfd414bce89cc78a7e1d25202e979b3f1af799e416010a20ab2b5ebb3a02425c \\\n    --hash=sha256:d0673b4b8f131890eb3a1ad058d6e065fb3c6e71f160089b65f8515373394698 \\\n    --hash=sha256:d3809916157ba871bcdd33d3493acd7fe3037db5daa917ca6e77975a94cef779 \\\n    --hash=sha256:dc6ab14a7ab3b4d813b88ba957fc05c79493a037f54e246162033591e770de6f \\\n    --hash=sha256:e8d2d0e3881b129268585bf4765ad3ee73a4591d77b9a18c214ac7e3a79fb2ba \\\n    --hash=sha256:e9b4bb156abb8fa5e5b2b460196f7db7264fc6d62678c03457979e7d5254b7be \\\n    --hash=sha256:ff2ae14910be903f4a24afdbb6d7d3a6c44da210fc7d42790b87aeac92238a16\nmatplotlib-inline==0.1.7 \\\n    --hash=sha256:8423b23ec666be3d16e16b60bdd8ac4e86e840ebd1dd11a30b9f117f2fa0ab90 \\\n    --hash=sha256:df192d39a4ff8f21b1895d72e6a13f5fcc5099f00fa84384e0ea28c2cc0653ca\nmediapy==1.2.2 \\\n    --hash=sha256:2b159c385120c6eca9c88ccf96a19fb2b5b5937c788cf430db637d27270618ba \\\n    --hash=sha256:42d9a1aa93c183550b824dbb4f0de5da61aa5c84db8f01f063acd1f23b90ef0a\nmpmath==1.3.0 \\\n    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \\\n    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c\nnetworkx==3.5 \\\n    --hash=sha256:0030d386a9a06dee3565298b4a734b68589749a544acbb6c412dc9e2489ec6ec \\\n    --hash=sha256:d4c6f9cf81f52d69230866796b82afbccdec3db7ae4fbd1b65ea750feed50037\nnumpy==2.0.1 \\\n    --hash=sha256:08458fbf403bff5e2b45f08eda195d4b0c9b35682311da5a5a0a0925b11b9bd8 \\\n    --hash=sha256:0fbb536eac80e27a2793ffd787895242b7f18ef792563d742c2d673bfcb75134 \\\n    --hash=sha256:12f5d865d60fb9734e60a60f1d5afa6d962d8d4467c120a1c0cda6eb2964437d \\\n    --hash=sha256:15eb4eca47d36ec3f78cde0a3a2ee24cf05ca7396ef808dda2c0ddad7c2bde67 \\\n    --hash=sha256:173a00b9995f73b79eb0191129f2455f1e34c203f559dd118636858cc452a1bf \\\n    --hash=sha256:1b902ce0e0a5bb7704556a217c4f63a7974f8f43e090aff03fcf262e0b135e02 \\\n    --hash=sha256:1f682ea61a88479d9498bf2091fdcd722b090724b08b31d63e022adc063bad59 \\\n    --hash=sha256:1f87fec1f9bc1efd23f4227becff04bd0e979e23ca50cc92ec88b38489db3b55 \\\n    --hash=sha256:24a0e1befbfa14615b49ba9659d3d8818a0f4d8a1c5822af8696706fbda7310c \\\n    --hash=sha256:2c3a346ae20cfd80b6cfd3e60dc179963ef2ea58da5ec074fd3d9e7a1e7ba97f \\\n    --hash=sha256:36d3a9405fd7c511804dc56fc32974fa5533bdeb3cd1604d6b8ff1d292b819c4 \\\n    --hash=sha256:3fdabe3e2a52bc4eff8dc7a5044342f8bd9f11ef0934fcd3289a788c0eb10018 \\\n    --hash=sha256:4127d4303b9ac9f94ca0441138acead39928938660ca58329fe156f84b9f3015 \\\n    --hash=sha256:4658c398d65d1b25e1760de3157011a80375da861709abd7cef3bad65d6543f9 \\\n    --hash=sha256:485b87235796410c3519a699cfe1faab097e509e90ebb05dcd098db2ae87e7b3 \\\n    --hash=sha256:529af13c5f4b7a932fb0e1911d3a75da204eff023ee5e0e79c1751564221a5c8 \\\n    --hash=sha256:5a3d94942c331dd4e0e1147f7a8699a4aa47dffc11bf8a1523c12af8b2e91bbe \\\n    --hash=sha256:5daab361be6ddeb299a918a7c0864fa8618af66019138263247af405018b04e1 \\\n    --hash=sha256:61728fba1e464f789b11deb78a57805c70b2ed02343560456190d0501ba37b0f \\\n    --hash=sha256:6790654cb13eab303d8402354fabd47472b24635700f631f041bd0b65e37298a \\\n    --hash=sha256:69ff563d43c69b1baba77af455dd0a839df8d25e8590e79c90fcbe1499ebde42 \\\n    --hash=sha256:6bf4e6f4a2a2e26655717a1983ef6324f2664d7011f6ef7482e8c0b3d51e82ac \\\n    --hash=sha256:6e4eeb6eb2fced786e32e6d8df9e755ce5be920d17f7ce00bc38fcde8ccdbf9e \\\n    --hash=sha256:72dc22e9ec8f6eaa206deb1b1355eb2e253899d7347f5e2fae5f0af613741d06 \\\n    --hash=sha256:75b4e316c5902d8163ef9d423b1c3f2f6252226d1aa5cd8a0a03a7d01ffc6268 \\\n    --hash=sha256:7b9853803278db3bdcc6cd5beca37815b133e9e77ff3d4733c247414e78eb8d1 \\\n    --hash=sha256:7d6fddc5fe258d3328cd8e3d7d3e02234c5d70e01ebe377a6ab92adb14039cb4 \\\n    --hash=sha256:81b0893a39bc5b865b8bf89e9ad7807e16717f19868e9d234bdaf9b1f1393868 \\\n    --hash=sha256:8efc84f01c1cd7e34b3fb310183e72fcdf55293ee736d679b6d35b35d80bba26 \\\n    --hash=sha256:8fae4ebbf95a179c1156fab0b142b74e4ba4204c87bde8d3d8b6f9c34c5825ef \\\n    --hash=sha256:99d0d92a5e3613c33a5f01db206a33f8fdf3d71f2912b0de1739894668b7a93b \\\n    --hash=sha256:9adbd9bb520c866e1bfd7e10e1880a1f7749f1f6e5017686a5fbb9b72cf69f82 \\\n    --hash=sha256:a1e01dcaab205fbece13c1410253a9eea1b1c9b61d237b6fa59bcc46e8e89343 \\\n    --hash=sha256:a8fc2de81ad835d999113ddf87d1ea2b0f4704cbd947c948d2f5513deafe5a7b \\\n    --hash=sha256:b83e16a5511d1b1f8a88cbabb1a6f6a499f82c062a4251892d9ad5d609863fb7 \\\n    --hash=sha256:bb2124fdc6e62baae159ebcfa368708867eb56806804d005860b6007388df171 \\\n    --hash=sha256:bfc085b28d62ff4009364e7ca34b80a9a080cbd97c2c0630bb5f7f770dae9414 \\\n    --hash=sha256:cbab9fc9c391700e3e1287666dfd82d8666d10e69a6c4a09ab97574c0b7ee0a7 \\\n    --hash=sha256:e5eeca8067ad04bc8a2a8731183d51d7cbaac66d86085d5f4766ee6bf19c7f87 \\\n    --hash=sha256:e9e81fa9017eaa416c056e5d9e71be93d05e2c3c2ab308d23307a8bc4443c368 \\\n    --hash=sha256:ea2326a4dca88e4a274ba3a4405eb6c6467d3ffbd8c7d38632502eaae3820587 \\\n    --hash=sha256:eacf3291e263d5a67d8c1a581a8ebbcfd6447204ef58828caf69a5e3e8c75990 \\\n    --hash=sha256:ec87f5f8aca726117a1c9b7083e7656a9d0d606eec7299cc067bb83d26f16e0c \\\n    --hash=sha256:f1659887361a7151f89e79b276ed8dff3d75877df906328f14d8bb40bb4f5101 \\\n    --hash=sha256:f9cf5ea551aec449206954b075db819f52adc1638d46a6738253a712d553c7b4\nnvidia-cublas-cu12==12.4.5.8 \\\n    --hash=sha256:0f8aa1706812e00b9f19dfe0cdb3999b092ccb8ca168c0db5b8ea712456fd9b3 \\\n    --hash=sha256:2fc8da60df463fdefa81e323eef2e36489e1c94335b5358bcb38360adf75ac9b \\\n    --hash=sha256:5a796786da89203a0657eda402bcdcec6180254a8ac22d72213abc42069522dc\nnvidia-cuda-cupti-cu12==12.4.127 \\\n    --hash=sha256:5688d203301ab051449a2b1cb6690fbe90d2b372f411521c86018b950f3d7922 \\\n    --hash=sha256:79279b35cf6f91da114182a5ce1864997fd52294a87a16179ce275773799458a \\\n    --hash=sha256:9dec60f5ac126f7bb551c055072b69d85392b13311fcc1bcda2202d172df30fb\nnvidia-cuda-nvrtc-cu12==12.4.127 \\\n    --hash=sha256:0eedf14185e04b76aa05b1fea04133e59f465b6f960c0cbf4e37c3cb6b0ea198 \\\n    --hash=sha256:a178759ebb095827bd30ef56598ec182b85547f1508941a3d560eb7ea1fbf338 \\\n    --hash=sha256:a961b2f1d5f17b14867c619ceb99ef6fcec12e46612711bcec78eb05068a60ec\nnvidia-cuda-runtime-cu12==12.4.127 \\\n    --hash=sha256:09c2e35f48359752dfa822c09918211844a3d93c100a715d79b59591130c5e1e \\\n    --hash=sha256:64403288fa2136ee8e467cdc9c9427e0434110899d07c779f25b5c068934faa5 \\\n    --hash=sha256:961fe0e2e716a2a1d967aab7caee97512f71767f852f67432d572e36cb3a11f3\nnvidia-cudnn-cu12==9.1.0.70 \\\n    --hash=sha256:165764f44ef8c61fcdfdfdbe769d687e06374059fbb388b6c89ecb0e28793a6f \\\n    --hash=sha256:6278562929433d68365a07a4a1546c237ba2849852c0d4b2262a486e805b977a\nnvidia-cufft-cu12==11.2.1.3 \\\n    --hash=sha256:5dad8008fc7f92f5ddfa2101430917ce2ffacd86824914c82e28990ad7f00399 \\\n    --hash=sha256:d802f4954291101186078ccbe22fc285a902136f974d369540fd4a5333d1440b \\\n    --hash=sha256:f083fc24912aa410be21fa16d157fed2055dab1cc4b6934a0e03cba69eb242b9\nnvidia-curand-cu12==10.3.5.147 \\\n    --hash=sha256:1f173f09e3e3c76ab084aba0de819c49e56614feae5c12f69883f4ae9bb5fad9 \\\n    --hash=sha256:a88f583d4e0bb643c49743469964103aa59f7f708d862c3ddb0fc07f851e3b8b \\\n    --hash=sha256:f307cc191f96efe9e8f05a87096abc20d08845a841889ef78cb06924437f6771\nnvidia-cusolver-cu12==11.6.1.9 \\\n    --hash=sha256:19e33fa442bcfd085b3086c4ebf7e8debc07cfe01e11513cc6d332fd918ac260 \\\n    --hash=sha256:d338f155f174f90724bbde3758b7ac375a70ce8e706d70b018dd3375545fc84e \\\n    --hash=sha256:e77314c9d7b694fcebc84f58989f3aa4fb4cb442f12ca1a9bde50f5e8f6d1b9c\nnvidia-cusparse-cu12==12.3.1.170 \\\n    --hash=sha256:9bc90fb087bc7b4c15641521f31c0371e9a612fc2ba12c338d3ae032e6b6797f \\\n    --hash=sha256:9d32f62896231ebe0480efd8a7f702e143c98cfaa0e8a76df3386c1ba2b54df3 \\\n    --hash=sha256:ea4f11a2904e2a8dc4b1833cc1b5181cde564edd0d5cd33e3c168eff2d1863f1\nnvidia-cusparselt-cu12==0.6.2 \\\n    --hash=sha256:0057c91d230703924c0422feabe4ce768841f9b4b44d28586b6f6d2eb86fbe70 \\\n    --hash=sha256:067a7f6d03ea0d4841c85f0c6f1991c5dda98211f6302cb83a4ab234ee95bef8 \\\n    --hash=sha256:df2c24502fd76ebafe7457dbc4716b2fec071aabaed4fb7691a201cde03704d9\nnvidia-nccl-cu12==2.21.5 \\\n    --hash=sha256:8579076d30a8c24988834445f8d633c697d42397e92ffc3f63fa26766d25e0a0\nnvidia-nvjitlink-cu12==12.4.127 \\\n    --hash=sha256:06b3b9b25bf3f8af351d664978ca26a16d2c5127dbd53c0497e28d1fb9611d57 \\\n    --hash=sha256:4abe7fef64914ccfa909bc2ba39739670ecc9e820c83ccc7a6ed414122599b83 \\\n    --hash=sha256:fd9020c501d27d135f983c6d3e244b197a7ccad769e34df53a42e276b0e25fa1\nnvidia-nvtx-cu12==12.4.127 \\\n    --hash=sha256:641dccaaa1139f3ffb0d3164b4b84f9d253397e38246a4f2f36728b48566d485 \\\n    --hash=sha256:781e950d9b9f60d8241ccea575b32f5105a5baf4c2351cab5256a24869f12a1a \\\n    --hash=sha256:7959ad635db13edf4fc65c06a6e9f9e55fc2f92596db928d169c0bb031e88ef3\nomegaconf==2.3.0 \\\n    --hash=sha256:7b4df175cdb08ba400f45cae3bdcae7ba8365db4d165fc65fd04b050ab63b46b \\\n    --hash=sha256:d5d4b6d29955cc50ad50c46dc269bcd92c6e00f5f90d23ab5fee7bfca4ba4cc7\nopencv-python==4.10.0.84 \\\n    --hash=sha256:09a332b50488e2dda866a6c5573ee192fe3583239fb26ff2f7f9ceb0bc119ea6 \\\n    --hash=sha256:2db02bb7e50b703f0a2d50c50ced72e95c574e1e5a0bb35a8a86d0b35c98c236 \\\n    --hash=sha256:32dbbd94c26f611dc5cc6979e6b7aa1f55a64d6b463cc1dcd3c95505a63e48fe \\\n    --hash=sha256:71e575744f1d23f79741450254660442785f45a0797212852ee5199ef12eed98 \\\n    --hash=sha256:72d234e4582e9658ffea8e9cae5b63d488ad06994ef12d81dc303b17472f3526 \\\n    --hash=sha256:9ace140fc6d647fbe1c692bcb2abce768973491222c067c131d80957c595b71f \\\n    --hash=sha256:fc182f8f4cda51b45f01c64e4cbedfc2f00aff799debebc305d8d0210c43f251\npackaging==25.0 \\\n    --hash=sha256:29572ef2b1f17581046b3a2227d5c611fb25ec70ca1ba8554b24b0e69331a484 \\\n    --hash=sha256:d443872c98d677bf60f6a1f2f8c1cb748e8fe762d2bf9d3148b5599295b0fc4f\npandas==2.2.3 \\\n    --hash=sha256:062309c1b9ea12a50e8ce661145c6aab431b1e99530d3cd60640e255778bd43a \\\n    --hash=sha256:15c0e1e02e93116177d29ff83e8b1619c93ddc9c49083f237d4312337a61165d \\\n    --hash=sha256:1948ddde24197a0f7add2bdc4ca83bf2b1ef84a1bc8ccffd95eda17fd836ecb5 \\\n    --hash=sha256:1db71525a1538b30142094edb9adc10be3f3e176748cd7acc2240c2f2e5aa3a4 \\\n    --hash=sha256:22a9d949bfc9a502d320aa04e5d02feab689d61da4e7764b62c30b991c42c5f0 \\\n    --hash=sha256:29401dbfa9ad77319367d36940cd8a0b3a11aba16063e39632d98b0e931ddf32 \\\n    --hash=sha256:31d0ced62d4ea3e231a9f228366919a5ea0b07440d9d4dac345376fd8e1477ea \\\n    --hash=sha256:3508d914817e153ad359d7e069d752cdd736a247c322d932eb89e6bc84217f28 \\\n    --hash=sha256:37e0aced3e8f539eccf2e099f65cdb9c8aa85109b0be6e93e2baff94264bdc6f \\\n    --hash=sha256:381175499d3802cde0eabbaf6324cce0c4f5d52ca6f8c377c29ad442f50f6348 \\\n    --hash=sha256:38cf8125c40dae9d5acc10fa66af8ea6fdf760b2714ee482ca691fc66e6fcb18 \\\n    --hash=sha256:3b71f27954685ee685317063bf13c7709a7ba74fc996b84fc6821c59b0f06468 \\\n    --hash=sha256:3fc6873a41186404dad67245896a6e440baacc92f5b716ccd1bc9ed2995ab2c5 \\\n    --hash=sha256:4850ba03528b6dd51d6c5d273c46f183f39a9baf3f0143e566b89450965b105e \\\n    --hash=sha256:4f18ba62b61d7e192368b84517265a99b4d7ee8912f8708660fb4a366cc82667 \\\n    --hash=sha256:56534ce0746a58afaf7942ba4863e0ef81c9c50d3f0ae93e9497d6a41a057645 \\\n    --hash=sha256:59ef3764d0fe818125a5097d2ae867ca3fa64df032331b7e0917cf5d7bf66b13 \\\n    --hash=sha256:5dbca4c1acd72e8eeef4753eeca07de9b1db4f398669d5994086f788a5d7cc30 \\\n    --hash=sha256:5de54125a92bb4d1c051c0659e6fcb75256bf799a732a87184e5ea503965bce3 \\\n    --hash=sha256:61c5ad4043f791b61dd4752191d9f07f0ae412515d59ba8f005832a532f8736d \\\n    --hash=sha256:6374c452ff3ec675a8f46fd9ab25c4ad0ba590b71cf0656f8b6daa5202bca3fb \\\n    --hash=sha256:63cc132e40a2e084cf01adf0775b15ac515ba905d7dcca47e9a251819c575ef3 \\\n    --hash=sha256:66108071e1b935240e74525006034333f98bcdb87ea116de573a6a0dccb6c039 \\\n    --hash=sha256:6dfcb5ee8d4d50c06a51c2fffa6cff6272098ad6540aed1a76d15fb9318194d8 \\\n    --hash=sha256:7c2875855b0ff77b2a64a0365e24455d9990730d6431b9e0ee18ad8acee13dbd \\\n    --hash=sha256:7eee9e7cea6adf3e3d24e304ac6b8300646e2a5d1cd3a3c2abed9101b0846761 \\\n    --hash=sha256:800250ecdadb6d9c78eae4990da62743b857b470883fa27f652db8bdde7f6659 \\\n    --hash=sha256:86976a1c5b25ae3f8ccae3a5306e443569ee3c3faf444dfd0f41cda24667ad57 \\\n    --hash=sha256:8cd6d7cc958a3910f934ea8dbdf17b2364827bb4dafc38ce6eef6bb3d65ff09c \\\n    --hash=sha256:99df71520d25fade9db7c1076ac94eb994f4d2673ef2aa2e86ee039b6746d20c \\\n    --hash=sha256:a5a1595fe639f5988ba6a8e5bc9649af3baf26df3998a0abe56c02609392e0a4 \\\n    --hash=sha256:ad5b65698ab28ed8d7f18790a0dc58005c7629f227be9ecc1072aa74c0c1d43a \\\n    --hash=sha256:b1d432e8d08679a40e2a6d8b2f9770a5c21793a6f9f47fdd52c5ce1948a5a8a9 \\\n    --hash=sha256:b8661b0238a69d7aafe156b7fa86c44b881387509653fdf857bebc5e4008ad42 \\\n    --hash=sha256:ba96630bc17c875161df3818780af30e43be9b166ce51c9a18c1feae342906c2 \\\n    --hash=sha256:bc6b93f9b966093cb0fd62ff1a7e4c09e6d546ad7c1de191767baffc57628f39 \\\n    --hash=sha256:c124333816c3a9b03fbeef3a9f230ba9a737e9e5bb4060aa2107a86cc0a497fc \\\n    --hash=sha256:cd8d0c3be0515c12fed0bdbae072551c8b54b7192c7b1fda0ba56059a0179698 \\\n    --hash=sha256:d9c45366def9a3dd85a6454c0e7908f2b3b8e9c138f5dc38fed7ce720d8453ed \\\n    --hash=sha256:f00d1345d84d8c86a63e476bb4955e46458b304b9575dcf71102b5c705320015 \\\n    --hash=sha256:f3a255b2c19987fbbe62a9dfd6cff7ff2aa9ccab3fc75218fd4b7530f01efa24 \\\n    --hash=sha256:fffb8ae78d8af97f849404f21411c95062db1496aeb3e56f146f0355c9989319\nparso==0.8.5 \\\n    --hash=sha256:034d7354a9a018bdce352f48b2a8a450f05e9d6ee85db84764e9b6bd96dafe5a \\\n    --hash=sha256:646204b5ee239c396d040b90f9e272e9a8017c630092bf59980beb62fd033887\npexpect==4.9.0 \\\n    --hash=sha256:7236d1e080e4936be2dc3e326cec0af72acf9212a7e1d060210e70a47e253523 \\\n    --hash=sha256:ee7d41123f3c9911050ea2c2dac107568dc43b2d3b0c7557a33212c398ead30f\npillow==11.1.0 \\\n    --hash=sha256:015c6e863faa4779251436db398ae75051469f7c903b043a48f078e437656f83 \\\n    --hash=sha256:0a2f91f8a8b367e7a57c6e91cd25af510168091fb89ec5146003e424e1558a96 \\\n    --hash=sha256:11633d58b6ee5733bde153a8dafd25e505ea3d32e261accd388827ee987baf65 \\\n    --hash=sha256:2062ffb1d36544d42fcaa277b069c88b01bb7298f4efa06731a7fd6cc290b81a \\\n    --hash=sha256:31eba6bbdd27dde97b0174ddf0297d7a9c3a507a8a1480e1e60ef914fe23d352 \\\n    --hash=sha256:3362c6ca227e65c54bf71a5f88b3d4565ff1bcbc63ae72c34b07bbb1cc59a43f \\\n    --hash=sha256:368da70808b36d73b4b390a8ffac11069f8a5c85f29eff1f1b01bcf3ef5b2a20 \\\n    --hash=sha256:36ba10b9cb413e7c7dfa3e189aba252deee0602c86c309799da5a74009ac7a1c \\\n    --hash=sha256:3764d53e09cdedd91bee65c2527815d315c6b90d7b8b79759cc48d7bf5d4f114 \\\n    --hash=sha256:3a5fe20a7b66e8135d7fd617b13272626a28278d0e578c98720d9ba4b2439d49 \\\n    --hash=sha256:3cdcdb0b896e981678eee140d882b70092dac83ac1cdf6b3a60e2216a73f2b91 \\\n    --hash=sha256:4637b88343166249fe8aa94e7c4a62a180c4b3898283bb5d3d2fd5fe10d8e4e0 \\\n    --hash=sha256:4db853948ce4e718f2fc775b75c37ba2efb6aaea41a1a5fc57f0af59eee774b2 \\\n    --hash=sha256:4dd43a78897793f60766563969442020e90eb7847463eca901e41ba186a7d4a5 \\\n    --hash=sha256:54251ef02a2309b5eec99d151ebf5c9904b77976c8abdcbce7891ed22df53884 \\\n    --hash=sha256:54ce1c9a16a9561b6d6d8cb30089ab1e5eb66918cb47d457bd996ef34182922e \\\n    --hash=sha256:593c5fd6be85da83656b93ffcccc2312d2d149d251e98588b14fbc288fd8909c \\\n    --hash=sha256:5bb94705aea800051a743aa4874bb1397d4695fb0583ba5e425ee0328757f196 \\\n    --hash=sha256:67cd427c68926108778a9005f2a04adbd5e67c442ed21d95389fe1d595458756 \\\n    --hash=sha256:70ca5ef3b3b1c4a0812b5c63c57c23b63e53bc38e758b37a951e5bc466449861 \\\n    --hash=sha256:73ddde795ee9b06257dac5ad42fcb07f3b9b813f8c1f7f870f402f4dc54b5269 \\\n    --hash=sha256:758e9d4ef15d3560214cddbc97b8ef3ef86ce04d62ddac17ad39ba87e89bd3b1 \\\n    --hash=sha256:7d33d2fae0e8b170b6a6c57400e077412240f6f5bb2a342cf1ee512a787942bb \\\n    --hash=sha256:7fdadc077553621911f27ce206ffcbec7d3f8d7b50e0da39f10997e8e2bb7f6a \\\n    --hash=sha256:8000376f139d4d38d6851eb149b321a52bb8893a88dae8ee7d95840431977081 \\\n    --hash=sha256:837060a8599b8f5d402e97197d4924f05a2e0d68756998345c829c33186217b1 \\\n    --hash=sha256:89dbdb3e6e9594d512780a5a1c42801879628b38e3efc7038094430844e271d8 \\\n    --hash=sha256:8c730dc3a83e5ac137fbc92dfcfe1511ce3b2b5d7578315b63dbbb76f7f51d90 \\\n    --hash=sha256:8e275ee4cb11c262bd108ab2081f750db2a1c0b8c12c1897f27b160c8bd57bbc \\\n    --hash=sha256:9044b5e4f7083f209c4e35aa5dd54b1dd5b112b108648f5c902ad586d4f945c5 \\\n    --hash=sha256:93a18841d09bcdd774dcdc308e4537e1f867b3dec059c131fde0327899734aa1 \\\n    --hash=sha256:9409c080586d1f683df3f184f20e36fb647f2e0bc3988094d4fd8c9f4eb1b3b3 \\\n    --hash=sha256:96f82000e12f23e4f29346e42702b6ed9a2f2fea34a740dd5ffffcc8c539eb35 \\\n    --hash=sha256:9aa9aeddeed452b2f616ff5507459e7bab436916ccb10961c4a382cd3e03f47f \\\n    --hash=sha256:9ee85f0696a17dd28fbcfceb59f9510aa71934b483d1f5601d1030c3c8304f3c \\\n    --hash=sha256:a07dba04c5e22824816b2615ad7a7484432d7f540e6fa86af60d2de57b0fcee2 \\\n    --hash=sha256:a3cd561ded2cf2bbae44d4605837221b987c216cff94f49dfeed63488bb228d2 \\\n    --hash=sha256:a697cd8ba0383bba3d2d3ada02b34ed268cb548b369943cd349007730c92bddf \\\n    --hash=sha256:a76da0a31da6fcae4210aa94fd779c65c75786bc9af06289cd1c184451ef7a65 \\\n    --hash=sha256:a85b653980faad27e88b141348707ceeef8a1186f75ecc600c395dcac19f385b \\\n    --hash=sha256:a8d65b38173085f24bc07f8b6c505cbb7418009fa1a1fcb111b1f4961814a442 \\\n    --hash=sha256:aa8dd43daa836b9a8128dbe7d923423e5ad86f50a7a14dc688194b7be5c0dea2 \\\n    --hash=sha256:ab8a209b8485d3db694fa97a896d96dd6533d63c22829043fd9de627060beade \\\n    --hash=sha256:abc56501c3fd148d60659aae0af6ddc149660469082859fa7b066a298bde9482 \\\n    --hash=sha256:ad5db5781c774ab9a9b2c4302bbf0c1014960a0a7be63278d13ae6fdf88126fe \\\n    --hash=sha256:ae98e14432d458fc3de11a77ccb3ae65ddce70f730e7c76140653048c71bfcbc \\\n    --hash=sha256:b20be51b37a75cc54c2c55def3fa2c65bb94ba859dde241cd0a4fd302de5ae0a \\\n    --hash=sha256:b523466b1a31d0dcef7c5be1f20b942919b62fd6e9a9be199d035509cbefc0ec \\\n    --hash=sha256:b5d658fbd9f0d6eea113aea286b21d3cd4d3fd978157cbf2447a6035916506d3 \\\n    --hash=sha256:b6123aa4a59d75f06e9dd3dac5bf8bc9aa383121bb3dd9a7a612e05eabc9961a \\\n    --hash=sha256:bd165131fd51697e22421d0e467997ad31621b74bfc0b75956608cb2906dda07 \\\n    --hash=sha256:bf902d7413c82a1bfa08b06a070876132a5ae6b2388e2712aab3a7cbc02205c6 \\\n    --hash=sha256:c12fc111ef090845de2bb15009372175d76ac99969bdf31e2ce9b42e4b8cd88f \\\n    --hash=sha256:c1eec9d950b6fe688edee07138993e54ee4ae634c51443cfb7c1e7613322718e \\\n    --hash=sha256:c640e5a06869c75994624551f45e5506e4256562ead981cce820d5ab39ae2192 \\\n    --hash=sha256:cc1331b6d5a6e144aeb5e626f4375f5b7ae9934ba620c0ac6b3e43d5e683a0f0 \\\n    --hash=sha256:cfd5cd998c2e36a862d0e27b2df63237e67273f2fc78f47445b14e73a810e7e6 \\\n    --hash=sha256:d3d8da4a631471dfaf94c10c85f5277b1f8e42ac42bade1ac67da4b4a7359b73 \\\n    --hash=sha256:d44ff19eea13ae4acdaaab0179fa68c0c6f2f45d66a4d8ec1eda7d6cecbcc15f \\\n    --hash=sha256:dd0052e9db3474df30433f83a71b9b23bd9e4ef1de13d92df21a52c0303b8ab6 \\\n    --hash=sha256:dd0e081319328928531df7a0e63621caf67652c8464303fd102141b785ef9547 \\\n    --hash=sha256:dda60aa465b861324e65a78c9f5cf0f4bc713e4309f83bc387be158b077963d9 \\\n    --hash=sha256:e06695e0326d05b06833b40b7ef477e475d0b1ba3a6d27da1bb48c23209bf457 \\\n    --hash=sha256:e1abe69aca89514737465752b4bcaf8016de61b3be1397a8fc260ba33321b3a8 \\\n    --hash=sha256:e267b0ed063341f3e60acd25c05200df4193e15a4a5807075cd71225a2386e26 \\\n    --hash=sha256:e5449ca63da169a2e6068dd0e2fcc8d91f9558aba89ff6d02121ca8ab11e79e5 \\\n    --hash=sha256:e63e4e5081de46517099dc30abe418122f54531a6ae2ebc8680bcd7096860eab \\\n    --hash=sha256:f189805c8be5ca5add39e6f899e6ce2ed824e65fb45f3c28cb2841911da19070 \\\n    --hash=sha256:f7955ecf5609dee9442cbface754f2c6e541d9e6eda87fad7f7a989b0bdb9d71 \\\n    --hash=sha256:f86d3a7a9af5d826744fabf4afd15b9dfef44fe69a98541f666f66fbb8d3fef9 \\\n    --hash=sha256:fbd43429d0d7ed6533b25fc993861b8fd512c42d04514a0dd6337fb3ccf22761\npip==25.0.1 \\\n    --hash=sha256:88f96547ea48b940a3a385494e181e29fb8637898f88d88737c5049780f196ea \\\n    --hash=sha256:c46efd13b6aa8279f33f2864459c8ce587ea6a1a59ee20de055868d8f7688f7f\nplatformdirs==4.4.0 \\\n    --hash=sha256:abd01743f24e5287cd7a5db3752faf1a2d65353f38ec26d98e25a6db65958c85 \\\n    --hash=sha256:ca753cf4d81dc309bc67b0ea38fd15dc97bc30ce419a7f58d13eb3bf14c4febf\nportalocker==3.2.0 \\\n    --hash=sha256:1f3002956a54a8c3730586c5c77bf18fae4149e07eaf1c29fc3faf4d5a3f89ac \\\n    --hash=sha256:3cdc5f565312224bc570c49337bd21428bba0ef363bbcf58b9ef4a9f11779968\nprompt-toolkit==3.0.52 \\\n    --hash=sha256:28cde192929c8e7321de85de1ddbe736f1375148b02f2e17edd840042b1be855 \\\n    --hash=sha256:9aac639a3bbd33284347de5ad8d68ecc044b91a762dc39b7c21095fcd6a19955\nprotobuf==5.29.5 \\\n    --hash=sha256:3f1c6468a2cfd102ff4703976138844f78ebd1fb45f49011afc5139e9e283079 \\\n    --hash=sha256:3f76e3a3675b4a4d867b52e4a5f5b78a2ef9565549d4037e06cf7b0942b1d3fc \\\n    --hash=sha256:470f3af547ef17847a28e1f47200a1cbf0ba3ff57b7de50d22776607cd2ea353 \\\n    --hash=sha256:63848923da3325e1bf7e9003d680ce6e14b07e55d0473253a690c3a8b8fd6e61 \\\n    --hash=sha256:6cf42630262c59b2d8de33954443d94b746c952b01434fc58a417fdbd2e84bd5 \\\n    --hash=sha256:6f642dc9a61782fa72b90878af134c5afe1917c89a568cd3476d758d3c3a0736 \\\n    --hash=sha256:7318608d56b6402d2ea7704ff1e1e4597bee46d760e7e4dd42a3d45e24b87f2e \\\n    --hash=sha256:bc1463bafd4b0929216c35f437a8e28731a2b7fe3d98bb77a600efced5a15c84 \\\n    --hash=sha256:e38c5add5a311f2a6eb0340716ef9b039c1dfa428b28f25a7838ac329204a671 \\\n    --hash=sha256:ef91363ad4faba7b25d844ef1ada59ff1604184c0bcd8b39b8a6bef15e1af238 \\\n    --hash=sha256:fa18533a299d7ab6c55a238bf8629311439995f2e7eca5caaff08663606e9015\npsutil==7.0.0 \\\n    --hash=sha256:101d71dc322e3cffd7cea0650b09b3d08b8e7c4109dd6809fe452dfd00e58b25 \\\n    --hash=sha256:1e744154a6580bc968a0195fd25e80432d3afec619daf145b9e5ba16cc1d688e \\\n    --hash=sha256:1fcee592b4c6f146991ca55919ea3d1f8926497a713ed7faaf8225e174581e91 \\\n    --hash=sha256:39db632f6bb862eeccf56660871433e111b6ea58f2caea825571951d4b6aa3da \\\n    --hash=sha256:4b1388a4f6875d7e2aff5c4ca1cc16c545ed41dd8bb596cefea80111db353a34 \\\n    --hash=sha256:4cf3d4eb1aa9b348dec30105c55cd9b7d4629285735a102beb4441e38db90553 \\\n    --hash=sha256:7be9c3eba38beccb6495ea33afd982a44074b78f28c434a1f51cc07fd315c456 \\\n    --hash=sha256:84df4eb63e16849689f76b1ffcb36db7b8de703d1bc1fe41773db487621b6c17 \\\n    --hash=sha256:a5f098451abc2828f7dc6b58d44b532b22f2088f4999a937557b603ce72b1993 \\\n    --hash=sha256:ba3fcef7523064a6c9da440fc4d6bd07da93ac726b5733c29027d7dc95b39d99\nptyprocess==0.7.0 \\\n    --hash=sha256:4b41f3967fce3af57cc7e94b888626c18bf37a083e3651ca8feeb66d492fef35 \\\n    --hash=sha256:5c5d0a3b48ceee0b48485e0c26037c0acd7d29765ca3fbb5cb3831d347423220\npure-eval==0.2.3 \\\n    --hash=sha256:1db8e35b67b3d218d818ae653e27f06c3aa420901fa7b081ca98cbedc874e0d0 \\\n    --hash=sha256:5f4e983f40564c576c7c8635ae88db5956bb2229d7e9237d03b3c0b0190eaf42\npydantic==2.11.7 \\\n    --hash=sha256:d989c3c6cb79469287b1569f7447a17848c998458d49ebe294e975b9baf0f0db \\\n    --hash=sha256:dde5df002701f6de26248661f6835bbe296a47bf73990135c7d07ce741b9623b\npydantic-core==2.33.2 \\\n    --hash=sha256:0069c9acc3f3981b9ff4cdfaf088e98d83440a4c7ea1bc07460af3d4dc22e72d \\\n    --hash=sha256:031c57d67ca86902726e0fae2214ce6770bbe2f710dc33063187a68744a5ecac \\\n    --hash=sha256:0405262705a123b7ce9f0b92f123334d67b70fd1f20a9372b907ce1080c7ba02 \\\n    --hash=sha256:04a1a413977ab517154eebb2d326da71638271477d6ad87a769102f7c2488c56 \\\n    --hash=sha256:09fb9dd6571aacd023fe6aaca316bd01cf60ab27240d7eb39ebd66a3a15293b4 \\\n    --hash=sha256:0a39979dcbb70998b0e505fb1556a1d550a0781463ce84ebf915ba293ccb7e22 \\\n    --hash=sha256:0a9f2c9dd19656823cb8250b0724ee9c60a82f3cdf68a080979d13092a3b0fef \\\n    --hash=sha256:0e03262ab796d986f978f79c943fc5f620381be7287148b8010b4097f79a39ec \\\n    --hash=sha256:0e5b2671f05ba48b94cb90ce55d8bdcaaedb8ba00cc5359f6810fc918713983d \\\n    --hash=sha256:0e6116757f7959a712db11f3e9c0a99ade00a5bbedae83cb801985aa154f071b \\\n    --hash=sha256:0fb2d542b4d66f9470e8065c5469ec676978d625a8b7a363f07d9a501a9cb36a \\\n    --hash=sha256:1082dd3e2d7109ad8b7da48e1d4710c8d06c253cbc4a27c1cff4fbcaa97a9e3f \\\n    --hash=sha256:1a8695a8d00c73e50bff9dfda4d540b7dee29ff9b8053e38380426a85ef10052 \\\n    --hash=sha256:1e063337ef9e9820c77acc768546325ebe04ee38b08703244c1309cccc4f1bab \\\n    --hash=sha256:1ea40a64d23faa25e62a70ad163571c0b342b8bf66d5fa612ac0dec4f069d916 \\\n    --hash=sha256:2058a32994f1fde4ca0480ab9d1e75a0e8c87c22b53a3ae66554f9af78f2fe8c \\\n    --hash=sha256:235f45e5dbcccf6bd99f9f472858849f73d11120d76ea8707115415f8e5ebebf \\\n    --hash=sha256:2807668ba86cb38c6817ad9bc66215ab8584d1d304030ce4f0887336f28a5e27 \\\n    --hash=sha256:2b0a451c263b01acebe51895bfb0e1cc842a5c666efe06cdf13846c7418caa9a \\\n    --hash=sha256:2b3d326aaef0c0399d9afffeb6367d5e26ddc24d351dbc9c636840ac355dc5d8 \\\n    --hash=sha256:2bfb5112df54209d820d7bf9317c7a6c9025ea52e49f46b6a2060104bba37de7 \\\n    --hash=sha256:2f82865531efd18d6e07a04a17331af02cb7a651583c418df8266f17a63c6612 \\\n    --hash=sha256:329467cecfb529c925cf2bbd4d60d2c509bc2fb52a20c1045bf09bb70971a9c1 \\\n    --hash=sha256:3a1c81334778f9e3af2f8aeb7a960736e5cab1dfebfb26aabca09afd2906c039 \\\n    --hash=sha256:3abcd9392a36025e3bd55f9bd38d908bd17962cc49bc6da8e7e96285336e2bca \\\n    --hash=sha256:3c6db6e52c6d70aa0d00d45cdb9b40f0433b96380071ea80b09277dba021ddf7 \\\n    --hash=sha256:3dc625f4aa79713512d1976fe9f0bc99f706a9dee21dfd1810b4bbbf228d0e8a \\\n    --hash=sha256:3eb3fe62804e8f859c49ed20a8451342de53ed764150cb14ca71357c765dc2a6 \\\n    --hash=sha256:44857c3227d3fb5e753d5fe4a3420d6376fa594b07b621e220cd93703fe21782 \\\n    --hash=sha256:4b25d91e288e2c4e0662b8038a28c6a07eaac3e196cfc4ff69de4ea3db992a1b \\\n    --hash=sha256:4c5b0a576fb381edd6d27f0a85915c6daf2f8138dc5c267a57c08a62900758c7 \\\n    --hash=sha256:4e61206137cbc65e6d5256e1166f88331d3b6238e082d9f74613b9b765fb9025 \\\n    --hash=sha256:52fb90784e0a242bb96ec53f42196a17278855b0f31ac7c3cc6f5c1ec4811849 \\\n    --hash=sha256:53a57d2ed685940a504248187d5685e49eb5eef0f696853647bf37c418c538f7 \\\n    --hash=sha256:572c7e6c8bb4774d2ac88929e3d1f12bc45714ae5ee6d9a788a9fb35e60bb04b \\\n    --hash=sha256:5c4aa4e82353f65e548c476b37e64189783aa5384903bfea4f41580f255fddfa \\\n    --hash=sha256:5c92edd15cd58b3c2d34873597a1e20f13094f59cf88068adb18947df5455b4e \\\n    --hash=sha256:5f483cfb75ff703095c59e365360cb73e00185e01aaea067cd19acffd2ab20ea \\\n    --hash=sha256:61c18fba8e5e9db3ab908620af374db0ac1baa69f0f32df4f61ae23f15e586ac \\\n    --hash=sha256:6368900c2d3ef09b69cb0b913f9f8263b03786e5b2a387706c5afb66800efd51 \\\n    --hash=sha256:64632ff9d614e5eecfb495796ad51b0ed98c453e447a76bcbeeb69615079fc7e \\\n    --hash=sha256:65132b7b4a1c0beded5e057324b7e16e10910c106d43675d9bd87d4f38dde162 \\\n    --hash=sha256:6b99022f1d19bc32a4c2a0d544fc9a76e3be90f0b3f4af413f87d38749300e65 \\\n    --hash=sha256:6bdfe4b3789761f3bcb4b1ddf33355a71079858958e3a552f16d5af19768fef2 \\\n    --hash=sha256:6fa6dfc3e4d1f734a34710f391ae822e0a8eb8559a85c6979e14e65ee6ba2954 \\\n    --hash=sha256:73662edf539e72a9440129f231ed3757faab89630d291b784ca99237fb94db2b \\\n    --hash=sha256:73cf6373c21bc80b2e0dc88444f41ae60b2f070ed02095754eb5a01df12256de \\\n    --hash=sha256:7cb8bc3605c29176e1b105350d2e6474142d7c1bd1d9327c4a9bdb46bf827acc \\\n    --hash=sha256:7f92c15cd1e97d4b12acd1cc9004fa092578acfa57b67ad5e43a197175d01a64 \\\n    --hash=sha256:82f68293f055f51b51ea42fafc74b6aad03e70e191799430b90c13d643059ebb \\\n    --hash=sha256:83aa99b1285bc8f038941ddf598501a86f1536789740991d7d8756e34f1e74d9 \\\n    --hash=sha256:87acbfcf8e90ca885206e98359d7dca4bcbb35abdc0ff66672a293e1d7a19101 \\\n    --hash=sha256:87b31b6846e361ef83fedb187bb5b4372d0da3f7e28d85415efa92d6125d6e6d \\\n    --hash=sha256:881b21b5549499972441da4758d662aeea93f1923f953e9cbaff14b8b9565aef \\\n    --hash=sha256:8d55ab81c57b8ff8548c3e4947f119551253f4e3787a7bbc0b6b3ca47498a9d3 \\\n    --hash=sha256:8f57a69461af2a5fa6e6bbd7a5f60d3b7e6cebb687f55106933188e79ad155c1 \\\n    --hash=sha256:95237e53bb015f67b63c91af7518a62a8660376a6a0db19b89acc77a4d6199f5 \\\n    --hash=sha256:96081f1605125ba0855dfda83f6f3df5ec90c61195421ba72223de35ccfb2f88 \\\n    --hash=sha256:970919794d126ba8645f3837ab6046fb4e72bbc057b3709144066204c19a455d \\\n    --hash=sha256:9cb1da0f5a471435a7bc7e439b8a728e8b61e59784b2af70d7c169f8dd8ae290 \\\n    --hash=sha256:9fcd347d2cc5c23b06de6d3b7b8275be558a0c90549495c699e379a80bf8379e \\\n    --hash=sha256:9fdac5d6ffa1b5a83bca06ffe7583f5576555e6c8b3a91fbd25ea7780f825f7d \\\n    --hash=sha256:a11c8d26a50bfab49002947d3d237abe4d9e4b5bdc8846a63537b6488e197808 \\\n    --hash=sha256:a144d4f717285c6d9234a66778059f33a89096dfb9b39117663fd8413d582dcc \\\n    --hash=sha256:a2b911a5b90e0374d03813674bf0a5fbbb7741570dcd4b4e85a2e48d17def29d \\\n    --hash=sha256:a7ec89dc587667f22b6a0b6579c249fca9026ce7c333fc142ba42411fa243cdc \\\n    --hash=sha256:aa9d91b338f2df0508606f7009fde642391425189bba6d8c653afd80fd6bb64e \\\n    --hash=sha256:b0379a2b24882fef529ec3b4987cb5d003b9cda32256024e6fe1586ac45fc640 \\\n    --hash=sha256:bc7aee6f634a6f4a95676fcb5d6559a2c2a390330098dba5e5a5f28a2e4ada30 \\\n    --hash=sha256:bdc25f3681f7b78572699569514036afe3c243bc3059d3942624e936ec93450e \\\n    --hash=sha256:c083a3bdd5a93dfe480f1125926afcdbf2917ae714bdb80b36d34318b2bec5d9 \\\n    --hash=sha256:c20c462aa4434b33a2661701b861604913f912254e441ab8d78d30485736115a \\\n    --hash=sha256:c2fc0a768ef76c15ab9238afa6da7f69895bb5d1ee83aeea2e3509af4472d0b9 \\\n    --hash=sha256:c52b02ad8b4e2cf14ca7b3d918f3eb0ee91e63b3167c32591e57c4317e134f8f \\\n    --hash=sha256:c54c939ee22dc8e2d545da79fc5381f1c020d6d3141d3bd747eab59164dc89fb \\\n    --hash=sha256:c8e7af2f4e0194c22b5b37205bfb293d166a7344a5b0d0eaccebc376546d77d5 \\\n    --hash=sha256:cca3868ddfaccfbc4bfb1d608e2ccaaebe0ae628e1416aeb9c4d88c001bb45ab \\\n    --hash=sha256:d3f26877a748dc4251cfcfda9dfb5f13fcb034f5308388066bcfe9031b63ae7d \\\n    --hash=sha256:d53b22f2032c42eaaf025f7c40c2e3b94568ae077a606f006d206a463bc69572 \\\n    --hash=sha256:d87c561733f66531dced0da6e864f44ebf89a8fba55f31407b00c2f7f9449593 \\\n    --hash=sha256:d946c8bf0d5c24bf4fe333af284c59a19358aa3ec18cb3dc4370080da1e8ad29 \\\n    --hash=sha256:dac89aea9af8cd672fa7b510e7b8c33b0bba9a43186680550ccf23020f32d535 \\\n    --hash=sha256:db4b41f9bd95fbe5acd76d89920336ba96f03e149097365afe1cb092fceb89a1 \\\n    --hash=sha256:dc46a01bf8d62f227d5ecee74178ffc448ff4e5197c756331f71efcc66dc980f \\\n    --hash=sha256:dd14041875d09cc0f9308e37a6f8b65f5585cf2598a53aa0123df8b129d481f8 \\\n    --hash=sha256:de4b83bb311557e439b9e186f733f6c645b9417c84e2eb8203f3f820a4b988bf \\\n    --hash=sha256:e799c050df38a639db758c617ec771fd8fb7a5f8eaaa4b27b101f266b216a246 \\\n    --hash=sha256:e80b087132752f6b3d714f041ccf74403799d3b23a72722ea2e6ba2e892555b9 \\\n    --hash=sha256:eb8c529b2819c37140eb51b914153063d27ed88e3bdc31b71198a198e921e011 \\\n    --hash=sha256:eb9b459ca4df0e5c87deb59d37377461a538852765293f9e6ee834f0435a93b9 \\\n    --hash=sha256:efec8db3266b76ef9607c2c4c419bdb06bf335ae433b80816089ea7585816f6a \\\n    --hash=sha256:f481959862f57f29601ccced557cc2e817bce7533ab8e01a797a48b49c9692b3 \\\n    --hash=sha256:f517ca031dfc037a9c07e748cefd8d96235088b83b4f4ba8939105d20fa1dcd6 \\\n    --hash=sha256:f889f7a40498cc077332c7ab6b4608d296d852182211787d4f3ee377aaae66e8 \\\n    --hash=sha256:f8de619080e944347f5f20de29a975c2d815d9ddd8be9b9b7268e2e3ef68605a \\\n    --hash=sha256:f941635f2a3d96b2973e867144fde513665c87f13fe0e193c158ac51bfaaa7b2 \\\n    --hash=sha256:fa754d1850735a0b0e03bcffd9d4b4343eb417e47196e4485d9cca326073a42c \\\n    --hash=sha256:fa854f5cf7e33842a892e5c73f45327760bc7bc516339fda888c75ae60edaeb6 \\\n    --hash=sha256:fe5b32187cbc0c862ee201ad66c30cf218e5ed468ec8dc1cf49dec66e160cc4d\npygments==2.19.2 \\\n    --hash=sha256:636cb2477cec7f8952536970bc533bc43743542f70392ae026374600add5b887 \\\n    --hash=sha256:86540386c03d588bb81d44bc3928634ff26449851e99741617ecb9037ee5ec0b\npyparsing==3.2.3 \\\n    --hash=sha256:a749938e02d6fd0b59b356ca504a24982314bb090c383e3cf201c95ef7e2bfcf \\\n    --hash=sha256:b9c13f1ab8b3b542f72e28f634bad4de758ab3ce4546e4301970ad6fa77c38be\npython-dateutil==2.9.0.post0 \\\n    --hash=sha256:37dd54208da7e1cd875388217d5e00ebd4179249f90fb72437e91a35459a0ad3 \\\n    --hash=sha256:a8b2bc7bffae282281c8140a97d3aa9c14da0b136dfe83f850eea9a5f7470427\npytz==2025.2 \\\n    --hash=sha256:360b9e3dbb49a209c21ad61809c7fb453643e048b38924c765813546746e81c3 \\\n    --hash=sha256:5ddf76296dd8c44c26eb8f4b6f35488f3ccbf6fbbd7adee0b7262d43f0ec2f00\npyyaml==6.0.2 \\\n    --hash=sha256:01179a4a8559ab5de078078f37e5c1a30d76bb88519906844fd7bdea1b7729ff \\\n    --hash=sha256:0833f8694549e586547b576dcfaba4a6b55b9e96098b36cdc7ebefe667dfed48 \\\n    --hash=sha256:0a9a2848a5b7feac301353437eb7d5957887edbf81d56e903999a75a3d743086 \\\n    --hash=sha256:0b69e4ce7a131fe56b7e4d770c67429700908fc0752af059838b1cfb41960e4e \\\n    --hash=sha256:0ffe8360bab4910ef1b9e87fb812d8bc0a308b0d0eef8c8f44e0254ab3b07133 \\\n    --hash=sha256:11d8f3dd2b9c1207dcaf2ee0bbbfd5991f571186ec9cc78427ba5bd32afae4b5 \\\n    --hash=sha256:17e311b6c678207928d649faa7cb0d7b4c26a0ba73d41e99c4fff6b6c3276484 \\\n    --hash=sha256:1e2120ef853f59c7419231f3bf4e7021f1b936f6ebd222406c3b60212205d2ee \\\n    --hash=sha256:1f71ea527786de97d1a0cc0eacd1defc0985dcf6b3f17bb77dcfc8c34bec4dc5 \\\n    --hash=sha256:23502f431948090f597378482b4812b0caae32c22213aecf3b55325e049a6c68 \\\n    --hash=sha256:24471b829b3bf607e04e88d79542a9d48bb037c2267d7927a874e6c205ca7e9a \\\n    --hash=sha256:29717114e51c84ddfba879543fb232a6ed60086602313ca38cce623c1d62cfbf \\\n    --hash=sha256:2e99c6826ffa974fe6e27cdb5ed0021786b03fc98e5ee3c5bfe1fd5015f42b99 \\\n    --hash=sha256:39693e1f8320ae4f43943590b49779ffb98acb81f788220ea932a6b6c51004d8 \\\n    --hash=sha256:3ad2a3decf9aaba3d29c8f537ac4b243e36bef957511b4766cb0057d32b0be85 \\\n    --hash=sha256:3b1fdb9dc17f5a7677423d508ab4f243a726dea51fa5e70992e59a7411c89d19 \\\n    --hash=sha256:41e4e3953a79407c794916fa277a82531dd93aad34e29c2a514c2c0c5fe971cc \\\n    --hash=sha256:43fa96a3ca0d6b1812e01ced1044a003533c47f6ee8aca31724f78e93ccc089a \\\n    --hash=sha256:50187695423ffe49e2deacb8cd10510bc361faac997de9efef88badc3bb9e2d1 \\\n    --hash=sha256:5ac9328ec4831237bec75defaf839f7d4564be1e6b25ac710bd1a96321cc8317 \\\n    --hash=sha256:5d225db5a45f21e78dd9358e58a98702a0302f2659a3c6cd320564b75b86f47c \\\n    --hash=sha256:6395c297d42274772abc367baaa79683958044e5d3835486c16da75d2a694631 \\\n    --hash=sha256:688ba32a1cffef67fd2e9398a2efebaea461578b0923624778664cc1c914db5d \\\n    --hash=sha256:68ccc6023a3400877818152ad9a1033e3db8625d899c72eacb5a668902e4d652 \\\n    --hash=sha256:70b189594dbe54f75ab3a1acec5f1e3faa7e8cf2f1e08d9b561cb41b845f69d5 \\\n    --hash=sha256:797b4f722ffa07cc8d62053e4cff1486fa6dc094105d13fea7b1de7d8bf71c9e \\\n    --hash=sha256:7c36280e6fb8385e520936c3cb3b8042851904eba0e58d277dca80a5cfed590b \\\n    --hash=sha256:7e7401d0de89a9a855c839bc697c079a4af81cf878373abd7dc625847d25cbd8 \\\n    --hash=sha256:80bab7bfc629882493af4aa31a4cfa43a4c57c83813253626916b8c7ada83476 \\\n    --hash=sha256:82d09873e40955485746739bcb8b4586983670466c23382c19cffecbf1fd8706 \\\n    --hash=sha256:8388ee1976c416731879ac16da0aff3f63b286ffdd57cdeb95f3f2e085687563 \\\n    --hash=sha256:8824b5a04a04a047e72eea5cec3bc266db09e35de6bdfe34c9436ac5ee27d237 \\\n    --hash=sha256:8b9c7197f7cb2738065c481a0461e50ad02f18c78cd75775628afb4d7137fb3b \\\n    --hash=sha256:9056c1ecd25795207ad294bcf39f2db3d845767be0ea6e6a34d856f006006083 \\\n    --hash=sha256:936d68689298c36b53b29f23c6dbb74de12b4ac12ca6cfe0e047bedceea56180 \\\n    --hash=sha256:9b22676e8097e9e22e36d6b7bda33190d0d400f345f23d4065d48f4ca7ae0425 \\\n    --hash=sha256:a4d3091415f010369ae4ed1fc6b79def9416358877534caf6a0fdd2146c87a3e \\\n    --hash=sha256:a8786accb172bd8afb8be14490a16625cbc387036876ab6ba70912730faf8e1f \\\n    --hash=sha256:a9f8c2e67970f13b16084e04f134610fd1d374bf477b17ec1599185cf611d725 \\\n    --hash=sha256:bc2fa7c6b47d6bc618dd7fb02ef6fdedb1090ec036abab80d4681424b84c1183 \\\n    --hash=sha256:c70c95198c015b85feafc136515252a261a84561b7b1d51e3384e0655ddf25ab \\\n    --hash=sha256:cc1c1159b3d456576af7a3e4d1ba7e6924cb39de8f67111c735f6fc832082774 \\\n    --hash=sha256:ce826d6ef20b1bc864f0a68340c8b3287705cae2f8b4b1d932177dcc76721725 \\\n    --hash=sha256:d584d9ec91ad65861cc08d42e834324ef890a082e591037abe114850ff7bbc3e \\\n    --hash=sha256:d7fded462629cfa4b685c5416b949ebad6cec74af5e2d42905d41e257e0869f5 \\\n    --hash=sha256:d84a1718ee396f54f3a086ea0a66d8e552b2ab2017ef8b420e92edbc841c352d \\\n    --hash=sha256:d8e03406cac8513435335dbab54c0d385e4a49e4945d2909a581c83647ca0290 \\\n    --hash=sha256:e10ce637b18caea04431ce14fabcf5c64a1c61ec9c56b071a4b7ca131ca52d44 \\\n    --hash=sha256:ec031d5d2feb36d1d1a24380e4db6d43695f3748343d99434e6f5f9156aaa2ed \\\n    --hash=sha256:ef6107725bd54b262d6dedcc2af448a266975032bc85ef0172c5f059da6325b4 \\\n    --hash=sha256:efdca5630322a10774e8e98e1af481aad470dd62c3170801852d752aa7a783ba \\\n    --hash=sha256:f753120cb8181e736c57ef7636e83f31b9c0d1722c516f7e86cf15b7aa57ff12 \\\n    --hash=sha256:ff3824dc5261f50c9b0dfb3be22b4567a6f938ccce4587b38952d85fd9e9afe4\nregex==2025.9.1 \\\n    --hash=sha256:09a41dc039e1c97d3c2ed3e26523f748e58c4de3ea7a31f95e1cf9ff973fff5a \\\n    --hash=sha256:0aeb0fe80331059c152a002142699a89bf3e44352aee28261315df0c9874759b \\\n    --hash=sha256:0fa9a7477288717f42dbd02ff5d13057549e9a8cdb81f224c313154cc10bab52 \\\n    --hash=sha256:10a450cba5cd5409526ee1d4449f42aad38dd83ac6948cbd6d7f71ca7018f7db \\\n    --hash=sha256:113d5aa950f428faf46fd77d452df62ebb4cc6531cb619f6cc30a369d326bfbd \\\n    --hash=sha256:1e978e5a35b293ea43f140c92a3269b6ab13fe0a2bf8a881f7ac740f5a6ade85 \\\n    --hash=sha256:1ec2bd3bdf0f73f7e9f48dca550ba7d973692d5e5e9a90ac42cc5f16c4432d8b \\\n    --hash=sha256:22213527df4c985ec4a729b055a8306272d41d2f45908d7bacb79be0fa7a75ad \\\n    --hash=sha256:34679a86230e46164c9e0396b56cab13c0505972343880b9e705083cc5b8ec86 \\\n    --hash=sha256:3b9a62107a7441b81ca98261808fed30ae36ba06c8b7ee435308806bd53c1ed8 \\\n    --hash=sha256:43ebc77a7dfe36661192afd8d7df5e8be81ec32d2ad0c65b536f66ebfec3dece \\\n    --hash=sha256:47829ffaf652f30d579534da9085fe30c171fa2a6744a93d52ef7195dc38218b \\\n    --hash=sha256:47d7c2dab7e0b95b95fd580087b6ae196039d62306a592fa4e162e49004b6299 \\\n    --hash=sha256:49865e78d147a7a4f143064488da5d549be6bfc3f2579e5044cac61f5c92edd4 \\\n    --hash=sha256:4a62d033cd9ebefc7c5e466731a508dfabee827d80b13f455de68a50d3c2543d \\\n    --hash=sha256:4bcdff370509164b67a6c8ec23c9fb40797b72a014766fdc159bb809bd74f7d8 \\\n    --hash=sha256:4cf09903e72411f4bf3ac1eddd624ecfd423f14b2e4bf1c8b547b72f248b7bf7 \\\n    --hash=sha256:4f0b4258b161094f66857a26ee938d3fe7b8a5063861e44571215c44fbf0e5df \\\n    --hash=sha256:4f6e935e98ea48c7a2e8be44494de337b57a204470e7f9c9c42f912c414cd6f5 \\\n    --hash=sha256:588c161a68a383478e27442a678e3b197b13c5ba51dbba40c1ccb8c4c7bee9e9 \\\n    --hash=sha256:5aba22dfbc60cda7c0853516104724dc904caa2db55f2c3e6e984eb858d3edf3 \\\n    --hash=sha256:5c3b96ed0223b32dbdc53a83149b6de7ca3acd5acd9c8e64b42a166228abe29c \\\n    --hash=sha256:5d74b557cf5554001a869cda60b9a619be307df4d10155894aeaad3ee67c9899 \\\n    --hash=sha256:645e88a73861c64c1af558dd12294fb4e67b5c1eae0096a60d7d8a2143a611c7 \\\n    --hash=sha256:656563e620de6908cd1c9d4f7b9e0777e3341ca7db9d4383bcaa44709c90281e \\\n    --hash=sha256:67a0295a3c31d675a9ee0238d20238ff10a9a2fdb7a1323c798fc7029578b15c \\\n    --hash=sha256:67a74456f410fe5e869239ee7a5423510fe5121549af133809d9591a8075893f \\\n    --hash=sha256:6aeff21de7214d15e928fb5ce757f9495214367ba62875100d4c18d293750cc1 \\\n    --hash=sha256:6b81d7dbc5466ad2c57ce3a0ddb717858fe1a29535c8866f8514d785fdb9fc5b \\\n    --hash=sha256:6c0226fb322b82709e78c49cc33484206647f8a39954d7e9de1567f5399becd0 \\\n    --hash=sha256:6d40e6b49daae9ebbd7fa4e600697372cba85b826592408600068e83a3c47211 \\\n    --hash=sha256:6f1dae2cf6c2dbc6fd2526653692c144721b3cf3f769d2a3c3aa44d0f38b9a58 \\\n    --hash=sha256:6ff623271e0b0cc5a95b802666bbd70f17ddd641582d65b10fb260cc0c003529 \\\n    --hash=sha256:72fb7a016467d364546f22b5ae86c45680a4e0de6b2a6f67441d22172ff641f1 \\\n    --hash=sha256:7383efdf6e8e8c61d85e00cfb2e2e18da1a621b8bfb4b0f1c2747db57b942b8f \\\n    --hash=sha256:74df7c74a63adcad314426b1f4ea6054a5ab25d05b0244f0c07ff9ce640fa597 \\\n    --hash=sha256:7e786d9e4469698fc63815b8de08a89165a0aa851720eb99f5e0ea9d51dd2b6a \\\n    --hash=sha256:84a25164bd8dcfa9f11c53f561ae9766e506e580b70279d05a7946510bdd6f6a \\\n    --hash=sha256:88ac07b38d20b54d79e704e38aa3bd2c0f8027432164226bdee201a1c0c9c9ff \\\n    --hash=sha256:8c2ff5c01d5e47ad5fc9d31bcd61e78c2fa0068ed00cab86b7320214446da766 \\\n    --hash=sha256:8e3f6e3c5a5a1adc3f7ea1b5aec89abfc2f4fbfba55dafb4343cd1d084f715b2 \\\n    --hash=sha256:91892a7a9f0a980e4c2c85dd19bc14de2b219a3a8867c4b5664b9f972dcc0c78 \\\n    --hash=sha256:94533e32dc0065eca43912ee6649c90ea0681d59f56d43c45b5bcda9a740b3dd \\\n    --hash=sha256:94f6cff6f7e2149c7e6499a6ecd4695379eeda8ccbccb9726e8149f2fe382e92 \\\n    --hash=sha256:9627e887116c4e9c0986d5c3b4f52bcfe3df09850b704f62ec3cbf177a0ae374 \\\n    --hash=sha256:a1196e530a6bfa5f4bde029ac5b0295a6ecfaaffbfffede4bbaf4061d9455b70 \\\n    --hash=sha256:a12f59c7c380b4fcf7516e9cbb126f95b7a9518902bcf4a852423ff1dcd03e6a \\\n    --hash=sha256:a13d20007dce3c4b00af5d84f6c191ed1c0f70928c6d9b6cd7b8d2f125df7f46 \\\n    --hash=sha256:a32291add816961aab472f4fad344c92871a2ee33c6c219b6598e98c1f0108f2 \\\n    --hash=sha256:a34ef82216189d823bc82f614d1031cb0b919abef27cecfd7b07d1e9a8bdeeb4 \\\n    --hash=sha256:a874a61bb580d48642ffd338570ee24ab13fa023779190513fcacad104a6e251 \\\n    --hash=sha256:a90014d29cb3098403d82a879105d1418edbbdf948540297435ea6e377023ea7 \\\n    --hash=sha256:aa88d5a82dfe80deaf04e8c39c8b0ad166d5d527097eb9431cb932c44bf88715 \\\n    --hash=sha256:b0e2f95413eb0c651cd1516a670036315b91b71767af83bc8525350d4375ccba \\\n    --hash=sha256:b2b3ad150c6bc01a8cd5030040675060e2adbe6cbc50aadc4da42c6d32ec266e \\\n    --hash=sha256:b38afecc10c177eb34cfae68d669d5161880849ba70c05cbfbe409f08cc939d7 \\\n    --hash=sha256:b84036511e1d2bb0a4ff1aec26951caa2dea8772b223c9e8a19ed8885b32dbac \\\n    --hash=sha256:bc6834727d1b98d710a63e6c823edf6ffbf5792eba35d3fa119531349d4142ef \\\n    --hash=sha256:bcb89c02a0d6c2bec9b0bb2d8c78782699afe8434493bfa6b4021cc51503f249 \\\n    --hash=sha256:bf70e18ac390e6977ea7e56f921768002cb0fa359c4199606c7219854ae332e0 \\\n    --hash=sha256:c2e05dcdfe224047f2a59e70408274c325d019aad96227ab959403ba7d58d2d7 \\\n    --hash=sha256:c3dc05b6d579875719bccc5f3037b4dc80433d64e94681a0061845bd8863c025 \\\n    --hash=sha256:c5aa2a6a73bf218515484b36a0d20c6ad9dc63f6339ff6224147b0e2c095ee55 \\\n    --hash=sha256:c905d925d194c83a63f92422af7544ec188301451b292c8b487f0543726107ca \\\n    --hash=sha256:c9527fa74eba53f98ad86be2ba003b3ebe97e94b6eb2b916b31b5f055622ef03 \\\n    --hash=sha256:ca3affe8ddea498ba9d294ab05f5f2d3b5ad5d515bc0d4a9016dd592a03afe52 \\\n    --hash=sha256:cd4890e184a6feb0ef195338a6ce68906a8903a0f2eb7e0ab727dbc0a3156273 \\\n    --hash=sha256:d016b0f77be63e49613c9e26aaf4a242f196cd3d7a4f15898f5f0ab55c9b24d2 \\\n    --hash=sha256:d161bfdeabe236290adfd8c7588da7f835d67e9e7bf2945f1e9e120622839ba6 \\\n    --hash=sha256:d34b901f6f2f02ef60f4ad3855d3a02378c65b094efc4b80388a3aeb700a5de7 \\\n    --hash=sha256:d49dc84e796b666181de8a9973284cad6616335f01b52bf099643253094920fc \\\n    --hash=sha256:d6b046b0a01cb713fd53ef36cb59db4b0062b343db28e83b52ac6aa01ee5b368 \\\n    --hash=sha256:d89f1bbbbbc0885e1c230f7770d5e98f4f00b0ee85688c871d10df8b184a6323 \\\n    --hash=sha256:d936a1db208bdca0eca1f2bb2c1ba1d8370b226785c1e6db76e32a228ffd0ad5 \\\n    --hash=sha256:d9914fe1040874f83c15fcea86d94ea54091b0666eab330aaab69e30d106aabe \\\n    --hash=sha256:df33f4ef07b68f7ab637b1dbd70accbf42ef0021c201660656601e8a9835de45 \\\n    --hash=sha256:e1cb40406f4ae862710615f9f636c1e030fd6e6abe0e0f65f6a695a2721440c6 \\\n    --hash=sha256:e5bcf112b09bfd3646e4db6bf2e598534a17d502b0c01ea6550ba4eca780c5e6 \\\n    --hash=sha256:e71bceb3947362ec5eabd2ca0870bb78eae4edfc60c6c21495133c01b6cd2df4 \\\n    --hash=sha256:e9dc5991592933a4192c166eeb67b29d9234f9c86344481173d1bc52f73a7104 \\\n    --hash=sha256:ea8267fbadc7d4bd7c1301a50e85c2ff0de293ff9452a1a9f8d82c6cafe38179 \\\n    --hash=sha256:ec1efb4c25e1849c2685fa95da44bfde1b28c62d356f9c8d861d4dad89ed56e9 \\\n    --hash=sha256:ec329890ad5e7ed9fc292858554d28d58d56bf62cf964faf0aa57964b21155a0 \\\n    --hash=sha256:ef971ebf2b93bdc88d8337238be4dfb851cc97ed6808eb04870ef67589415171 \\\n    --hash=sha256:f46d525934871ea772930e997d577d48c6983e50f206ff7b66d4ac5f8941e993 \\\n    --hash=sha256:fcdeb38de4f7f3d69d798f4f371189061446792a84e7c92b50054c87aae9c07c \\\n    --hash=sha256:ff62a3022914fc19adaa76b65e03cf62bc67ea16326cbbeb170d280710a7d719\nrequests==2.32.5 \\\n    --hash=sha256:2462f94637a34fd532264295e186976db0f5d453d1cdd31473c85a6a161affb6 \\\n    --hash=sha256:dbba0bac56e100853db0ea71b82b4dfd5fe2bf6d3754a8893c3af500cec7d7cf\nsafetensors==0.6.2 \\\n    --hash=sha256:1d2d2b3ce1e2509c68932ca03ab8f20570920cd9754b05063d4368ee52833ecd \\\n    --hash=sha256:43ff2aa0e6fa2dc3ea5524ac7ad93a9839256b8703761e76e2d0b2a3fa4f15d9 \\\n    --hash=sha256:8045db2c872db8f4cbe3faa0495932d89c38c899c603f21e9b6486951a5ecb8f \\\n    --hash=sha256:81e67e8bab9878bb568cffbc5f5e655adb38d2418351dc0859ccac158f753e19 \\\n    --hash=sha256:89a89b505f335640f9120fac65ddeb83e40f1fd081cb8ed88b505bdccec8d0a1 \\\n    --hash=sha256:93de35a18f46b0f5a6a1f9e26d91b442094f2df02e9fd7acf224cfec4238821a \\\n    --hash=sha256:9c85ede8ec58f120bad982ec47746981e210492a6db876882aa021446af8ffba \\\n    --hash=sha256:b0e4d029ab0a0e0e4fdf142b194514695b1d7d3735503ba700cf36d0fc7136ce \\\n    --hash=sha256:c7b214870df923cbc1593c3faee16bec59ea462758699bd3fee399d00aac072c \\\n    --hash=sha256:cab75ca7c064d3911411461151cb69380c9225798a20e712b102edda2542ddb1 \\\n    --hash=sha256:d6675cf4b39c98dbd7d940598028f3742e0375a6b4d4277e76beb0c35f4b843b \\\n    --hash=sha256:d83c20c12c2d2f465997c51b7ecb00e407e5f94d7dec3ea0cc11d86f60d3fde5 \\\n    --hash=sha256:d944cea65fad0ead848b6ec2c37cc0b197194bec228f8020054742190e9312ac \\\n    --hash=sha256:fa48268185c52bfe8771e46325a1e21d317207bcabcb72e65c6e28e9ffeb29c7 \\\n    --hash=sha256:fc4d0d0b937e04bdf2ae6f70cd3ad51328635fe0e6214aa1fc811f3b576b3bda\nscikit-image==0.25.2 \\\n    --hash=sha256:24cc986e1f4187a12aa319f777b36008764e856e5013666a4a83f8df083c2641 \\\n    --hash=sha256:28182a9d3e2ce3c2e251383bdda68f8d88d9fff1a3ebe1eb61206595c9773341 \\\n    --hash=sha256:330d061bd107d12f8d68f1d611ae27b3b813b8cdb0300a71d07b1379178dd4cd \\\n    --hash=sha256:483bd8cc10c3d8a7a37fae36dfa5b21e239bd4ee121d91cad1f81bba10cfb0ed \\\n    --hash=sha256:5c311069899ce757d7dbf1d03e32acb38bb06153236ae77fcd820fd62044c063 \\\n    --hash=sha256:60516257c5a2d2f74387c502aa2f15a0ef3498fbeaa749f730ab18f0a40fd054 \\\n    --hash=sha256:64785a8acefee460ec49a354706db0b09d1f325674107d7fa3eadb663fb56d6f \\\n    --hash=sha256:7efa888130f6c548ec0439b1a7ed7295bc10105458a421e9bf739b457730b6da \\\n    --hash=sha256:8db8dd03663112783221bf01ccfc9512d1cc50ac9b5b0fe8f4023967564719fb \\\n    --hash=sha256:9d1e80107bcf2bf1291acfc0bf0425dceb8890abe9f38d8e94e23497cbf7ee0d \\\n    --hash=sha256:a17e17eb8562660cc0d31bb55643a4da996a81944b82c54805c91b3fe66f4824 \\\n    --hash=sha256:a4c464b90e978d137330be433df4e76d92ad3c5f46a22f159520ce0fdbea8a09 \\\n    --hash=sha256:b2cfc96b27afe9a05bc92f8c6235321d3a66499995675b27415e0d0c76625173 \\\n    --hash=sha256:b4f6b61fc2db6340696afe3db6b26e0356911529f5f6aee8c322aa5157490c9b \\\n    --hash=sha256:b8abd3c805ce6944b941cfed0406d88faeb19bab3ed3d4b50187af55cf24d147 \\\n    --hash=sha256:bdd2b8c1de0849964dbc54037f36b4e9420157e67e45a8709a80d727f52c7da2 \\\n    --hash=sha256:be455aa7039a6afa54e84f9e38293733a2622b8c2fb3362b822d459cc5605e99 \\\n    --hash=sha256:d3278f586793176599df6a4cf48cb6beadae35c31e58dc01a98023af3dc31c78 \\\n    --hash=sha256:d989d64ff92e0c6c0f2018c7495a5b20e2451839299a018e0e5108b2680f71e0 \\\n    --hash=sha256:dd8011efe69c3641920614d550f5505f83658fe33581e49bed86feab43a180fc \\\n    --hash=sha256:e5a37e6cd4d0c018a7a55b9d601357e3382826d3888c10d0213fc63bff977dde \\\n    --hash=sha256:f4bac9196fb80d37567316581c6060763b0f4893d3aca34a9ede3825bc035b17\nscikit-learn==1.6.1 \\\n    --hash=sha256:0650e730afb87402baa88afbf31c07b84c98272622aaba002559b614600ca691 \\\n    --hash=sha256:0c8d036eb937dbb568c6242fa598d551d88fb4399c0344d95c001980ec1c7d36 \\\n    --hash=sha256:1061b7c028a8663fb9a1a1baf9317b64a257fcb036dae5c8752b2abef31d136f \\\n    --hash=sha256:25fc636bdaf1cc2f4a124a116312d837148b5e10872147bdaf4887926b8c03d8 \\\n    --hash=sha256:2c2cae262064e6a9b77eee1c8e768fc46aa0b8338c6a8297b9b6759720ec0ff2 \\\n    --hash=sha256:2e69fab4ebfc9c9b580a7a80111b43d214ab06250f8a7ef590a4edf72464dd86 \\\n    --hash=sha256:2ffa1e9e25b3d93990e74a4be2c2fc61ee5af85811562f1288d5d055880c4322 \\\n    --hash=sha256:3f59fe08dc03ea158605170eb52b22a105f238a5d512c4470ddeca71feae8e5f \\\n    --hash=sha256:44a17798172df1d3c1065e8fcf9019183f06c87609b49a124ebdf57ae6cb0107 \\\n    --hash=sha256:6849dd3234e87f55dce1db34c89a810b489ead832aaf4d4550b7ea85628be6c1 \\\n    --hash=sha256:6a7aa5f9908f0f28f4edaa6963c0a6183f1911e63a69aa03782f0d924c830a35 \\\n    --hash=sha256:70b1d7e85b1c96383f872a519b3375f92f14731e279a7b4c6cfd650cf5dffc52 \\\n    --hash=sha256:72abc587c75234935e97d09aa4913a82f7b03ee0b74111dcc2881cba3c5a7b33 \\\n    --hash=sha256:775da975a471c4f6f467725dff0ced5c7ac7bda5e9316b260225b48475279a1b \\\n    --hash=sha256:7a1c43c8ec9fde528d664d947dc4c0789be4077a3647f232869f41d9bf50e0fb \\\n    --hash=sha256:7a73d457070e3318e32bdb3aa79a8d990474f19035464dfd8bede2883ab5dc3b \\\n    --hash=sha256:8634c4bd21a2a813e0a7e3900464e6d593162a29dd35d25bdf0103b3fce60ed5 \\\n    --hash=sha256:8a600c31592bd7dab31e1c61b9bbd6dea1b3433e67d264d17ce1017dbdce8002 \\\n    --hash=sha256:926f207c804104677af4857b2c609940b743d04c4c35ce0ddc8ff4f053cddc1b \\\n    --hash=sha256:a17c1dea1d56dcda2fac315712f3651a1fea86565b64b48fa1bc090249cbf236 \\\n    --hash=sha256:b3b00cdc8f1317b5f33191df1386c0befd16625f49d979fe77a8d44cae82410d \\\n    --hash=sha256:b4fc2525eca2c69a59260f583c56a7557c6ccdf8deafdba6e060f94c1c59738e \\\n    --hash=sha256:b8b7a3b86e411e4bce21186e1c180d792f3d99223dcfa3b4f597ecc92fa1a422 \\\n    --hash=sha256:c06beb2e839ecc641366000ca84f3cf6fa9faa1777e29cf0c04be6e4d096a348 \\\n    --hash=sha256:d056391530ccd1e501056160e3c9673b4da4805eb67eb2bdf4e983e1f9c9204e \\\n    --hash=sha256:dc4765af3386811c3ca21638f63b9cf5ecf66261cc4815c1db3f1e7dc7b79db2 \\\n    --hash=sha256:dc5cf3d68c5a20ad6d571584c0750ec641cc46aeef1c1507be51300e6003a7e1 \\\n    --hash=sha256:e7be3fa5d2eb9be7d77c3734ff1d599151bb523674be9b834e8da6abe132f44e \\\n    --hash=sha256:e8ca8cb270fee8f1f76fa9bfd5c3507d60c6438bbee5687f81042e2bb98e5a97 \\\n    --hash=sha256:fa909b1a36e000a03c382aade0bd2063fd5680ff8b8e501660c0f59f021a6415\nscipy==1.15.2 \\\n    --hash=sha256:01edfac9f0798ad6b46d9c4c9ca0e0ad23dbf0b1eb70e96adb9fa7f525eff0bf \\\n    --hash=sha256:03205d57a28e18dfd39f0377d5002725bf1f19a46f444108c29bdb246b6c8a11 \\\n    --hash=sha256:08b57a9336b8e79b305a143c3655cc5bdbe6d5ece3378578888d2afbb51c4e37 \\\n    --hash=sha256:11e7ad32cf184b74380f43d3c0a706f49358b904fa7d5345f16ddf993609184d \\\n    --hash=sha256:28a0d2c2075946346e4408b211240764759e0fabaeb08d871639b5f3b1aca8a0 \\\n    --hash=sha256:2b871df1fe1a3ba85d90e22742b93584f8d2b8e6124f8372ab15c71b73e428b8 \\\n    --hash=sha256:302093e7dfb120e55515936cb55618ee0b895f8bcaf18ff81eca086c17bd80af \\\n    --hash=sha256:42dabaaa798e987c425ed76062794e93a243be8f0f20fff6e7a89f4d61cb3d40 \\\n    --hash=sha256:447ce30cee6a9d5d1379087c9e474628dab3db4a67484be1b7dc3196bfb2fac9 \\\n    --hash=sha256:4c6676490ad76d1c2894d77f976144b41bd1a4052107902238047fb6a473e971 \\\n    --hash=sha256:54c462098484e7466362a9f1672d20888f724911a74c22ae35b61f9c5919183d \\\n    --hash=sha256:597a0c7008b21c035831c39927406c6181bcf8f60a73f36219b69d010aa04737 \\\n    --hash=sha256:5a6fd6eac1ce74a9f77a7fc724080d507c5812d61e72bd5e4c489b042455865e \\\n    --hash=sha256:5ea7ed46d437fc52350b028b1d44e002646e28f3e8ddc714011aaf87330f2f32 \\\n    --hash=sha256:601881dfb761311045b03114c5fe718a12634e5608c3b403737ae463c9885d53 \\\n    --hash=sha256:62ca1ff3eb513e09ed17a5736929429189adf16d2d740f44e53270cc800ecff1 \\\n    --hash=sha256:69ea6e56d00977f355c0f84eba69877b6df084516c602d93a33812aa04d90a3d \\\n    --hash=sha256:6a8e34cf4c188b6dd004654f88586d78f95639e48a25dfae9c5e34a6dc34547e \\\n    --hash=sha256:6d0194c37037707b2afa7a2f2a924cf7bac3dc292d51b6a925e5fcb89bc5c776 \\\n    --hash=sha256:6f223753c6ea76983af380787611ae1291e3ceb23917393079dcc746ba60cfb5 \\\n    --hash=sha256:6f5e296ec63c5da6ba6fa0343ea73fd51b8b3e1a300b0a8cae3ed4b1122c7462 \\\n    --hash=sha256:7cd5b77413e1855351cdde594eca99c1f4a588c2d63711388b6a1f1c01f62274 \\\n    --hash=sha256:869269b767d5ee7ea6991ed7e22b3ca1f22de73ab9a49c44bad338b725603301 \\\n    --hash=sha256:87994da02e73549dfecaed9e09a4f9d58a045a053865679aeb8d6d43747d4df3 \\\n    --hash=sha256:888307125ea0c4466287191e5606a2c910963405ce9671448ff9c81c53f85f58 \\\n    --hash=sha256:92233b2df6938147be6fa8824b8136f29a18f016ecde986666be5f4d686a91a4 \\\n    --hash=sha256:9412f5e408b397ff5641080ed1e798623dbe1ec0d78e72c9eca8992976fa65aa \\\n    --hash=sha256:9b18aa747da280664642997e65aab1dd19d0c3d17068a04b3fe34e2559196cb9 \\\n    --hash=sha256:9de9d1416b3d9e7df9923ab23cd2fe714244af10b763975bea9e4f2e81cebd27 \\\n    --hash=sha256:a2ec871edaa863e8213ea5df811cd600734f6400b4af272e1c011e69401218e9 \\\n    --hash=sha256:a5080a79dfb9b78b768cebf3c9dcbc7b665c5875793569f48bf0e2b1d7f68f6f \\\n    --hash=sha256:a8bf5cb4a25046ac61d38f8d3c3426ec11ebc350246a4642f2f315fe95bda655 \\\n    --hash=sha256:b09ae80010f52efddb15551025f9016c910296cf70adbf03ce2a8704f3a5ad20 \\\n    --hash=sha256:b5e025e903b4f166ea03b109bb241355b9c42c279ea694d8864d033727205e65 \\\n    --hash=sha256:bad78d580270a4d32470563ea86c6590b465cb98f83d760ff5b0990cb5518a93 \\\n    --hash=sha256:bae43364d600fdc3ac327db99659dcb79e6e7ecd279a75fe1266669d9a652828 \\\n    --hash=sha256:c4697a10da8f8765bb7c83e24a470da5797e37041edfd77fd95ba3811a47c4fd \\\n    --hash=sha256:c90ebe8aaa4397eaefa8455a8182b164a6cc1d59ad53f79943f266d99f68687f \\\n    --hash=sha256:cd58a314d92838f7e6f755c8a2167ead4f27e1fd5c1251fd54289569ef3495ec \\\n    --hash=sha256:cf72ff559a53a6a6d77bd8eefd12a17995ffa44ad86c77a5df96f533d4e6c6bb \\\n    --hash=sha256:def751dd08243934c884a3221156d63e15234a3155cf25978b0a668409d45eb6 \\\n    --hash=sha256:e7c68b6a43259ba0aab737237876e5c2c549a031ddb7abc28c7b47f22e202ded \\\n    --hash=sha256:ecf797d2d798cf7c838c6d98321061eb3e72a74710e6c40540f0e8087e3b499e \\\n    --hash=sha256:f031846580d9acccd0044efd1a90e6f4df3a6e12b4b6bd694a7bc03a89892b28 \\\n    --hash=sha256:fb530e4794fc8ea76a4a21ccb67dea33e5e0e60f07fc38a49e821e1eae3b71a0 \\\n    --hash=sha256:fe8a9eb875d430d81755472c5ba75e84acc980e4a8f6204d402849234d3017db\nsentry-sdk==2.36.0 \\\n    --hash=sha256:0f95586a141068d215376e5bf8ebd279e126f7f42805e9570190ef82a7e232b3 \\\n    --hash=sha256:af9260e8155e41e8217615a453828e98aa40740865ac4b16b1ccb6a63b4b2e31\nsetproctitle==1.3.6 \\\n    --hash=sha256:082413db8a96b1f021088e8ec23f0a61fec352e649aba20881895815388b66d3 \\\n    --hash=sha256:0dba8faee2e4a96e934797c9f0f2d093f8239bf210406a99060b3eabe549628e \\\n    --hash=sha256:0e6b5633c94c5111f7137f875e8f1ff48f53b991d5d5b90932f27dc8c1fa9ae4 \\\n    --hash=sha256:1065ed36bd03a3fd4186d6c6de5f19846650b015789f72e2dea2d77be99bdca1 \\\n    --hash=sha256:109fc07b1cd6cef9c245b2028e3e98e038283342b220def311d0239179810dbe \\\n    --hash=sha256:13624d9925bb481bc0ccfbc7f533da38bfbfe6e80652314f789abc78c2e513bd \\\n    --hash=sha256:156795b3db976611d09252fc80761fcdb65bb7c9b9581148da900851af25ecf4 \\\n    --hash=sha256:163dba68f979c61e4e2e779c4d643e968973bdae7c33c3ec4d1869f7a9ba8390 \\\n    --hash=sha256:17d7c833ed6545ada5ac4bb606b86a28f13a04431953d4beac29d3773aa00b1d \\\n    --hash=sha256:18d0667bafaaae4c1dee831e2e59841c411ff399b9b4766822ba2685d419c3be \\\n    --hash=sha256:1aa1935aa2195b76f377e5cb018290376b7bf085f0b53f5a95c0c21011b74367 \\\n    --hash=sha256:2156d55308431ac3b3ec4e5e05b1726d11a5215352d6a22bb933171dee292f8c \\\n    --hash=sha256:23a57d3b8f1549515c2dbe4a2880ebc1f27780dc126c5e064167563e015817f5 \\\n    --hash=sha256:2407955dc359d735a20ac6e797ad160feb33d529a2ac50695c11a1ec680eafab \\\n    --hash=sha256:2940cf13f4fc11ce69ad2ed37a9f22386bfed314b98d8aebfd4f55459aa59108 \\\n    --hash=sha256:2e51ec673513465663008ce402171192a053564865c2fc6dc840620871a9bd7c \\\n    --hash=sha256:3393859eb8f19f5804049a685bf286cb08d447e28ba5c6d8543c7bf5500d5970 \\\n    --hash=sha256:3884002b3a9086f3018a32ab5d4e1e8214dd70695004e27b1a45c25a6243ad0b \\\n    --hash=sha256:38ca045626af693da042ac35d7332e7b9dbd52e6351d6973b310612e3acee6d6 \\\n    --hash=sha256:391bb6a29c4fe7ccc9c30812e3744060802d89b39264cfa77f3d280d7f387ea5 \\\n    --hash=sha256:3cca16fd055316a48f0debfcbfb6af7cea715429fc31515ab3fcac05abd527d8 \\\n    --hash=sha256:3cde5b83ec4915cd5e6ae271937fd60d14113c8f7769b4a20d51769fe70d8717 \\\n    --hash=sha256:3f8194b4d631b003a1176a75d1acd545e04b1f54b821638e098a93e6e62830ef \\\n    --hash=sha256:3fc97805f9d74444b027babff710bf39df1541437a6a585a983d090ae00cedde \\\n    --hash=sha256:4431629c178193f23c538cb1de3da285a99ccc86b20ee91d81eb5f1a80e0d2ba \\\n    --hash=sha256:49498ebf68ca3e75321ffe634fcea5cc720502bfaa79bd6b03ded92ce0dc3c24 \\\n    --hash=sha256:4ac3eb04bcf0119aadc6235a2c162bae5ed5f740e3d42273a7228b915722de20 \\\n    --hash=sha256:4adf6a0013fe4e0844e3ba7583ec203ca518b9394c6cc0d3354df2bf31d1c034 \\\n    --hash=sha256:4efc91b437f6ff2578e89e3f17d010c0a0ff01736606473d082913ecaf7859ba \\\n    --hash=sha256:50706b9c0eda55f7de18695bfeead5f28b58aa42fd5219b3b1692d554ecbc9ec \\\n    --hash=sha256:5313a4e9380e46ca0e2c681ba739296f9e7c899e6f4d12a6702b2dc9fb846a31 \\\n    --hash=sha256:543f59601a4e32daf44741b52f9a23e0ee374f9f13b39c41d917302d98fdd7b0 \\\n    --hash=sha256:57bc54763bf741813a99fbde91f6be138c8706148b7b42d3752deec46545d470 \\\n    --hash=sha256:63cc10352dc6cf35a33951656aa660d99f25f574eb78132ce41a85001a638aa7 \\\n    --hash=sha256:6a1d3aa13acfe81f355b0ce4968facc7a19b0d17223a0f80c011a1dba8388f37 \\\n    --hash=sha256:6af330ddc2ec05a99c3933ab3cba9365357c0b8470a7f2fa054ee4b0984f57d1 \\\n    --hash=sha256:6d50bfcc1d1692dc55165b3dd2f0b9f8fb5b1f7b571a93e08d660ad54b9ca1a5 \\\n    --hash=sha256:70100e2087fe05359f249a0b5f393127b3a1819bf34dec3a3e0d4941138650c9 \\\n    --hash=sha256:74973aebea3543ad033b9103db30579ec2b950a466e09f9c2180089e8346e0ec \\\n    --hash=sha256:751ba352ed922e0af60458e961167fa7b732ac31c0ddd1476a2dfd30ab5958c5 \\\n    --hash=sha256:785cd210c0311d9be28a70e281a914486d62bfd44ac926fcd70cf0b4d65dff1c \\\n    --hash=sha256:7890e291bf4708e3b61db9069ea39b3ab0651e42923a5e1f4d78a7b9e4b18301 \\\n    --hash=sha256:793a23e8d9cb6c231aa3023d700008224c6ec5b8fd622d50f3c51665e3d0a190 \\\n    --hash=sha256:797f2846b546a8741413c57d9fb930ad5aa939d925c9c0fa6186d77580035af7 \\\n    --hash=sha256:7df5fcc48588f82b6cc8073db069609ddd48a49b1e9734a20d0efb32464753c4 \\\n    --hash=sha256:8050c01331135f77ec99d99307bfbc6519ea24d2f92964b06f3222a804a3ff1f \\\n    --hash=sha256:805bb33e92fc3d8aa05674db3068d14d36718e3f2c5c79b09807203f229bf4b5 \\\n    --hash=sha256:807796fe301b7ed76cf100113cc008c119daf4fea2f9f43c578002aef70c3ebf \\\n    --hash=sha256:81c443310831e29fabbd07b75ebbfa29d0740b56f5907c6af218482d51260431 \\\n    --hash=sha256:83066ffbf77a5f82b7e96e59bdccbdda203c8dccbfc3f9f0fdad3a08d0001d9c \\\n    --hash=sha256:8834ab7be6539f1bfadec7c8d12249bbbe6c2413b1d40ffc0ec408692232a0c6 \\\n    --hash=sha256:92df0e70b884f5da35f2e01489dca3c06a79962fb75636985f1e3a17aec66833 \\\n    --hash=sha256:9483aa336687463f5497dd37a070094f3dff55e2c888994f8440fcf426a1a844 \\\n    --hash=sha256:97a138fa875c6f281df7720dac742259e85518135cd0e3551aba1c628103d853 \\\n    --hash=sha256:9b50700785eccac0819bea794d968ed8f6055c88f29364776b7ea076ac105c5d \\\n    --hash=sha256:9b73cf0fe28009a04a35bb2522e4c5b5176cc148919431dcb73fdbdfaab15781 \\\n    --hash=sha256:9d5a369eb7ec5b2fdfa9927530b5259dd21893fa75d4e04a223332f61b84b586 \\\n    --hash=sha256:a094b7ce455ca341b59a0f6ce6be2e11411ba6e2860b9aa3dbb37468f23338f4 \\\n    --hash=sha256:a0d6252098e98129a1decb59b46920d4eca17b0395f3d71b0d327d086fefe77d \\\n    --hash=sha256:a1d856b0f4e4a33e31cdab5f50d0a14998f3a2d726a3fd5cb7c4d45a57b28d1b \\\n    --hash=sha256:a4ae2ea9afcfdd2b931ddcebf1cf82532162677e00326637b31ed5dff7d985ca \\\n    --hash=sha256:a5963b663da69ad25fa1559ee064584935570def665917918938c1f1289f5ebc \\\n    --hash=sha256:ad1c2c2baaba62823a7f348f469a967ece0062140ca39e7a48e4bbb1f20d54c4 \\\n    --hash=sha256:ae82507fe458f7c0c8227017f2158111a4c9e7ce94de05178894a7ea9fefc8a1 \\\n    --hash=sha256:af188f3305f0a65c3217c30c6d4c06891e79144076a91e8b454f14256acc7279 \\\n    --hash=sha256:af44bb7a1af163806bbb679eb8432fa7b4fb6d83a5d403b541b675dcd3798638 \\\n    --hash=sha256:b0174ca6f3018ddeaa49847f29b69612e590534c1d2186d54ab25161ecc42975 \\\n    --hash=sha256:b2b17855ed7f994f3f259cf2dfbfad78814538536fa1a91b50253d84d87fd88d \\\n    --hash=sha256:b2e54f4a2dc6edf0f5ea5b1d0a608d2af3dcb5aa8c8eeab9c8841b23e1b054fe \\\n    --hash=sha256:b6f4abde9a2946f57e8daaf1160b2351bcf64274ef539e6675c1d945dbd75e2a \\\n    --hash=sha256:b70c07409d465f3a8b34d52f863871fb8a00755370791d2bd1d4f82b3cdaf3d5 \\\n    --hash=sha256:bb465dd5825356c1191a038a86ee1b8166e3562d6e8add95eec04ab484cfb8a2 \\\n    --hash=sha256:c051f46ed1e13ba8214b334cbf21902102807582fbfaf0fef341b9e52f0fafbf \\\n    --hash=sha256:c1b20a5f4164cec7007be55c9cf18d2cd08ed7c3bf6769b3cd6d044ad888d74b \\\n    --hash=sha256:c86e9e82bfab579327dbe9b82c71475165fbc8b2134d24f9a3b2edaf200a5c3d \\\n    --hash=sha256:c9f32b96c700bb384f33f7cf07954bb609d35dd82752cef57fb2ee0968409169 \\\n    --hash=sha256:cce0ed8b3f64c71c140f0ec244e5fdf8ecf78ddf8d2e591d4a8b6aa1c1214235 \\\n    --hash=sha256:cdd7315314b0744a7dd506f3bd0f2cf90734181529cdcf75542ee35ad885cab7 \\\n    --hash=sha256:cf355fbf0d4275d86f9f57be705d8e5eaa7f8ddb12b24ced2ea6cbd68fdb14dc \\\n    --hash=sha256:d136fbf8ad4321716e44d6d6b3d8dffb4872626010884e07a1db54b7450836cf \\\n    --hash=sha256:d2c8e20487b3b73c1fa72c56f5c89430617296cd380373e7af3a538a82d4cd6d \\\n    --hash=sha256:d483cc23cc56ab32911ea0baa0d2d9ea7aa065987f47de847a0a93a58bf57905 \\\n    --hash=sha256:d5a6c4864bb6fa9fcf7b57a830d21aed69fd71742a5ebcdbafda476be673d212 \\\n    --hash=sha256:d714e002dd3638170fe7376dc1b686dbac9cb712cde3f7224440af722cc9866a \\\n    --hash=sha256:d73f14b86d0e2858ece6bf5807c9889670e392c001d414b4293d0d9b291942c3 \\\n    --hash=sha256:d88c63bd395c787b0aa81d8bbc22c1809f311032ce3e823a6517b711129818e4 \\\n    --hash=sha256:db608db98ccc21248370d30044a60843b3f0f3d34781ceeea67067c508cd5a28 \\\n    --hash=sha256:de004939fc3fd0c1200d26ea9264350bfe501ffbf46c8cf5dc7f345f2d87a7f1 \\\n    --hash=sha256:ded9e86397267732a0641d4776c7c663ea16b64d7dbc4d9cc6ad8536363a2d29 \\\n    --hash=sha256:e288f8a162d663916060beb5e8165a8551312b08efee9cf68302687471a6545d \\\n    --hash=sha256:e2a9e62647dc040a76d55563580bf3bb8fe1f5b6ead08447c2ed0d7786e5e794 \\\n    --hash=sha256:e3e44d08b61de0dd6f205528498f834a51a5c06689f8fb182fe26f3a3ce7dca9 \\\n    --hash=sha256:ea002088d5554fd75e619742cefc78b84a212ba21632e59931b3501f0cfc8f67 \\\n    --hash=sha256:eb7452849f6615871eabed6560ffedfe56bc8af31a823b6be4ce1e6ff0ab72c5 \\\n    --hash=sha256:ebcf34b69df4ca0eabaaaf4a3d890f637f355fed00ba806f7ebdd2d040658c26 \\\n    --hash=sha256:f24d5b9383318cbd1a5cd969377937d66cf0542f24aa728a4f49d9f98f9c0da8 \\\n    --hash=sha256:f33fbf96b52d51c23b6cff61f57816539c1c147db270cfc1cc3bc012f4a560a9\nsetuptools==75.8.0 \\\n    --hash=sha256:c5afc8f407c626b8313a86e10311dd3f661c6cd9c09d4bf8c15c0e11f9f2b0e6 \\\n    --hash=sha256:e3982f444617239225d675215d51f6ba05f845d4eec313da4418fdbb56fb27e3\nsix==1.17.0 \\\n    --hash=sha256:4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274 \\\n    --hash=sha256:ff70335d468e7eb6ec65b95b99d3a2836546063f63acc5171de367e834932a81\nsmmap==5.0.2 \\\n    --hash=sha256:26ea65a03958fa0c8a1c7e8c7a58fdc77221b8910f6be2131affade476898ad5 \\\n    --hash=sha256:b30115f0def7d7531d22a0fb6502488d879e75b260a9db4d0819cfb25403af5e\nstack-data==0.6.3 \\\n    --hash=sha256:836a778de4fec4dcd1dcd89ed8abff8a221f58308462e1c4aa2a3cf30148f0b9 \\\n    --hash=sha256:d5558e0c25a4cb0853cddad3d77da9891a08cb85dd9f9f91b9f8cd66e511e695\nsubmitit==1.5.2 \\\n    --hash=sha256:36a8a54ad4e10171111e7618eefe28fe819f931a89c9cd1f6d2770900c013f12 \\\n    --hash=sha256:c6d5867fbcc78588d0ded3338436903f8db9fdb759f80e9639e6025a9ea32ade\nsympy==1.13.1 \\\n    --hash=sha256:9cebf7e04ff162015ce31c9c6c9144daa34a93bd082f54fd8f12deca4f47515f \\\n    --hash=sha256:db36cdc64bf61b9b24578b6f7bab1ecdd2452cf008f34faa33776680c26d66f8\nthreadpoolctl==3.6.0 \\\n    --hash=sha256:43a0b8fd5a2928500110039e43a5eed8480b918967083ea48dc3ab9f13c4a7fb \\\n    --hash=sha256:8ab8b4aa3491d812b623328249fab5302a68d2d71745c8a4c719a2fcaba9f44e\ntifffile==2025.8.28 \\\n    --hash=sha256:82929343c70f6f776983f6a817f0b92e913a1bbb3dc3f436af44419b872bb467 \\\n    --hash=sha256:b274a6d9eeba65177cf7320af25ef38ecf910b3369ac6bc494a94a3f6bd99c78\ntokenizers==0.19.1 \\\n    --hash=sha256:01d62812454c188306755c94755465505836fd616f75067abcae529c35edeb57 \\\n    --hash=sha256:02e81bf089ebf0e7f4df34fa0207519f07e66d8491d963618252f2e0729e0b46 \\\n    --hash=sha256:04ce49e82d100594715ac1b2ce87d1a36e61891a91de774755f743babcd0dd52 \\\n    --hash=sha256:07f9295349bbbcedae8cefdbcfa7f686aa420be8aca5d4f7d1ae6016c128c0c5 \\\n    --hash=sha256:08a44864e42fa6d7d76d7be4bec62c9982f6f6248b4aa42f7302aa01e0abfd26 \\\n    --hash=sha256:0b5ca92bfa717759c052e345770792d02d1f43b06f9e790ca0a1db62838816f3 \\\n    --hash=sha256:0b9394bd204842a2a1fd37fe29935353742be4a3460b6ccbaefa93f58a8df43d \\\n    --hash=sha256:0bcce02bf1ad9882345b34d5bd25ed4949a480cf0e656bbd468f4d8986f7a3f1 \\\n    --hash=sha256:0e64bfde9a723274e9a71630c3e9494ed7b4c0f76a1faacf7fe294cd26f7ae7c \\\n    --hash=sha256:10a707cc6c4b6b183ec5dbfc5c34f3064e18cf62b4a938cb41699e33a99e03c1 \\\n    --hash=sha256:16baac68651701364b0289979ecec728546133e8e8fe38f66fe48ad07996b88b \\\n    --hash=sha256:1de5bc8652252d9357a666e609cb1453d4f8e160eb1fb2830ee369dd658e8975 \\\n    --hash=sha256:1f0360cbea28ea99944ac089c00de7b2e3e1c58f479fb8613b6d8d511ce98267 \\\n    --hash=sha256:2e8a3dd055e515df7054378dc9d6fa8c8c34e1f32777fb9a01fea81496b3f9d3 \\\n    --hash=sha256:3174c76efd9d08f836bfccaca7cfec3f4d1c0a4cf3acbc7236ad577cc423c840 \\\n    --hash=sha256:35583cd46d16f07c054efd18b5d46af4a2f070a2dd0a47914e66f3ff5efb2b1e \\\n    --hash=sha256:39c1ec76ea1027438fafe16ecb0fb84795e62e9d643444c1090179e63808c69d \\\n    --hash=sha256:3b11853f17b54c2fe47742c56d8a33bf49ce31caf531e87ac0d7d13d327c9334 \\\n    --hash=sha256:427c4f0f3df9109314d4f75b8d1f65d9477033e67ffaec4bca53293d3aca286d \\\n    --hash=sha256:43350270bfc16b06ad3f6f07eab21f089adb835544417afda0f83256a8bf8b75 \\\n    --hash=sha256:453e4422efdfc9c6b6bf2eae00d5e323f263fff62b29a8c9cd526c5003f3f642 \\\n    --hash=sha256:4692ab92f91b87769d950ca14dbb61f8a9ef36a62f94bad6c82cc84a51f76f6a \\\n    --hash=sha256:4ad23d37d68cf00d54af184586d79b84075ada495e7c5c0f601f051b162112dc \\\n    --hash=sha256:4f3fefdc0446b1a1e6d81cd4c07088ac015665d2e812f6dbba4a06267d1a2c95 \\\n    --hash=sha256:56ae39d4036b753994476a1b935584071093b55c7a72e3b8288e68c313ca26e7 \\\n    --hash=sha256:5c88d1481f1882c2e53e6bb06491e474e420d9ac7bdff172610c4f9ad3898059 \\\n    --hash=sha256:61b7fe8886f2e104d4caf9218b157b106207e0f2a4905c9c7ac98890688aabeb \\\n    --hash=sha256:621d670e1b1c281a1c9698ed89451395d318802ff88d1fc1accff0867a06f153 \\\n    --hash=sha256:6258c2ef6f06259f70a682491c78561d492e885adeaf9f64f5389f78aa49a051 \\\n    --hash=sha256:6309271f57b397aa0aff0cbbe632ca9d70430839ca3178bf0f06f825924eca22 \\\n    --hash=sha256:638e43936cc8b2cbb9f9d8dde0fe5e7e30766a3318d2342999ae27f68fdc9bd6 \\\n    --hash=sha256:63c38f45d8f2a2ec0f3a20073cccb335b9f99f73b3c69483cd52ebc75369d8a1 \\\n    --hash=sha256:670b802d4d82bbbb832ddb0d41df7015b3e549714c0e77f9bed3e74d42400fbe \\\n    --hash=sha256:6852c5b2a853b8b0ddc5993cd4f33bfffdca4fcc5d52f89dd4b8eada99379285 \\\n    --hash=sha256:6b2da5c32ed869bebd990c9420df49813709e953674c0722ff471a116d97b22d \\\n    --hash=sha256:6c330c0eb815d212893c67a032e9dc1b38a803eccb32f3e8172c19cc69fbb439 \\\n    --hash=sha256:6f8a20266e695ec9d7a946a019c1d5ca4eddb6613d4f466888eee04f16eedb85 \\\n    --hash=sha256:706a37cc5332f85f26efbe2bdc9ef8a9b372b77e4645331a405073e4b3a8c1c6 \\\n    --hash=sha256:71e3ec71f0e78780851fef28c2a9babe20270404c921b756d7c532d280349214 \\\n    --hash=sha256:72791f9bb1ca78e3ae525d4782e85272c63faaef9940d92142aa3eb79f3407a3 \\\n    --hash=sha256:76951121890fea8330d3a0df9a954b3f2a37e3ec20e5b0530e9a0044ca2e11fe \\\n    --hash=sha256:78e769eb3b2c79687d9cb0f89ef77223e8e279b75c0a968e637ca7043a84463f \\\n    --hash=sha256:7c9d5b6c0e7a1e979bec10ff960fae925e947aab95619a6fdb4c1d8ff3708ce3 \\\n    --hash=sha256:7fb297edec6c6841ab2e4e8f357209519188e4a59b557ea4fafcf4691d1b4c98 \\\n    --hash=sha256:7ff898780a155ea053f5d934925f3902be2ed1f4d916461e1a93019cc7250837 \\\n    --hash=sha256:82c8b8063de6c0468f08e82c4e198763e7b97aabfe573fd4cf7b33930ca4df77 \\\n    --hash=sha256:85aa3ab4b03d5e99fdd31660872249df5e855334b6c333e0bc13032ff4469c4a \\\n    --hash=sha256:89183e55fb86e61d848ff83753f64cded119f5d6e1f553d14ffee3700d0a4a49 \\\n    --hash=sha256:8a6298bde623725ca31c9035a04bf2ef63208d266acd2bed8c2cb7d2b7d53ce6 \\\n    --hash=sha256:8b01afb7193d47439f091cd8f070a1ced347ad0f9144952a30a41836902fe09e \\\n    --hash=sha256:952078130b3d101e05ecfc7fc3640282d74ed26bcf691400f872563fca15ac97 \\\n    --hash=sha256:952b80dac1a6492170f8c2429bd11fcaa14377e097d12a1dbe0ef2fb2241e16c \\\n    --hash=sha256:9620b78e0b2d52ef07b0d428323fb34e8ea1219c5eac98c2596311f20f1f9266 \\\n    --hash=sha256:9ed240c56b4403e22b9584ee37d87b8bfa14865134e3e1c3fb4b2c42fafd3256 \\\n    --hash=sha256:a179856d1caee06577220ebcfa332af046d576fb73454b8f4d4b0ba8324423ea \\\n    --hash=sha256:a2b718f316b596f36e1dae097a7d5b91fc5b85e90bf08b01ff139bd8953b25af \\\n    --hash=sha256:ac11016d0a04aa6487b1513a3a36e7bee7eec0e5d30057c9c0408067345c48d2 \\\n    --hash=sha256:ad57d59341710b94a7d9dbea13f5c1e7d76fd8d9bcd944a7a6ab0b0da6e0cc66 \\\n    --hash=sha256:b07c538ba956843833fee1190cf769c60dc62e1cf934ed50d77d5502194d63b1 \\\n    --hash=sha256:b279ab506ec4445166ac476fb4d3cc383accde1ea152998509a94d82547c8e2a \\\n    --hash=sha256:b2edbc75744235eea94d595a8b70fe279dd42f3296f76d5a86dde1d46e35f574 \\\n    --hash=sha256:b342d2ce8fc8d00f376af068e3274e2e8649562e3bc6ae4a67784ded6b99428d \\\n    --hash=sha256:b4399b59d1af5645bcee2072a463318114c39b8547437a7c2d6a186a1b5a0e2d \\\n    --hash=sha256:b4c89aa46c269e4e70c4d4f9d6bc644fcc39bb409cb2a81227923404dd6f5227 \\\n    --hash=sha256:b70bfbe3a82d3e3fb2a5e9b22a39f8d1740c96c68b6ace0086b39074f08ab89a \\\n    --hash=sha256:b82931fa619dbad979c0ee8e54dd5278acc418209cc897e42fac041f5366d626 \\\n    --hash=sha256:bac0b0eb952412b0b196ca7a40e7dce4ed6f6926489313414010f2e6b9ec2adf \\\n    --hash=sha256:bb9dfe7dae85bc6119d705a76dc068c062b8b575abe3595e3c6276480e67e3f1 \\\n    --hash=sha256:bcd266ae85c3d39df2f7e7d0e07f6c41a55e9a3123bb11f854412952deacd828 \\\n    --hash=sha256:bea6f9947e9419c2fda21ae6c32871e3d398cba549b93f4a65a2d369662d9403 \\\n    --hash=sha256:c27b99889bd58b7e301468c0838c5ed75e60c66df0d4db80c08f43462f82e0d3 \\\n    --hash=sha256:c2a0d47a89b48d7daa241e004e71fb5a50533718897a4cd6235cb846d511a478 \\\n    --hash=sha256:c5c2ff13d157afe413bf7e25789879dd463e5a4abfb529a2d8f8473d8042e28f \\\n    --hash=sha256:c85cf76561fbd01e0d9ea2d1cbe711a65400092bc52b5242b16cfd22e51f0c58 \\\n    --hash=sha256:ca407133536f19bdec44b3da117ef0d12e43f6d4b56ac4c765f37eca501c7bda \\\n    --hash=sha256:cbf001afbbed111a79ca47d75941e9e5361297a87d186cbfc11ed45e30b5daba \\\n    --hash=sha256:ce05fde79d2bc2e46ac08aacbc142bead21614d937aac950be88dc79f9db9022 \\\n    --hash=sha256:d16ff18907f4909dca9b076b9c2d899114dd6abceeb074eca0c93e2353f943aa \\\n    --hash=sha256:d26194ef6c13302f446d39972aaa36a1dda6450bc8949f5eb4c27f51191375bd \\\n    --hash=sha256:d8c5d59d7b59885eab559d5bc082b2985555a54cda04dda4c65528d90ad252ad \\\n    --hash=sha256:d924204a3dbe50b75630bd16f821ebda6a5f729928df30f582fb5aade90c818a \\\n    --hash=sha256:dadc509cc8a9fe460bd274c0e16ac4184d0958117cf026e0ea8b32b438171594 \\\n    --hash=sha256:dd26e3afe8a7b61422df3176e06664503d3f5973b94f45d5c45987e1cb711876 \\\n    --hash=sha256:ddf672ed719b4ed82b51499100f5417d7d9f6fb05a65e232249268f35de5ed14 \\\n    --hash=sha256:dfedf31824ca4915b511b03441784ff640378191918264268e6923da48104acc \\\n    --hash=sha256:e28cab1582e0eec38b1f38c1c1fb2e56bce5dc180acb1724574fc5f47da2a4fe \\\n    --hash=sha256:e742d76ad84acbdb1a8e4694f915fe59ff6edc381c97d6dfdd054954e3478ad4 \\\n    --hash=sha256:e83a31c9cf181a0a3ef0abad2b5f6b43399faf5da7e696196ddd110d332519ee \\\n    --hash=sha256:e8d1ed93beda54bbd6131a2cb363a576eac746d5c26ba5b7556bc6f964425594 \\\n    --hash=sha256:e8ff5b90eabdcdaa19af697885f70fe0b714ce16709cf43d4952f1f85299e73a \\\n    --hash=sha256:ec11802450a2487cdf0e634b750a04cbdc1c4d066b97d94ce7dd2cb51ebb325b \\\n    --hash=sha256:ecb2651956eea2aa0a2d099434134b1b68f1c31f9a5084d6d53f08ed43d45ff2 \\\n    --hash=sha256:ed69af290c2b65169f0ba9034d1dc39a5db9459b32f1dd8b5f3f32a3fcf06eab \\\n    --hash=sha256:eddd5783a4a6309ce23432353cdb36220e25cbb779bfa9122320666508b44b88 \\\n    --hash=sha256:ee59e6680ed0fdbe6b724cf38bd70400a0c1dd623b07ac729087270caeac88e3 \\\n    --hash=sha256:f03727225feaf340ceeb7e00604825addef622d551cbd46b7b775ac834c1e1c4 \\\n    --hash=sha256:f3bbb7a0c5fcb692950b041ae11067ac54826204318922da754f908d95619fbc \\\n    --hash=sha256:f8a9c828277133af13f3859d1b6bf1c3cb6e9e1637df0e45312e6b7c2e622b1f \\\n    --hash=sha256:f97660f6c43efd3e0bfd3f2e3e5615bf215680bad6ee3d469df6454b8c6e8256 \\\n    --hash=sha256:f9939ca7e58c2758c01b40324a59c034ce0cebad18e0d4563a9b1beab3018243\ntorch==2.6.0 \\\n    --hash=sha256:09e06f9949e1a0518c5b09fe95295bc9661f219d9ecb6f9893e5123e10696628 \\\n    --hash=sha256:265f70de5fd45b864d924b64be1797f86e76c8e48a02c2a3a6fc7ec247d2226c \\\n    --hash=sha256:2bb8987f3bb1ef2675897034402373ddfc8f5ef0e156e2d8cfc47cacafdda4a9 \\\n    --hash=sha256:46763dcb051180ce1ed23d1891d9b1598e07d051ce4c9d14307029809c4d64f7 \\\n    --hash=sha256:4874a73507a300a5d089ceaff616a569e7bb7c613c56f37f63ec3ffac65259cf \\\n    --hash=sha256:510c73251bee9ba02ae1cb6c9d4ee0907b3ce6020e62784e2d7598e0cfa4d6cc \\\n    --hash=sha256:56eeaf2ecac90da5d9e35f7f35eb286da82673ec3c582e310a8d1631a1c02341 \\\n    --hash=sha256:683410f97984103148e31b38a8631acf31c3034c020c0f4d26171e7626d8317a \\\n    --hash=sha256:6860df13d9911ac158f4c44031609700e1eba07916fff62e21e6ffa0a9e01961 \\\n    --hash=sha256:7979834102cd5b7a43cc64e87f2f3b14bd0e1458f06e9f88ffa386d07c7446e1 \\\n    --hash=sha256:7e1448426d0ba3620408218b50aa6ada88aeae34f7a239ba5431f6c8774b1239 \\\n    --hash=sha256:94fc63b3b4bedd327af588696559f68c264440e2503cc9e6954019473d74ae21 \\\n    --hash=sha256:9a610afe216a85a8b9bc9f8365ed561535c93e804c2a317ef7fabcc5deda0989 \\\n    --hash=sha256:9ea955317cfcd3852b1402b62af258ce735c2edeee42ca9419b6bc889e5ae053 \\\n    --hash=sha256:a0d5e1b9874c1a6c25556840ab8920569a7a4137afa8a63a32cee0bc7d89bd4b \\\n    --hash=sha256:b789069020c5588c70d5c2158ac0aa23fd24a028f34a8b4fcb8fcb4d7efcf5fb \\\n    --hash=sha256:bb2c6c3e65049f081940f5ab15c9136c7de40d3f01192541c920a07c7c585b7e \\\n    --hash=sha256:c4f103a49830ce4c7561ef4434cc7926e5a5fe4e5eb100c19ab36ea1e2b634ab \\\n    --hash=sha256:ccbd0320411fe1a3b3fec7b4d3185aa7d0c52adac94480ab024b5c8f74a0bf1d \\\n    --hash=sha256:ff96f4038f8af9f7ec4231710ed4549da1bdebad95923953a25045dcf6fd87e2\ntorchaudio==2.6.0 \\\n    --hash=sha256:04803a969710bdb77a4ddfdb85a32fa9b9e0310dc91f7eb7e54d6083dd69bfab \\\n    --hash=sha256:0eda1cd876f44fc014dc04aa680db2fa355a83df5d834398db6dd5f5cd911f4c \\\n    --hash=sha256:0f0db5c997d031c34066d8be1c0ce7d2a1f2b6c016a92885b20b00bfeb17b753 \\\n    --hash=sha256:22798d5d8e37869bd5875d37f42270efbeb8ae94bda97fed40c1c5e0e1c62fa3 \\\n    --hash=sha256:377b177a3d683a9163e4cab5a06f0346dac9ff96fa527477338fd90fc6a2a4b6 \\\n    --hash=sha256:393fa74ec40d167f0170728ea21c9b5e0f830648fd02df7db2bf7e62f64245ec \\\n    --hash=sha256:52182f6de4e7b342d139e54b703185d428de9cce3c4cf914a9b2ab2359d192a3 \\\n    --hash=sha256:52f15185349c370fc1faa84e8b8b2782c007472db9d586a16bba314130b322f2 \\\n    --hash=sha256:6291d9507dc1d6b4ffe8843fbfb201e6c8270dd8c42ad70bb76226c0ebdcad56 \\\n    --hash=sha256:66f2e0bd5ab56fd81419d2f5afb74a9a70141688594646441756c8c24f424a73 \\\n    --hash=sha256:715aa21f6bdbd085454c313ae3a2c7cc07bf2e8cf05752f819afb5b4c57f4e6f \\\n    --hash=sha256:72e77055d8e742475c6dfacf59fab09b1fc94d4423e14897e188b67cad3851c6 \\\n    --hash=sha256:7d0e4b08c42325bf4b887de9a25c44ed882997001740e1bd7d901f65581cf1ab \\\n    --hash=sha256:86d6239792bf94741a41acd6fe3d549faaf0d50e7275d17d076a190bd007e2f9 \\\n    --hash=sha256:8c1a4d08e35a9ceaadadbff6e60bcb3442482f800369be350103dfd08b4ddf52 \\\n    --hash=sha256:9d8e07789452efdb8132d62afe21f2293a72805f26c2891c6c53e4e4df38ddf6 \\\n    --hash=sha256:b521ea9618fb4c29a6f8071628170c222291f46a48a3bf424cfeb488f54af714 \\\n    --hash=sha256:c12fc41241b8dfce3ccc1917f1c81a0f92f532d9917706600046f1eb21d2d765 \\\n    --hash=sha256:c6386bfa478afae2137715bb60f35520e3b05f5fc6d3bcc6969cf9cdfb11c09c \\\n    --hash=sha256:d855da878a28c2e5e6fb3d76fcddd544f4d957a320b29602cea5af2fe0ad1f3a\ntorchvision==0.21.0 \\\n    --hash=sha256:044ea420b8c6c3162a234cada8e2025b9076fa82504758cd11ec5d0f8cd9fa37 \\\n    --hash=sha256:084ac3f5a1f50c70d630a488d19bf62f323018eae1b1c1232f2b7047d3a7b76d \\\n    --hash=sha256:110d115333524d60e9e474d53c7d20f096dbd8a080232f88dddb90566f90064c \\\n    --hash=sha256:3891cd086c5071bda6b4ee9d266bb2ac39c998c045c2ebcd1e818b8316fb5d41 \\\n    --hash=sha256:49bcfad8cfe2c27dee116c45d4f866d7974bcf14a5a9fbef893635deae322f2f \\\n    --hash=sha256:5045a3a5f21ec3eea6962fa5f2fa2d4283f854caec25ada493fcf4aab2925467 \\\n    --hash=sha256:5083a5b1fec2351bf5ea9900a741d54086db75baec4b1d21e39451e00977f1b1 \\\n    --hash=sha256:54454923a50104c66a9ab6bd8b73a11c2fc218c964b1006d5d1fe5b442c3dcb6 \\\n    --hash=sha256:54815e0a56dde95cc6ec952577f67e0dc151eadd928e8d9f6a7f821d69a4a734 \\\n    --hash=sha256:5568c5a1ff1b2ec33127b629403adb530fab81378d9018ca4ed6508293f76e2b \\\n    --hash=sha256:5c22caeaae8b3c36d93459f1a5294e6f43306cff856ed243189a229331a404b4 \\\n    --hash=sha256:659b76c86757cb2ee4ca2db245e0740cfc3081fef46f0f1064d11adb4a8cee31 \\\n    --hash=sha256:669575b290ec27304569e188a960d12b907d5173f9cd65e86621d34c4e5b6c30 \\\n    --hash=sha256:6bdce3890fa949219de129e85e4f6d544598af3c073afe5c44e14aed15bdcbb2 \\\n    --hash=sha256:6eb75d41e3bbfc2f7642d0abba9383cc9ae6c5a4ca8d6b00628c225e1eaa63b3 \\\n    --hash=sha256:7e9e9afa150e40cd2a8f0701c43cb82a8d724f512896455c0918b987f94b84a4 \\\n    --hash=sha256:8c44b6924b530d0702e88ff383b65c4b34a0eaf666e8b399a73245574d546947 \\\n    --hash=sha256:9147f5e096a9270684e3befdee350f3cacafd48e0c54ab195f45790a9c146d67 \\\n    --hash=sha256:97a5814a93c793aaf0179cfc7f916024f4b63218929aee977b645633d074a49f \\\n    --hash=sha256:abbf1d7b9d52c00d2af4afa8dac1fb3e2356f662a4566bd98dfaaa3634f4eb34 \\\n    --hash=sha256:b0c0b264b89ab572888244f2e0bad5b7eaf5b696068fc0b93e96f7c3c198953f \\\n    --hash=sha256:b578bcad8a4083b40d34f689b19ca9f7c63e511758d806510ea03c29ac568f7b \\\n    --hash=sha256:e6572227228ec521618cea9ac3a368c45b7f96f1f8622dc9f1afe891c044051f \\\n    --hash=sha256:ff96666b94a55e802ea6796cabe788541719e6f4905fc59c380fed3517b6a64d \\\n    --hash=sha256:ffa2a16499508fe6798323e455f312c7c55f2a88901c9a7c0fb1efa86cf7e327\ntqdm==4.66.4 \\\n    --hash=sha256:b75ca56b413b030bc3f00af51fd2c1a1a5eac6a0c1cca83cbb37a5c52abce644 \\\n    --hash=sha256:e4d936c9de8727928f3be6079590e97d9abfe8d39a590be678eb5919ffc186bb\ntraitlets==5.14.3 \\\n    --hash=sha256:9ed0579d3502c94b4b3732ac120375cda96f923114522847de4b3bb98b96b6b7 \\\n    --hash=sha256:b74e89e397b1ed28cc831db7aea759ba6640cb3de13090ca145426688ff1ac4f\ntransformers==4.44.0 \\\n    --hash=sha256:75699495e30b7635ca444d8d372e138c687ab51a875b387e33f1fb759c37f196 \\\n    --hash=sha256:ea0ff72def71e9f4812d9414d4803b22681b1617aa6f511bd51cfff2b44a6fca\ntriton==3.2.0 \\\n    --hash=sha256:30ceed0eff2c4a73b14eb63e052992f44bbdf175f3fad21e1ac8097a772de7ee \\\n    --hash=sha256:8009a1fb093ee8546495e96731336a33fb8856a38e45bb4ab6affd6dbc3ba220 \\\n    --hash=sha256:8d9b215efc1c26fa7eefb9a157915c92d52e000d2bf83e5f69704047e63f125c \\\n    --hash=sha256:b3e54983cd51875855da7c68ec05c05cf8bb08df361b1d5b69e05e40b0c9bd62 \\\n    --hash=sha256:e5dfa23ba84541d7c0a531dfce76d8bcd19159d50a4a8b14ad01e91734a5c1b0\ntyping-extensions==4.15.0 \\\n    --hash=sha256:0cea48d173cc12fa28ecabc3b837ea3cf6f38c6d1136f85cbaaf598984861466 \\\n    --hash=sha256:f0fa19c6845758ab08074a0cfa8b7aecb71c999ca73d62883bc25cc018c4e548\ntyping-inspection==0.4.1 \\\n    --hash=sha256:389055682238f53b04f7badcb49b989835495a96700ced5dab2d8feae4b26f51 \\\n    --hash=sha256:6ae134cc0203c33377d43188d4064e9b357dba58cff3185f22924610e70a9d28\ntzdata==2025.2 \\\n    --hash=sha256:1a403fada01ff9221ca8044d701868fa132215d84beb92242d9acd2147f667a8 \\\n    --hash=sha256:b60a638fcc0daffadf82fe0f57e53d06bdec2f36c4df66280ae79bce6bd6f2b9\nurllib3==2.5.0 \\\n    --hash=sha256:3fc47733c7e419d4bc3f6b3dc2b4f890bb743906a30d56ba4a5bfa4bbff92760 \\\n    --hash=sha256:e6b01673c0fa6a13e374b50871808eb3bf7046c4b125b216f6bf1cc604cff0dc\nwandb==0.19.8 \\\n    --hash=sha256:068eb0154f80be973ab291346d831e9cc80a9de1b8752bdeb48a997c3506fec4 \\\n    --hash=sha256:1781b36434d494d6b34e2149201bae8cab960cb31571f11b981c4a62462d5af8 \\\n    --hash=sha256:3a4844bb38758657b94b090e72ee355fe5b926e3a048232f0ca4248f801d8d80 \\\n    --hash=sha256:6556147ba33b7ff4a0111bb6bf5ea485e4974c22f520f1e2a5eaad670a058c80 \\\n    --hash=sha256:75dea834d579f38e0e1f857e644020e22c851f9b920e9c6c6345bacb98c3f3fc \\\n    --hash=sha256:82a956150e53df0b4c193933b3e62c3e8255dc8b43bb187270939ef35b03fda3 \\\n    --hash=sha256:96cb534b19c2d301ac4fb0e7cfbc32198a704e29e87337133d6b71fdad33cf2f \\\n    --hash=sha256:9d71f153cb9330e307b1b054be01971a1bd164fb9bd4190d7f57989c2d6b86e8 \\\n    --hash=sha256:c25f0e40025b838b7a424b51837a2a5fd071686c59e1c46d73f04e760d305f79 \\\n    --hash=sha256:f68517c2059d12912a90ae32ce95a2711e39f6c157c759eb191527739a12db8b \\\n    --hash=sha256:f7da8e6fc6693014c72fb7db3ecd5e1116066198d2aca96f6eb7220cea03081c\nwcwidth==0.2.13 \\\n    --hash=sha256:3da69048e4540d84af32131829ff948f1e022c1c6bdb8d6102117aac784f6859 \\\n    --hash=sha256:72ea0c06399eb286d978fdedb6923a9eb47e1c486ce63e9b4e64fc18303972b5\nwheel==0.45.1 \\\n    --hash=sha256:661e1abd9198507b1409a20c02106d9670b2576e916d58f520316666abca6729 \\\n    --hash=sha256:708e7481cc80179af0e556bbf0cc00b8444c7321e2700b8d8580231d13017248\n', 'patch_repo.py': '"""Apply narrowly scoped, idempotent compatibility patches to the pinned source."""\nimport argparse\nimport ast\nfrom pathlib import Path\nimport re\n\n\ndef replace_once(path, old, new):\n    text = path.read_text()\n    if new in text and old not in text:\n        return\n    if text.count(old) != 1 or new in text:\n        raise RuntimeError(f\'Unexpected source in {path}; refusing to guess a patch.\')\n    updated = text.replace(old, new, 1)\n    ast.parse(updated)\n    path.write_text(updated)\n\n\ndef patch(repo):\n    # These two checkpoints come from the specific project download sources.\n    replace_once(repo / \'inference.py\', \'torch.load(cfg.resume_path)\',\n                 \'torch.load(cfg.resume_path, weights_only=False)\')\n    replace_once(repo / \'preproc/run_tapir.py\', \'torch.load(ckpt_path)\',\n                 \'torch.load(ckpt_path, weights_only=False)\')\n    replace_once(repo / \'core/utils/dino_feat.py\',\n                 \'torch.hub.load("facebookresearch/dinov2", model_type)\',\n                 f\'torch.hub.load({str(repo / "preproc/dinov2")!r}, model_type, source="local")\')\n    replace_once(repo / \'core/utils/run_depth.py\',\n                 \'pipeline(task="depth-estimation", model=models[model_name], device=DEVICE)\',\n                 \'pipeline(task="depth-estimation", model=models[model_name], device=DEVICE, \'\n                 \'revision="7581137eff8d4e94f6e796d3baea0e9fa79b22d2" \'\n                 \'if model_name == "depth-anything-v2" else None)\')\n\n    # Use torch\'s efficient attention fallback instead of a separate xformers\n    # binary tied to one torch build. Keep the same attention scale and dropout.\n    path = repo / \'preproc/dinov2/dinov2/layers/attention.py\'\n    source = path.read_text()\n    if \'# SegAnyMo team SDPA fallback\' not in source:\n        before = re.compile(r\'        qkv = self\\.qkv\\(x\\)\\.reshape\\(B, N, 3, self\\.num_heads, C // self\\.num_heads\\)\\.permute\\(2, 0, 3, 1, 4\\).*?        x = \\(attn @ v\\)\\.transpose\\(1, 2\\)\\.reshape\\(B, N, C\\)\', re.S)\n        after = \'\'\'        # SegAnyMo team SDPA fallback\n        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads)\n        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)\n        x = nn.functional.scaled_dot_product_attention(\n            q, k, v, dropout_p=self.attn_drop.p if self.training else 0.0\n        ).transpose(1, 2).reshape(B, N, C)\'\'\'\n        source, count = before.subn(after, source)\n        if count != 1:\n            raise RuntimeError(\'Unexpected DINO attention code; refusing to patch.\')\n        ast.parse(source)\n        path.write_text(source)\n\n    source = (repo / \'configs/example_train.yaml\').read_text()\n    source, count = re.subn(r\'^resume_path:.*$\',\n                           f\'resume_path: "{repo / "checkpoints/moseg.pth"}"\',\n                           source, flags=re.M)\n    if count != 1:\n        raise RuntimeError(\'Missing or ambiguous resume_path in example_train.yaml\')\n    destination = repo / \'configs/colab.yaml\'\n    if destination.exists() and destination.read_text() != source:\n        raise RuntimeError(f\'{destination} was edited; move it aside before rerunning setup.\')\n    destination.write_text(source)\n    print(\'Repository compatibility patches and colab.yaml ready.\')\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'repo\', type=Path)\n    patch(parser.parse_args().repo.resolve())\n', 'patch_launcher.py': '#!/usr/bin/env python3\n"""Patch the pinned SegAnyMo launcher; refuse an unexpected or edited source.\n\nRun with the repository directory as the only argument. This does not import\nSegAnyMo or need third-party packages. The patch keeps each subprocess on the\nactive Python interpreter, quotes filesystem arguments, and reports failures\nthat the upstream process-pool launcher otherwise ignores.\n"""\n\nimport argparse\nimport ast\nimport hashlib\nfrom pathlib import Path\n\n\nUPSTREAM_SHA256 = "92eeca7873760a969c19259af6269c6956afd5a38ed0d4fa7256a36cb84e70d2"\nPATCHED_SHA256 = "f3cb51c3f5624a90cde869ceaaf112e4bb81466e034fc8122c3bc7b9c5c0cb4c"\nMARKER = "# SegAnyMo team launcher patch v1\\n"\n\n\ndef replace_exact(source, before, after, count):\n    actual = source.count(before)\n    if actual != count:\n        raise RuntimeError(\n            f"Unexpected launcher source: expected {count} matches for {before!r}, got {actual}"\n        )\n    return source.replace(before, after)\n\n\ndef transform(source):\n    """Transform only the verified upstream source, then validate Python syntax."""\n    source = replace_exact(\n        source,\n        "import subprocess\\nfrom concurrent.futures import ProcessPoolExecutor\\n",\n        "import subprocess\\nimport shlex\\nimport sys\\n"\n        "from concurrent.futures import ProcessPoolExecutor, as_completed\\n",\n        1,\n    )\n    source = replace_exact(\n        source,\n        "    with ProcessPoolExecutor(max_workers=len(gpus)) as exe:\\n",\n        "    with ProcessPoolExecutor(max_workers=len(gpus)) as exe:\\n"\n        "        futures = []\\n",\n        1,\n    )\n    source = replace_exact(\n        source,\n        "exe.submit(subprocess.call, cmd, shell=True)",\n        "futures.append(exe.submit(subprocess.run, cmd, shell=True, check=True))",\n        7,\n    )\n    source = replace_exact(\n        source,\n        "exe.submit(subprocess.run, cmd, shell=True)",\n        "futures.append(exe.submit(subprocess.run, cmd, shell=True, check=True))",\n        1,\n    )\n    source = replace_exact(\n        source,\n        " python {current_work_dir}/",\n        " {shlex.quote(sys.executable)} {shlex.quote(current_work_dir)}/",\n        8,\n    )\n\n    # These fields appear inside shell command f-strings. Quote the field,\n    # including the directory portion of script/checkpoint paths, as one token.\n    # Numeric fields (device id and step) are already parsed as integers.\n    for field, count in (\n        ("img_dir", 7),\n        ("sequence_dir", 1),\n        ("depth_dir", 2),\n        ("track_dir", 3),\n        ("motin_seg_dir", 1),\n        ("args.config_file", 1),\n        ("gt_dir", 2),\n        ("dynamic_dir", 1),\n        ("args.sam2dir", 1),\n        ("depth_model", 1),\n        ("dataset", 1),\n    ):\n        # img_dir also occurs in the informational "Skipping" f-string.\n        # Quoting it there would be harmless but would make the message noisy.\n        pattern = "{" + field + "}"\n        replacement = "{shlex.quote(str(" + field + "))}"\n        if field == "img_dir":\n            source = replace_exact(source, "--image_dir {img_dir}", "--image_dir " + replacement, 2)\n            source = replace_exact(source, "--img_dir {img_dir}", "--img_dir " + replacement, 1)\n            source = replace_exact(source, "--imgs_dir {img_dir}", "--imgs_dir " + replacement, 2)\n            source = replace_exact(source, "--data_dir {img_dir}", "--data_dir " + replacement, 1)\n            source = replace_exact(source, "--video_dir {img_dir}", "--video_dir " + replacement, 1)\n        elif field == "depth_model":\n            source = replace_exact(source, "--model " + pattern, "--model " + replacement, count)\n        else:\n            source = replace_exact(source, pattern, replacement, count)\n    source = replace_exact(\n        source,\n        "--ckpt_dir {current_work_dir}/preproc/checkpoints",\n        "--ckpt_dir {shlex.quote(current_work_dir)}/preproc/checkpoints",\n        1,\n    )\n    source = replace_exact(\n        source,\n        \'\\nif __name__ == "__main__":\',\n        \'\\n        # Propagate errors raised by each child process.\\n\'\n        \'        for future in as_completed(futures):\\n\'\n        \'            future.result()\\n\'\n        \'\\nif __name__ == "__main__":\',\n        1,\n    )\n    source = MARKER + source\n    ast.parse(source)\n    return source\n\n\ndef patch_file(path):\n    original = path.read_bytes()\n    digest = hashlib.sha256(original).hexdigest()\n    if digest == PATCHED_SHA256:\n        print(f"Launcher patch already applied: {path}")\n        return False\n    if digest != UPSTREAM_SHA256:\n        raise RuntimeError(\n            f"Refusing to patch an unexpected launcher: {path} (SHA256 {digest}). "\n            "Use the pinned repository checkout or review the local changes."\n        )\n    result = transform(original.decode("utf-8")).encode("utf-8")\n    result_digest = hashlib.sha256(result).hexdigest()\n    if result_digest != PATCHED_SHA256:\n        raise RuntimeError("Launcher patch produced an unexpected result; no file was written")\n    # Write beside the destination, then rename so an interrupted write cannot\n    # leave a truncated launcher. Existing permissions are preserved.\n    temporary = path.with_name(path.name + ".colab-patch.tmp")\n    temporary.write_bytes(result)\n    temporary.chmod(path.stat().st_mode)\n    temporary.replace(path)\n    print(f"Patched subprocess interpreter and failure reporting: {path}")\n    return True\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("repo", type=Path, help="SegAnyMo checkout directory")\n    args = parser.parse_args()\n    patch_file(args.repo / "core" / "utils" / "run_inference.py")\n\n\nif __name__ == "__main__":\n    main()\n', 'checkpoints.py': '#!/usr/bin/env python3\n"""Download validated SegAnyMo weights; optionally reuse a checkpoint-only cache.\n\nThe SAM2 SHA256 sidecar records local integrity, not upstream authenticity.\n"""\nimport argparse\nfrom contextlib import contextmanager\nfrom dataclasses import dataclass\nimport hashlib\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\nimport tempfile\nimport time\nfrom urllib.request import Request, urlopen\nimport zipfile\n\n\n@dataclass(frozen=True)\nclass Checkpoint:\n    relative: str\n    url: str\n    size: int = None\n    digest: str = None\n    algorithm: str = "sha256"\n\n\nCHECKPOINTS = (\n    Checkpoint("sam2/checkpoints/sam2_hiera_large.pt",\n               "https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt"),\n    Checkpoint("preproc/checkpoints/bootstapir_checkpoint_v2.pt",\n               "https://storage.googleapis.com/dm-tapnet/bootstap/bootstapir_checkpoint_v2.pt"\n               "?generation=1716468999470836", 218886140,\n               "c17a81a817601b95d61a8559e28d1160", "md5"),\n    Checkpoint("checkpoints/moseg.pth",\n               "https://huggingface.co/Changearthmore/moseg/resolve/"\n               "5ce31c82a80df8ec00ed1e7b1baee72d2b3eecc5/moseg.pth", 13858390,\n               "705445fbf6b2e42cd4e946390c5c6891046e70add8e8377e38f5368f503591d9"),\n)\n\n\ndef warn(message):\n    print(f"WARNING: {message}", file=sys.stderr, flush=True)\n\n\ndef sidecar(path):\n    return path.with_name(path.name + ".sha256")\n\n\n@contextmanager\ndef temporary(destination):\n    fd, name = tempfile.mkstemp(prefix=f".{destination.name}.", suffix=".partial",\n                                dir=destination.parent)\n    os.close(fd)\n    path = Path(name)\n    try:\n        yield path\n    finally:\n        path.unlink(missing_ok=True)\n\n\ndef validate(path, checkpoint, expected_sha=None):\n    size = path.stat().st_size\n    if size == 0 or (checkpoint.size is not None and size != checkpoint.size):\n        raise ValueError(f"unexpected size {size} bytes")\n    if checkpoint.digest is None and expected_sha is None and sidecar(path).exists():\n        expected_sha = sidecar(path).read_text().strip()\n        if len(expected_sha) != 64 or any(c not in "0123456789abcdef" for c in expected_sha):\n            raise ValueError("invalid local SHA256 sidecar")\n    sha = hashlib.sha256()\n    other = sha if checkpoint.algorithm == "sha256" else hashlib.new(\n        checkpoint.algorithm, usedforsecurity=False)\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            sha.update(chunk)\n            if other is not sha:\n                other.update(chunk)\n    digest = sha.hexdigest()\n    if checkpoint.digest and other.hexdigest() != checkpoint.digest:\n        raise ValueError(f"{checkpoint.algorithm} mismatch")\n    if expected_sha is not None and digest != expected_sha:\n        raise ValueError("local SHA256 mismatch")\n    if checkpoint.digest is None and expected_sha is None:\n        with zipfile.ZipFile(path) as archive:\n            if not archive.namelist() or archive.testzip() is not None:\n                raise ValueError("SAM2 ZIP integrity check failed")\n    return digest\n\n\ndef record(path, checkpoint, digest):\n    if checkpoint.digest is None:\n        with temporary(sidecar(path)) as temp:\n            temp.write_text(digest + "\\n")\n            os.replace(temp, sidecar(path))\n\n\ndef copy_verified(source, destination, checkpoint, digest):\n    with temporary(destination) as temp:\n        shutil.copyfile(source, temp)\n        validate(temp, checkpoint, expected_sha=digest)\n        os.replace(temp, destination)\n    record(destination, checkpoint, digest)\n\n\ndef download(destination, checkpoint):\n    for attempt in range(1, 4):\n        try:\n            with temporary(destination) as temp:\n                request = Request(checkpoint.url, headers={"User-Agent": "SegAnyMo-setup/1"})\n                with urlopen(request, timeout=60) as response, temp.open("wb") as output:\n                    content_length = response.headers.get("Content-Length")\n                    shutil.copyfileobj(response, output, length=8 * 1024 * 1024)\n                if content_length is not None and temp.stat().st_size != int(content_length):\n                    raise ValueError("download size differs from HTTP Content-Length")\n                digest = validate(temp, checkpoint)\n                os.replace(temp, destination)\n            record(destination, checkpoint, digest)\n            return digest\n        except Exception as error:\n            if attempt == 3:\n                raise RuntimeError(f"Failed downloading {destination.name} after 3 attempts") from error\n            warn(f"{destination.name}: attempt {attempt}/3 failed: {error}; retrying")\n            time.sleep(attempt)\n\n\ndef existing(path, checkpoint):\n    try:\n        if path.exists():\n            return validate(path, checkpoint)\n    except Exception as error:\n        warn(f"Ignoring invalid or unreadable checkpoint {path}: {error}")\n    return None\n\n\ndef ensure(repo, checkpoint, cache=None):\n    destination = repo / checkpoint.relative\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    digest = existing(destination, checkpoint)\n    cached = cache / destination.name if cache else None\n    if digest is None and cached:\n        cached_digest = existing(cached, checkpoint)\n        if cached_digest:\n            try:\n                copy_verified(cached, destination, checkpoint, cached_digest)\n                digest = cached_digest\n                print(f"Restored {destination.name} from checkpoint cache", flush=True)\n            except Exception as error:\n                warn(f"Cache restore failed for {cached}: {error}; downloading")\n    if digest is None:\n        print(f"Downloading {destination.name}", flush=True)\n        digest = download(destination, checkpoint)\n    record(destination, checkpoint, digest)\n    if cached:\n        try:\n            if existing(cached, checkpoint) != digest:\n                copy_verified(destination, cached, checkpoint, digest)\n            else:\n                record(cached, checkpoint, digest)\n        except Exception as error:\n            warn(f"Could not save checkpoint cache {cached}: {error}")\n    print(f"OK {destination.name} ({destination.stat().st_size:,} bytes)", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("repo", type=Path)\n    parser.add_argument("--cache", type=Path)\n    args = parser.parse_args()\n    cache = args.cache\n    if cache:\n        try:\n            cache = cache.expanduser().resolve()\n            if (cache.is_relative_to("/content/drive")\n                    and not Path("/content/drive/MyDrive").is_dir()):\n                raise OSError("Google Drive is not mounted at /content/drive/MyDrive")\n            cache.mkdir(parents=True, exist_ok=True)\n        except OSError as error:\n            warn(f"Checkpoint cache disabled: {error}")\n            cache = None\n    for checkpoint in CHECKPOINTS:\n        ensure(args.repo, checkpoint, cache)\n\n\nif __name__ == "__main__":\n    main()\n', 'smoke_test.py': '"""Installation checks run inside the managed Colab environment, before inference."""\nimport importlib\nimport os\nfrom pathlib import Path\nimport sys\n\nrepo = Path(sys.argv[1]).resolve()\nos.chdir(\'/tmp\')\nimport torch\nimport torchvision\nimport torchaudio\nimport transformers\nimport numpy\nfrom torchvision.ops import nms\nfrom sam2.build_sam import build_sam2_video_predictor\nfrom tapnet_torch import tapir_model\nfrom dinov2.layers.attention import MemEffAttention\n\nexpected = {\'torch\': \'2.6.0\', \'torchvision\': \'0.21.0\', \'torchaudio\': \'2.6.0\',\n            \'transformers\': \'4.44.0\', \'numpy\': \'2.0.1\'}\nassert sys.version_info[:3] == (3, 12, 11), sys.version\nfor name, version in expected.items():\n    actual = importlib.import_module(name).__version__\n    assert actual.split(\'+\')[0] == version, (name, actual)\n    print(f\'{name}: {actual}\')\nassert torch.cuda.is_available(), \'CUDA unavailable: choose a Colab GPU runtime.\'\nassert torch.version.cuda == \'12.4\', torch.version.cuda\nx = torch.randn(32, 32, device=\'cuda\')\nassert torch.isfinite(x @ x.T).all()\nboxes = torch.tensor([[0, 0, 1, 1], [2, 2, 3, 3]], dtype=torch.float32, device=\'cuda\')\nassert nms(boxes, torch.tensor([.9, .8], device=\'cuda\'), .5).numel() == 2\nattention = MemEffAttention(dim=64, num_heads=4).eval().cuda()\nwith torch.inference_mode():\n    assert attention(torch.randn(1, 32, 64, device=\'cuda\')).shape == (1, 32, 64)\ntorch.cuda.synchronize()\nprint(\'GPU:\', torch.cuda.get_device_name(0), \'| Python:\', sys.executable)\n\n# Import actual entrypoints: catches missing dependencies that basic imports miss.\nfor module in [\'inference\', \'core.utils.run_depth\', \'core.utils.dino_feat\',\n               \'core.utils.run_inference\', \'preproc.run_tapir\']:\n    importlib.import_module(module)\n    print(\'Imported:\', module)\nfor relative in [\'sam2/checkpoints/sam2_hiera_large.pt\',\n                 \'preproc/checkpoints/bootstapir_checkpoint_v2.pt\',\n                 \'checkpoints/moseg.pth\', \'configs/colab.yaml\']:\n    file = repo / relative\n    assert file.is_file() and file.stat().st_size > 0, str(file)\n    print(\'Ready:\', relative)\nprint(\'Import, CUDA operation, torchvision operator, and DINO attention checks passed.\')\nprint(\'Installation checks complete. Continue with your existing later notebook sections.\')\n', 'patch_depth.py': '#!/usr/bin/env python3\n"""Patch the two known SegAnyMo depth-output layouts without importing models."""\nimport argparse\nimport ast\nimport os\nfrom pathlib import Path\nimport tempfile\nimport textwrap\n\n\nANCHOR = \'disp = pipe(image)["predicted_depth"]\'\nORIGINAL = \'\'\'\\\ndisp = torch.nn.functional.interpolate(\n    disp.unsqueeze(1), size=image.size[::-1], mode="bicubic", align_corners=False\n)\ndisp = disp.squeeze().cpu().numpy()\n\'\'\'\nMANUAL = \'\'\'\\\ndisp = disp.unsqueeze(0).unsqueeze(0)\ndisp = torch.nn.functional.interpolate(\n    disp, size=image.size[::-1], mode="bicubic", align_corners=False\n)\ndisp = disp.squeeze().cpu().numpy()\n\'\'\'\nREPLACEMENT = \'\'\'\\\n# Normalize one predicted depth map to NCHW; retain both spatial axes.\nif not isinstance(disp, torch.Tensor):\n    raise TypeError("predicted_depth must be a torch.Tensor")\nif disp.ndim == 2:\n    disp = disp[None, None]\nelif disp.ndim == 3 and disp.shape[0] == 1:\n    disp = disp[None]\nelif disp.ndim != 4 or tuple(disp.shape[:2]) != (1, 1):\n    raise ValueError(f"Expected one depth map with shape HxW, 1xHxW, or 1x1xHxW; got {tuple(disp.shape)}")\nif disp.shape[-2] == 0 or disp.shape[-1] == 0:\n    raise ValueError("predicted_depth has an empty spatial dimension")\ndisp = disp.detach().to(dtype=torch.float32)\nif tuple(disp.shape[-2:]) != image.size[::-1]:\n    disp = torch.nn.functional.interpolate(\n        disp, size=image.size[::-1], mode="bicubic", align_corners=False\n    )\ndisp = disp[0, 0].cpu().numpy()\n\'\'\'\n\n\ndef signature(nodes):\n    return [ast.dump(node, include_attributes=False) for node in nodes]\n\n\ndef patch_file(path):\n    """Patch a run_depth.py path atomically; return False if already patched.\n\n    An unfamiliar function body raises RuntimeError before any write occurs.\n    Comments and formatting are ignored for matching. Source outside the\n    replaced statements is retained verbatim, including model revisions.\n    """\n    path = Path(path)\n    source = path.read_bytes().decode("utf-8")\n    try:\n        tree = ast.parse(source, filename=str(path))\n    except SyntaxError as error:\n        raise RuntimeError(f"Cannot patch invalid Python in {path}: {error}") from error\n    functions = [node for node in tree.body if isinstance(node, ast.FunctionDef)\n                 and node.name == "get_depth_anything_disp"]\n    if len(functions) != 1:\n        raise RuntimeError(f"Expected exactly one get_depth_anything_disp in {path}")\n    body = functions[0].body\n    anchor = signature(ast.parse(ANCHOR).body)[0]\n    starts = [index + 1 for index, node in enumerate(body)\n              if ast.dump(node, include_attributes=False) == anchor]\n    if len(starts) != 1:\n        raise RuntimeError(f"Unrecognized predicted_depth assignment in {path}")\n    start = starts[0]\n    normalized = signature(body[start:])\n    replacement = signature(ast.parse(REPLACEMENT).body)\n    if normalized[:len(replacement)] == replacement:\n        return False\n    length = None\n    for candidate in (ORIGINAL, MANUAL):\n        expected = signature(ast.parse(candidate).body)\n        if normalized[:len(expected)] == expected:\n            length = len(expected)\n            break\n    if length is None:\n        raise RuntimeError(f"Unsupported depth interpolation code in {path}; no changes made")\n\n    first, last = body[start], body[start + length - 1]\n    lines = source.splitlines(keepends=True)\n    indent = lines[first.lineno - 1][:first.col_offset]\n    trailing = lines[last.end_lineno - 1][last.end_col_offset:].strip()\n    if not indent.isspace() or (trailing and not trailing.startswith("#")):\n        raise RuntimeError(f"Unsupported inline depth statements in {path}; no changes made")\n    newline = "\\r\\n" if "\\r\\n" in lines[first.lineno - 1] else "\\n"\n    block = textwrap.indent(REPLACEMENT, indent).replace("\\n", newline)\n    patched = "".join(lines[:first.lineno - 1]) + block + "".join(lines[last.end_lineno:])\n    ast.parse(patched, filename=str(path))\n    fd, temporary = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)\n    try:\n        with os.fdopen(fd, "wb") as output:\n            output.write(patched.encode("utf-8"))\n        os.chmod(temporary, path.stat().st_mode & 0o777)\n        os.replace(temporary, path)\n    finally:\n        Path(temporary).unlink(missing_ok=True)\n    return True\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("repo", type=Path)\n    args = parser.parse_args()\n    path = args.repo / "core/utils/run_depth.py"\n    try:\n        changed = patch_file(path)\n    except (OSError, RuntimeError, SyntaxError, UnicodeError) as error:\n        parser.exit(1, f"Depth compatibility patch failed: {error}\\n")\n    print(f"Depth compatibility patch {\'applied\' if changed else \'already applied\'}: {path}")\n\n\nif __name__ == "__main__":\n    main()\n', 'post_setup.py': '"""Prepare the user\'s existing later notebook sections, without processing a video."""\nimport argparse\nimport ast\nimport os\nfrom pathlib import Path\nimport re\n\nfrom patch_depth import patch_file as patch_depth_file\n\n\ndef disable_tf32(env_file):\n    source = env_file.read_text()\n    line = \'export NVIDIA_TF32_OVERRIDE=0\'\n    pattern = r\'^export NVIDIA_TF32_OVERRIDE=.*$\'\n    if len(re.findall(pattern, source, flags=re.M)) > 1:\n        raise RuntimeError(f\'Multiple TF32 settings in {env_file}; review them first.\')\n    if re.search(pattern, source, flags=re.M):\n        source = re.sub(pattern, line, source, flags=re.M)\n    else:\n        source = source.rstrip() + \'\\n\' + line + \'\\n\'\n    env_file.write_text(source)\n    os.environ[\'NVIDIA_TF32_OVERRIDE\'] = \'0\'\n\n\ndef write_config(repo):\n    config = repo / \'configs/fish_colab.yaml\'\n    # Preserve an existing fish configuration\'s settings; only update its checkpoint.\n    source = config.read_text() if config.exists() else (repo / \'configs/example_train.yaml\').read_text()\n    source, count = re.subn(r\'^\\s*resume_path:.*$\',\n                           f\'resume_path: "{repo / "checkpoints/moseg.pth"}"\',\n                           source, flags=re.M)\n    if count != 1:\n        raise RuntimeError(\'Expected exactly one resume_path in the fish configuration.\')\n    config.write_text(source)\n    return config\n\n\ndef patch_sam2_dtype(path):\n    source = path.read_text()\n    new = (\'torch.autocast("cuda", dtype=(\'\n           \'torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16))\')\n    candidates = [\'torch.autocast("cuda", dtype=torch.bfloat16)\',\n                  \'torch.autocast("cuda", dtype=torch.float16)\']\n    count = sum(source.count(old) for old in candidates)\n    if source.count(new) == 1 and count == 0:\n        return\n    if count != 1 or new in source:\n        raise RuntimeError(f\'Unexpected SAM2 autocast code in {path}; no patch applied.\')\n    for old in candidates:\n        source = source.replace(old, new)\n    ast.parse(source)\n    path.write_text(source)\n\n\ndef prepare(repo, env_file):\n    repo = repo.resolve()\n    if not env_file.is_file():\n        raise RuntimeError(\'Missing seganymo-env.sh; complete the installation cell first.\')\n    disable_tf32(env_file)\n    # This program runs as a new process under the sourced environment.\n    # Set the override before importing torch/CUDA libraries.\n    import sys\n    import torch\n    import mediapy\n    import einshape\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'No GPU is available. Select a Colab GPU runtime.\')\n    required = [repo / \'checkpoints/moseg.pth\',\n                repo / \'preproc/checkpoints/bootstapir_checkpoint_v2.pt\',\n                repo / \'sam2/checkpoints/sam2_hiera_large.pt\']\n    missing = []\n    for path in required:\n        exists = path.is_file() and path.stat().st_size > 0\n        print(path.name, \'ready:\', exists)\n        if not exists:\n            missing.append(str(path))\n    if missing:\n        raise RuntimeError(\'Missing or empty checkpoints; rerun installation: \' + \', \'.join(missing))\n\n    patch_depth_file(repo / \'core/utils/run_depth.py\')\n    patch_sam2_dtype(repo / \'sam2/run_sam2.py\')\n    config = write_config(repo)\n    print(\'Config:\', config)\n    print(\'Python:\', sys.executable)\n    print(\'GPU:\', torch.cuda.get_device_name(0))\n    print(\'TF32 disabled; this does not disable mixed-precision autocast.\')\n    print(\'SAM2 autocast:\', \'bfloat16\' if torch.cuda.is_bf16_supported() else \'float16\')\n    print(\'Preparation complete. Continue with your existing later sections.\')\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'repo\', type=Path)\n    parser.add_argument(\'--env-file\', type=Path, default=Path(\'/content/seganymo-env.sh\'))\n    args = parser.parse_args()\n    prepare(args.repo, args.env_file)\n'}
for name, contents in payload.items():
    (setup_dir / name).write_text(contents)
print("Shared setup files written.")


In [ ]:
%%bash
set -euo pipefail
SETUP_DIR="/content/seganymo-setup"

for file in setup.sh requirements-linux-py312.lock patch_repo.py patch_launcher.py checkpoints.py smoke_test.py patch_depth.py post_setup.py; do
    if [[ ! -f "$SETUP_DIR/$file" ]]; then
        echo "Missing setup files. Rerun updated cell 3, then run this cell again." >&2
        exit 1
    fi
done

bash "$SETUP_DIR/setup.sh" 2>&1 | tee /content/seganymo-setup.log


In [ ]:
# Do not leave an earlier successful/partial storage selection active on failure.
globals().pop("DRIVE_DIR", None)
from pathlib import Path
from google.colab import drive
import os
from uuid import uuid4

mount_point = Path("/content/drive")
if not os.path.ismount(mount_point):
    if mount_point.is_symlink() or (mount_point.exists() and not mount_point.is_dir()):
        raise RuntimeError("/content/drive must be a directory, not a symlink or file.")
    if mount_point.is_dir() and any(mount_point.iterdir()):
        mountinfo = Path("/proc/self/mountinfo")
        if mountinfo.is_file():
            mounted_paths = [line.split()[4] for line in mountinfo.read_text().splitlines()]
            if any(p == str(mount_point) or p.startswith(str(mount_point) + "/") for p in mounted_paths):
                raise RuntimeError("A filesystem is mounted at or below /content/drive; leaving it in place.")
        backup = mount_point.with_name("drive-local-backup-" + uuid4().hex)
        mount_point.rename(backup)
        print("Preserved existing local files at:", backup)
        print("This backup is in the Colab runtime; it has not been saved to Google Drive.")
    try:
        drive.mount(str(mount_point))
    except Exception as error:
        if "credential propagation was unsuccessful" not in str(error):
            raise
        raise RuntimeError(
            "Google Drive authorization failed. Refresh the Colab tab and retry "
            "this cell, completing Google's sign-in/Drive permission prompt. "
            "If no prompt appears, check the browser's popup/site settings. "
            "Any local backup printed above is preserved. "
            "You do not need to reinstall SegAnyMo for this authentication error."
        ) from error
if not os.path.ismount(mount_point):
    raise RuntimeError("Google Drive is not mounted; storage folders were not created.")

my_drive = mount_point / "MyDrive"
if not my_drive.is_dir():
    raise RuntimeError("Drive is mounted but MyDrive is unavailable; check Drive access before continuing.")
storage_root = my_drive / "seganymo"
for sub in ["videos", "preproc_cache", "results"]:
    (storage_root / sub).mkdir(parents=True, exist_ok=True)
DRIVE_DIR = storage_root

os.environ["NVIDIA_TF32_OVERRIDE"] = "0"
print("Drive storage ready:", DRIVE_DIR)
print("TF32 override set for later subprocesses.")


In [ ]:
%%bash
set -euo pipefail
if [[ ! -f /content/seganymo-env.sh ]]; then
    echo "Installation is missing. Run setup cell 4 first." >&2
    exit 1
fi
source /content/seganymo-env.sh
export NVIDIA_TF32_OVERRIDE=0
cd "$SEGANYMO_REPO"
for file in patch_depth.py post_setup.py; do
    if [[ ! -f "/content/seganymo-setup/$file" ]]; then
        echo "Updated preparation files are missing. Rerun cell 3 from this notebook first." >&2
        exit 1
    fi
done
python /content/seganymo-setup/post_setup.py "$SEGANYMO_REPO"


In [ ]:
# Run every SegAnyMo command inside its installed environment.
import subprocess


def run_seganymo(*args):
    return subprocess.run(
        [
            "bash", "-c",
            'set -euo pipefail\n'
            'source /content/seganymo-env.sh\n'
            'cd "$SEGANYMO_REPO"\n'
            'exec python core/utils/run_inference.py "$@"',
            "seganymo",
            *map(str, args),
        ],
        check=True,
    )

print("SegAnyMo command helper ready. It sources the environment for each call.")


In [ ]:
# --- Video Processing Parameters ---
TARGET_SIZE = 640  # Resolution for processed video (e.g., 384 for 384x384)
VIDEO_PROCESS_FPS = None # Set to None to use original video FPS, or an integer to resample (e.g., 15)
SAMPLE_FPS = None

# --- Trajectory Filtering Parameters ---
# threashold on a trajectory's spatial extent
MIN_TRAVEL = 30  # Keep at 30 to filter out static noise

# threshold on mean tracker confidence, BootsTAPIR outputs a confidence score in [0,1] for everypoint
# at everyframe.
MIN_CONF   = 0.7  # Keep at 0.7, all points pass this currently

# Visibility fraction
MIN_VIS    = 0.04  # Further reduced from 0.2 to 0.04 to allow more tracks to pass

#Length of visible trail behind
TRAIL_LENGTH = 10

#spatial density criterion
NEIGH_R    = 80
MIN_NEIGH  = 2

In [ ]:
def _setup_chunk_globals(i, start_sec, TOTAL_VIDEO_DURATION, CHUNK_DURATION_IN_SECONDS, input_video):
    global START_SECONDS, DURATION_SECONDS, SEQUENCE_NAME, WORK_DIR, TEST_VIDEO

    START_SECONDS = start_sec
    DURATION_SECONDS = min(CHUNK_DURATION_IN_SECONDS, TOTAL_VIDEO_DURATION - start_sec)
    SEQUENCE_NAME = f"Chunk_{i+1}_s{START_SECONDS:.2f}s_d{DURATION_SECONDS:.2f}s"
    WORK_DIR = Path("/content/seganymo_runs") / SEQUENCE_NAME
    TEST_VIDEO = WORK_DIR / f"{SEQUENCE_NAME}.mp4"

    WORK_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Chunk parameters: Start={START_SECONDS:.2f}s, Duration={DURATION_SECONDS:.2f}s, SeqName={SEQUENCE_NAME}")

In [ ]:
def _clip_video_for_chunk(input_video, WORK_DIR, TEST_VIDEO, START_SECONDS, DURATION_SECONDS, SAMPLE_FPS, TARGET_SIZE):
    print("Clipping video for chunk...")
    params = {
        "video": input_video.name,
        "start": START_SECONDS,
        "duration": DURATION_SECONDS,
        "fps": SAMPLE_FPS,
        "size": TARGET_SIZE,
        "preproc": {"efficient": False, "step": 10},
    }

    params_file = WORK_DIR / "clip_params.json"

    if not (TEST_VIDEO.exists() and params_file.exists() and json.loads(params_file.read_text()) == params):
        filters = []
        if SAMPLE_FPS is not None:
            filters.append(f"fps={SAMPLE_FPS}")
        filters.append(f"scale={TARGET_SIZE}:{TARGET_SIZE}")

        command = [
            "ffmpeg",
            "-y",
            "-ss", str(START_SECONDS),
            "-i", str(input_video),
        ]

        if DURATION_SECONDS is not None:
            command += ["-t", str(DURATION_SECONDS)]

        command += [
            "-vf", ",".join(filters),
            "-an",
            "-c:v", "libx264",
            "-preset", "fast",
            "-crf", "18",
            "-pix_fmt", "yuv420p",
            str(TEST_VIDEO),
        ]
        subprocess.run(command, check=True)
        params_file.write_text(json.dumps(params))
        print(f"Clipped video saved to: {TEST_VIDEO}")
    else:
        print("Clip already cut with these settings — keeping existing outputs.")

In [ ]:
def _clear_stale_outputs(WORK_DIR, SEQUENCE_NAME):
    generated_folders_chunk = [
        WORK_DIR / "images" / SEQUENCE_NAME,
        WORK_DIR / "resize_images" / SEQUENCE_NAME,
        WORK_DIR / "bootstapir" / SEQUENCE_NAME,
        WORK_DIR / "depth_anything_v2" / SEQUENCE_NAME,
        WORK_DIR / "dinos" / SEQUENCE_NAME,
        WORK_DIR / "results" / "moseg" / SEQUENCE_NAME,
    ]

    for folder in generated_folders_chunk:
        if folder.exists():
            shutil.rmtree(folder)
            print(f"Removed stale outputs for chunk: {folder}")

In [ ]:
def _run_preprocessing(TEST_VIDEO, WORK_DIR, SEQUENCE_NAME):
    print("Running preprocessing for chunk...")
    run_seganymo(
        "--video_path", TEST_VIDEO,
        "--gpus", 0,
        "--depths", "--tracks", "--dinos", "--e",
    )

    src = WORK_DIR / "images" / SEQUENCE_NAME
    dst = WORK_DIR / "resize_images" / SEQUENCE_NAME

    if src.exists() and not dst.exists():
        shutil.copytree(src, dst)
        print("Mirrored images -> resize_images for chunk")

In [ ]:
def _check_preprocessing_completion(WORK_DIR, SEQUENCE_NAME, frames_current_chunk, OUTPUT_FPS):
    outputs_chunk = {
        "Frames": WORK_DIR / "resize_images" / SEQUENCE_NAME,
        "Depth maps": WORK_DIR / "depth_anything_v2" / SEQUENCE_NAME,
        "DINO features": WORK_DIR / "dinos" / SEQUENCE_NAME,
        "Point-track files": WORK_DIR / "bootstapir" / SEQUENCE_NAME,
    }

    counts_chunk = {}
    for name, folder in outputs_chunk.items():
        counts_chunk[name] = len(list(folder.glob("*"))) if folder.exists() else 0

    # Adjust expected DINO features: `run_inference.py` seems to produce
    # `frames_current_chunk // 10` DINO features (for a step of 10) for N frames,
    # rather than `math.ceil(frames_current_chunk / 10)`.
    # Ensure at least 1 DINO feature if frames exist.
    expected_dinos_current_chunk = max(1, frames_current_chunk // 10)

    if (frames_current_chunk == 0 or
        counts_chunk["Depth maps"] < frames_current_chunk or
        counts_chunk["DINO features"] < expected_dinos_current_chunk or
        counts_chunk["Point-track files"] == 0):
        raise RuntimeError("Preprocessing incomplete for chunk!")
    print("Preprocessing complete for chunk.")

In [ ]:
def _run_model_inference(WORK_DIR, SEQUENCE_NAME):
    print("Running model inference for chunk...")
    run_seganymo(
        "--data_dir", WORK_DIR / "resize_images",
        "--motin_seg_dir", WORK_DIR / "results" / "moseg",
        "--config_file", "/content/SegAnyMo/configs/fish_colab.yaml",
        "--gpus", 0,
        "--motion_seg_infer",
    )
    print("Model inference complete for chunk.")

In [ ]:
def _load_and_filter_tap_results(WORK_DIR, SEQUENCE_NAME, MIN_TRAVEL, MIN_CONF, MIN_VIS):
    moseg_chunk = WORK_DIR / "results" / "moseg" / SEQUENCE_NAME

    traj_chunk_raw = np.load(moseg_chunk / "dynamic_traj.npy")
    vis_raw_chunk = np.load(moseg_chunk / "dynamic_visibility.npy")
    conf_chunk_raw = np.load(moseg_chunk / "dynamic_confidences.npy")

    vis_chunk = vis_raw_chunk if vis_raw_chunk.dtype == bool else vis_raw_chunk >= 0.5

    x_chunk = traj_chunk_raw[0]
    y_chunk = traj_chunk_raw[1]

    x_visible_chunk = np.where(vis_chunk, x_chunk, np.nan)
    y_visible_chunk = np.where(vis_chunk, y_chunk, np.nan)
    conf_visible_chunk = np.where(vis_chunk, conf_chunk_raw, np.nan)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        x_travel_chunk = np.nanmax(x_visible_chunk, axis=1) - np.nanmin(x_visible_chunk, axis=1)
        y_travel_chunk = np.nanmax(y_visible_chunk, axis=1) - np.nanmin(y_visible_chunk, axis=1)
        travel_chunk = x_travel_chunk + y_travel_chunk
        conf_mean_chunk = np.nanmean(conf_visible_chunk, axis=1)

    travel_chunk = np.nan_to_num(travel_chunk, nan=0.0)
    conf_mean_chunk = np.nan_to_num(conf_mean_chunk, nan=0.0)
    vis_frac_chunk = vis_chunk.mean(axis=1)

    moved_enough_chunk = travel_chunk >= MIN_TRAVEL
    confident_chunk = conf_mean_chunk >= MIN_CONF
    visible_enough_chunk = vis_frac_chunk >= MIN_VIS

    keep_chunk = moved_enough_chunk & confident_chunk & visible_enough_chunk

    traj_k_chunk = traj_chunk_raw[:, keep_chunk, :]
    vis_k_chunk = vis_chunk[keep_chunk, :]
    conf_k_chunk = conf_chunk_raw[keep_chunk, :]

    valid_k_chunk = (
        vis_k_chunk
        & np.isfinite(traj_k_chunk[0])
        & np.isfinite(traj_k_chunk[1])
    )
    print(f"Filtering done. Kept points: {keep_chunk.sum()}")

    return traj_k_chunk, valid_k_chunk, conf_k_chunk

In [ ]:
def _prepare_for_rendering(WORK_DIR, SEQUENCE_NAME, traj_k_chunk, width_chunk, height_chunk):
    frame_folder_chunk = WORK_DIR / "resize_images" / SEQUENCE_NAME
    frame_paths_chunk = sorted(frame_folder_chunk.glob("*.png"))

    if len(frame_paths_chunk) == 0:
        print(f"No PNG frames found for chunk {SEQUENCE_NAME}")
        return [], []

    number_of_tracks_chunk = traj_k_chunk.shape[1]

    if number_of_tracks_chunk > 0:
        key_chunk = (
            traj_k_chunk[0, :, 0] / width_chunk * 0.5
            + traj_k_chunk[1, :, 0] / height_chunk * 0.5
        )

        key_chunk = np.nan_to_num(
            np.clip(key_chunk, 0, 1),
            nan=0.0
        )

        color_map_chunk = cv2.applyColorMap(
            (key_chunk * 255).astype(np.uint8).reshape(-1, 1),
            cv2.COLORMAP_HSV
        )

        colors_chunk = [
            tuple(int(value) for value in color_map_chunk[idx, 0])
            for idx in range(number_of_tracks_chunk)
        ]
    else:
        colors_chunk = []

    return frame_paths_chunk, colors_chunk

In [ ]:
def _create_chunk_dataframe(traj_k_chunk, vis_k_chunk, conf_k_chunk, OUTPUT_FPS, START_SECONDS):
    if traj_k_chunk.shape[1] == 0:
        return pd.DataFrame() # Return empty DataFrame if no tracks

    N_chunk, T_chunk = traj_k_chunk.shape[1], traj_k_chunk.shape[2]

    df_chunk = pd.DataFrame({
        "track_id": np.repeat(np.arange(N_chunk), T_chunk),
        "frame":    np.tile(np.arange(T_chunk), N_chunk),
        "x":        traj_k_chunk[0].reshape(-1),
        "y":        traj_k_chunk[1].reshape(-1),
        "visible":  vis_k_chunk.reshape(-1),
        "conf":     conf_k_chunk.reshape(-1),
    })
    df_chunk["t_sec"] = df_chunk["frame"] / OUTPUT_FPS + START_SECONDS
    df_chunk.loc[~df_chunk["visible"], ["x", "y"]] = np.nan

    df_chunk = df_chunk.sort_values(["track_id", "frame"]).reset_index(drop=True)
    g_chunk = df_chunk.groupby("track_id", sort=False)

    df_chunk["dx"] = g_chunk["x"].diff()
    df_chunk["dy"] = g_chunk["y"].diff()
    df_chunk["step_px"] = np.hypot(df_chunk["dx"], df_chunk["dy"])
    df_chunk["speed"]   = df_chunk["step_px"] * OUTPUT_FPS
    df_chunk["accel"]    = g_chunk["speed"].diff() * OUTPUT_FPS

    df_chunk["heading"] = np.arctan2(df_chunk["dy"], df_chunk["dx"])
    df_chunk.loc[df_chunk["step_px"] < 0.5, "heading"] = np.nan
    dh_chunk = g_chunk["heading"].diff()
    df_chunk["turn"] = (dh_chunk + np.pi) % (2 * np.pi) - np.pi
    print("DataFrame created.")

    return df_chunk

In [ ]:
def render_dots(traj_k_chunk, valid_k_chunk, colors_chunk, frame_paths_chunk, output_fps_chunk, sequence_name_chunk, trail=10, label="tracks"):
    """
    trail = 0  gives dots only
    trail = 10 gives a 10-frame tail
    trail = -1 gives the full trajectory
    """

    output_path = Path(
        f"/content/{sequence_name_chunk}_{label}.mp4"
    )

    writer = iio.get_writer(
        str(output_path),
        fps=output_fps_chunk,
        macro_block_size=1
    )

    number_of_frames = min(
        traj_k_chunk.shape[2],
        len(frame_paths_chunk)
    )
    number_of_tracks = traj_k_chunk.shape[1]

    try:
        for t in range(number_of_frames):

            frame = cv2.imread(
                str(frame_paths_chunk[t])
            )

            if frame is None:
                print("Could not read:", frame_paths_chunk[t])
                continue

            # Darken the original image
            canvas = (
                frame.astype(np.float32) * 0.8
            ).astype(np.uint8)

            if trail == -1:
                first_trail_frame = 0
            else:
                first_trail_frame = max(
                    0,
                    t - trail
                )

            for point_index in range(number_of_tracks):

                # The point must be visible at the current frame
                if not valid_k_chunk[point_index, t]:
                    continue

                color = colors_chunk[point_index]

                # Draw the trail
                for k in range(first_trail_frame, t):

                    x1_value = traj_k_chunk[0, point_index, k]
                    y1_value = traj_k_chunk[1, point_index, k]

                    x2_value = traj_k_chunk[0, point_index, k + 1]
                    y2_value = traj_k_chunk[1, point_index, k + 1]

                    # Avoid invalid coordinates
                    if not (
                        np.isfinite(x1_value)
                        and np.isfinite(y1_value)
                        and np.isfinite(x2_value)
                        and np.isfinite(y2_value)
                    ):
                        continue

                    x1 = int(x1_value)
                    y1 = int(y1_value)

                    x2 = int(x2_value)
                    y2 = int(y2_value)

                    # Older parts of the tail are darker
                    fade = (
                        (k - first_trail_frame + 1)
                        / max(t - first_trail_frame, 1)
                    )

                    faded_color = tuple(
                        int(value * (0.15 + 0.85 * fade))
                        for value in color
                    )

                    cv2.line(
                        canvas,
                        (x1, y1),
                        (x2, y2),
                        faded_color,
                        2,
                        cv2.LINE_AA
                    )

                # Draw the current point
                current_x = int(
                    traj_k_chunk[0, point_index, t]
                )

                current_y = int(
                    traj_k_chunk[1, point_index, t]
                )

                cv2.circle(
                    canvas,
                    (current_x, current_y),
                    3,
                    color,
                    -1,
                    cv2.LINE_AA
                )

            # Add one completed image to the output video
            canvas_rgb = cv2.cvtColor(
                canvas,
                cv2.COLOR_BGR2RGB
            )

            writer.append_data(canvas_rgb)

    finally:
        writer.close()

    print("Video saved to:", output_path)

    return output_path

In [ ]:
# A filename from seganymo/videos/ on Drive, or None to upload.
VIDEO_NAME = 'trimmed_fish2.mp4'

drive_videos = sorted((DRIVE_DIR / "videos").glob("*"))
print("Videos on Drive:", [v.name for v in drive_videos] or "none yet")

input_folder = Path("/content/private_input")
input_folder.mkdir(exist_ok=True)

if VIDEO_NAME is not None:
    local_current_dir_video = Path("/content") / VIDEO_NAME
    drive_video_path = DRIVE_DIR / "videos" / VIDEO_NAME
    input_video = input_folder / VIDEO_NAME

    if local_current_dir_video.exists():
        # Video is in /content/, move it to input_folder and copy to Drive for persistence
        if not input_video.exists():
            shutil.move(local_current_dir_video, input_video)
        if not drive_video_path.exists():
            shutil.copy(input_video, drive_video_path)
        print("Loaded from Colab session root and saved to Drive:", input_video)
    elif drive_video_path.exists():
        # Video is on Drive, copy it to input_folder
        if not input_video.exists():
            shutil.copy(drive_video_path, input_video)
        print("Loaded from Drive:", input_video)
    else:
        # Video not found locally or on Drive, prompt for upload
        print(f"Video '{VIDEO_NAME}' not found in /content/ or {DRIVE_DIR / 'videos'}.")
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        input_video = input_folder / uploaded_name
        shutil.move(str(Path("/content") / uploaded_name), str(input_video))
        uploaded.clear()
        shutil.copy(input_video, drive_video_path)
        print("Saved to Drive for future sessions:", drive_video_path)
elif VIDEO_NAME is None: # VIDEO_NAME is None, prompt for upload
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    input_video = input_folder / uploaded_name
    shutil.move(str(Path("/content") / uploaded_name), str(input_video))
    uploaded.clear()
    shutil.copy(input_video, DRIVE_DIR / "videos" / uploaded_name)
    print("Saved to Drive for future sessions:",
          DRIVE_DIR / "videos" / uploaded_name)

cap = cv2.VideoCapture(str(input_video))
ORIGINAL_VIDEO_FPS = cap.get(cv2.CAP_PROP_FPS)
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print("Video:", input_video)
print("Resolution:", width, "x", height)
print("Frame rate:", round(ORIGINAL_VIDEO_FPS, 2))
print("Frames:", frames)
print("Duration:", round(frames / ORIGINAL_VIDEO_FPS, 2), "seconds")

TOTAL_VIDEO_DURATION = frames / ORIGINAL_VIDEO_FPS

display(Video(str(input_video), width=600, embed=True))

In [ ]:
# Define video properties for chunking
# Use a 100-frame chunk size.
FRAME_CHUNK_SIZE = 100

# Determine the effective FPS for chunk duration calculation.
# If VIDEO_PROCESS_FPS is set, use it; otherwise, use the original video's FPS.
effective_fps_for_chunk_duration = VIDEO_PROCESS_FPS if VIDEO_PROCESS_FPS is not None else ORIGINAL_VIDEO_FPS

# Calculate the duration in seconds for a 100-frame chunk based on the effective FPS.
CHUNK_DURATION_IN_SECONDS = FRAME_CHUNK_SIZE / effective_fps_for_chunk_duration

# Calculate number of chunks and their start times
num_chunks = int(TOTAL_VIDEO_DURATION // CHUNK_DURATION_IN_SECONDS) + (1 if TOTAL_VIDEO_DURATION % CHUNK_DURATION_IN_SECONDS != 0 else 0)
chunk_starts = [i * CHUNK_DURATION_IN_SECONDS for i in range(num_chunks)]

all_dfs = []
output_video_paths = []

print(f"Total video duration: {TOTAL_VIDEO_DURATION} seconds")
print(f"Original Video FPS: {ORIGINAL_VIDEO_FPS:.2f}")
print(f"Processing FPS (for chunking logic): {effective_fps_for_chunk_duration:.2f}")
print(f"Frame chunk size: {FRAME_CHUNK_SIZE} frames")
print(f"Calculated chunk duration: {CHUNK_DURATION_IN_SECONDS:.2f} seconds")
print(f"Number of chunks: {num_chunks}")
print(f"Chunk start times: {chunk_starts}")

In [ ]:
for i, start_sec in enumerate(chunk_starts):
    print(f"\n--- Processing Chunk {i+1}/{num_chunks} (Start: {start_sec:.2f}s) ---")

    # Update global parameters for the current chunk
    globals()['START_SECONDS'] = start_sec
    globals()['DURATION_SECONDS'] = min(CHUNK_DURATION_IN_SECONDS, TOTAL_VIDEO_DURATION - start_sec)
    globals()['SEQUENCE_NAME'] = f"Chunk_{i+1}_s{start_sec:.2f}s_d{globals()['DURATION_SECONDS']:.2f}s"
    globals()['WORK_DIR'] = Path("/content/seganymo_runs") / globals()['SEQUENCE_NAME']
    globals()['TEST_VIDEO'] = globals()['WORK_DIR'] / f"{globals()['SEQUENCE_NAME']}.mp4"

    # Create WORK_DIR for the chunk
    globals()['WORK_DIR'].mkdir(parents=True, exist_ok=True)

    # Re-run video clipping (based on cell 0qV6CwlWIEYH)
    print("Clipping video for chunk...")
    params = {
        "video": input_video.name,
        "start": globals()['START_SECONDS'],
        "duration": globals()['DURATION_SECONDS'],
        "fps": SAMPLE_FPS,
        "size": TARGET_SIZE,
        "preproc": {"efficient": False, "step": 10},
    }

    params_file = globals()['WORK_DIR'] / "clip_params.json"

    # Only clip if params change or video doesn't exist
    if not (globals()['TEST_VIDEO'].exists() and params_file.exists() and json.loads(params_file.read_text()) == params):
        filters = []
        if SAMPLE_FPS is not None:
            filters.append(f"fps={SAMPLE_FPS}")
        filters.append(f"scale={TARGET_SIZE}:{TARGET_SIZE}")

        command = [
            "ffmpeg",
            "-y",
            "-ss", str(globals()['START_SECONDS']),
            "-i", str(input_video),
        ]

        if globals()['DURATION_SECONDS'] is not None:
            command += ["-t", str(globals()['DURATION_SECONDS'])]

        command += [
            "-vf", ",".join(filters),
            "-an",
            "-c:v", "libx264",
            "-preset", "fast",
            "-crf", "18",
            "-pix_fmt", "yuv420p",
            str(globals()['TEST_VIDEO']),
        ]
        subprocess.run(command, check=True)
        params_file.write_text(json.dumps(params))
    else:
        print("Clip already cut with these settings — keeping existing outputs.")

    cap = cv2.VideoCapture(str(globals()['TEST_VIDEO']))
    globals()['OUTPUT_FPS'] = cap.get(cv2.CAP_PROP_FPS) # Update OUTPUT_FPS for the chunk
    frames_in_chunk = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width_chunk = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height_chunk = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    print(f"Chunk video: {globals()['TEST_VIDEO']}")
    print(f"Chunk resolution: {width_chunk} x {height_chunk}")
    print(f"Chunk frame rate: {round(globals()['OUTPUT_FPS'], 2)}")
    print(f"Frames in chunk: {frames_in_chunk}")
    print(f"Chunk duration: {round(frames_in_chunk / globals()['OUTPUT_FPS'], 2)} seconds")

    # Clear stale outputs for the current chunk (based on cell 5YqIE9FDIGVL)
    generated_folders_chunk = [
        globals()['WORK_DIR'] / "images" / globals()['SEQUENCE_NAME'],
        globals()['WORK_DIR'] / "resize_images" / globals()['SEQUENCE_NAME'],
        globals()['WORK_DIR'] / "bootstapir" / globals()['SEQUENCE_NAME'],
        globals()['WORK_DIR'] / "depth_anything_v2" / globals()['SEQUENCE_NAME'],
        globals()['WORK_DIR'] / "dinos" / globals()['SEQUENCE_NAME'],
        globals()['WORK_DIR'] / "results" / "moseg" / globals()['SEQUENCE_NAME'],
    ]

    for folder in generated_folders_chunk:
        if folder.exists():
            shutil.rmtree(folder)
            print(f"Removed stale outputs for chunk: {folder}")

    # Preprocessing (based on cell VPw6FdusIIdP and MCBGvs9YIKIM)
    print("Running preprocessing for chunk...")
    run_seganymo(
        "--video_path", TEST_VIDEO,
        "--gpus", 0,
        "--depths", "--tracks", "--dinos", "--e",
    )

    src = globals()['WORK_DIR'] / "images" / globals()['SEQUENCE_NAME']
    dst = globals()['WORK_DIR'] / "resize_images" / globals()['SEQUENCE_NAME']

    if src.exists() and not dst.exists():
        shutil.copytree(src, dst)
        print("Mirrored images -> resize_images for chunk")

    # Check preprocessing completion
    outputs_chunk = {
        "Frames": globals()['WORK_DIR'] / "resize_images" / globals()['SEQUENCE_NAME'],
        "Depth maps": globals()['WORK_DIR'] / "depth_anything_v2" / globals()['SEQUENCE_NAME'],
        "DINO features": globals()['WORK_DIR'] / "dinos" / globals()['SEQUENCE_NAME'],
        "Point-track files": globals()['WORK_DIR'] / "bootstapir" / globals()['SEQUENCE_NAME'],
    }

    counts_chunk = {}
    for name, folder in outputs_chunk.items():
        counts_chunk[name] = len(list(folder.glob("*"))) if folder.exists() else 0

    frames_current_chunk = counts_chunk["Frames"]
    expected_dinos_current_chunk = math.ceil(frames_current_chunk / 10)

    if (frames_current_chunk == 0 or
        counts_chunk["Depth maps"] < frames_current_chunk or
        counts_chunk["DINO features"] < expected_dinos_current_chunk or
        counts_chunk["Point-track files"] == 0):
        raise RuntimeError(f"Preprocessing incomplete for chunk {i+1}!")
    print("Preprocessing complete for chunk.")

    # Model inference (based on cell GSSadU57INjA)
    print("Running model inference for chunk...")
    run_seganymo(
        "--data_dir", WORK_DIR / "resize_images",
        "--motin_seg_dir", WORK_DIR / "results" / "moseg",
        "--config_file", "/content/SegAnyMo/configs/fish_colab.yaml",
        "--gpus", 0,
        "--motion_seg_infer",
    )
    print("Model inference complete for chunk.")

    # Load TAP Results (based on cell LD2id927IQ8o)
    moseg_chunk = globals()['WORK_DIR'] / "results" / "moseg" / globals()['SEQUENCE_NAME']

    traj_chunk_raw = np.load(moseg_chunk / "dynamic_traj.npy")
    vis_raw_chunk = np.load(moseg_chunk / "dynamic_visibility.npy")
    conf_chunk_raw = np.load(moseg_chunk / "dynamic_confidences.npy")

    if vis_raw_chunk.dtype == bool:
      vis_chunk = vis_raw_chunk
    else:
      vis_chunk = vis_raw_chunk >= 0.5

    # Calculate Trajectory scores and filter (based on cell XxIL1WxvISV9)
    x_chunk = traj_chunk_raw[0]
    y_chunk = traj_chunk_raw[1]

    x_visible_chunk = np.where(vis_chunk, x_chunk, np.nan)
    y_visible_chunk = np.where(vis_chunk, y_chunk, np.nan)
    conf_visible_chunk = np.where(vis_chunk, conf_chunk_raw, np.nan)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        x_travel_chunk = np.nanmax(x_visible_chunk, axis=1) - np.nanmin(x_visible_chunk, axis=1)
        y_travel_chunk = np.nanmax(y_visible_chunk, axis=1) - np.nanmin(y_visible_chunk, axis=1)
        travel_chunk = x_travel_chunk + y_travel_chunk
        conf_mean_chunk = np.nanmean(conf_visible_chunk, axis=1)

    travel_chunk = np.nan_to_num(travel_chunk, nan=0.0)
    conf_mean_chunk = np.nan_to_num(conf_mean_chunk, nan=0.0)
    vis_frac_chunk = vis_chunk.mean(axis=1)

    moved_enough_chunk = travel_chunk >= MIN_TRAVEL
    confident_chunk = conf_mean_chunk >= MIN_CONF
    visible_enough_chunk = vis_frac_chunk >= MIN_VIS

    keep_chunk = moved_enough_chunk & confident_chunk & visible_enough_chunk

    traj_k_chunk = traj_chunk_raw[:, keep_chunk, :]
    vis_k_chunk = vis_chunk[keep_chunk, :]
    conf_k_chunk = conf_chunk_raw[keep_chunk, :]

    valid_k_chunk = (
        vis_k_chunk
        & np.isfinite(traj_k_chunk[0])
        & np.isfinite(traj_k_chunk[1])
    )
    print(f"Filtering for chunk {i+1} done. Kept points: {keep_chunk.sum()}")

    # Find Video Frames (based on cell jbbaUYjBIUjI)
    frame_folder_chunk = globals()['WORK_DIR'] / "resize_images" / globals()['SEQUENCE_NAME']
    frame_paths_chunk = sorted(frame_folder_chunk.glob("*.png"))

    if len(frame_paths_chunk) == 0:
        print(f"No PNG frames found for chunk {i+1}")
        continue

    first_frame_chunk = cv2.imread(str(frame_paths_chunk[0]))
    height_chunk, width_chunk = first_frame_chunk.shape[:2]

    # Color assignment for each trajectory
    number_of_tracks_chunk = traj_k_chunk.shape[1]

    if number_of_tracks_chunk > 0:
        key_chunk = (
            traj_k_chunk[0, :, 0] / width_chunk * 0.5
            + traj_k_chunk[1, :, 0] / height_chunk * 0.5
        )

        key_chunk = np.nan_to_num(
            np.clip(key_chunk, 0, 1),
            nan=0.0
        )

        color_map_chunk = cv2.applyColorMap(
            (key_chunk * 255).astype(np.uint8).reshape(-1, 1),
            cv2.COLORMAP_HSV
        )

        colors_chunk = [
            tuple(int(value) for value in color_map_chunk[idx, 0])
            for idx in range(number_of_tracks_chunk)
        ]
    else:
        colors_chunk = []

    # Render dots for the chunk using the modified function
    print(f"Rendering video for chunk {i+1}...")
    output_video_chunk_path = render_dots(
        traj_k_chunk=traj_k_chunk,
        valid_k_chunk=valid_k_chunk,
        colors_chunk=colors_chunk,
        frame_paths_chunk=frame_paths_chunk,
        output_fps_chunk=globals()['OUTPUT_FPS'],
        sequence_name_chunk=globals()['SEQUENCE_NAME'],
        trail=TRAIL_LENGTH,
        label="filtered_tracks"
    )
    output_video_paths.append(output_video_chunk_path)

    # Create DataFrame for the chunk (based on cells ZeZ6eanzIdtU and tD89OGXOIfPx)
    print(f"Creating DataFrame for chunk {i+1}...")
    N_chunk, T_chunk = traj_k_chunk.shape[1], traj_k_chunk.shape[2]

    df_chunk = pd.DataFrame({
        "track_id": np.repeat(np.arange(N_chunk), T_chunk),
        "frame":    np.tile(np.arange(T_chunk), N_chunk),
        "x":        traj_k_chunk[0].reshape(-1),
        "y":        traj_k_chunk[1].reshape(-1),
        "visible":  vis_k_chunk.reshape(-1),
        "conf":     conf_k_chunk.reshape(-1),
    })
    df_chunk["t_sec"] = df_chunk["frame"] / globals()['OUTPUT_FPS'] + globals()['START_SECONDS'] # Adjust t_sec for chunk start
    df_chunk.loc[~df_chunk["visible"], ["x", "y"]] = np.nan

    df_chunk = df_chunk.sort_values(["track_id", "frame"]).reset_index(drop=True)
    g_chunk = df_chunk.groupby("track_id", sort=False)

    df_chunk["dx"] = g_chunk["x"].diff()
    df_chunk["dy"] = g_chunk["y"].diff()
    df_chunk["step_px"] = np.hypot(df_chunk["dx"], df_chunk["dy"])
    df_chunk["speed"]   = df_chunk["step_px"] * globals()['OUTPUT_FPS']
    df_chunk["accel"]    = g_chunk["speed"].diff() * globals()['OUTPUT_FPS']

    df_chunk["heading"] = np.arctan2(df_chunk["dy"], df_chunk["dx"])
    df_chunk.loc[df_chunk["step_px"] < 0.5, "heading"] = np.nan
    dh_chunk = g_chunk["heading"].diff()
    df_chunk["turn"] = (dh_chunk + np.pi) % (2 * np.pi) - np.pi

    all_dfs.append(df_chunk)
    print(f"DataFrame for chunk {i+1} created.")

In [ ]:
# Merge all dataframes
print("\n--- Merging DataFrames ---")
final_df = pd.concat(all_dfs, ignore_index=True)
print(f"Final DataFrame shape: {final_df.shape}")

# Display the head of the final DataFrame
display(final_df.head())

# User requested to keep videos separate, so skipping concatenation.
print("\n--- Individual Output Videos ---")
print("Paths to individual chunk videos:")
for p in output_video_paths:
    print(f"- {p}")

# Display the first chunk video as an example, or all of them if desired
if output_video_paths:
    print("\nDisplaying the first chunk video:")
    display(Video(str(output_video_paths[0]), width=600, embed=True))
else:
    print("No chunk videos were generated.")

In [ ]:
video_list_file = "/content/video_list.txt"
with open(video_list_file, "w") as f:
    for video_path in output_video_paths:
        f.write(f"file '{video_path}'\n")

print(f"Video list written to: {video_list_file}")

In [ ]:
final_output_video = "/content/Full_Video_filtered_tracks.mp4"

ffmpeg_concat_command = [
    "ffmpeg",
    "-y", # Overwrite output files without asking
    "-f", "concat", # Use the concat demuxer
    "-safe", "0", # Allow unsafe file paths (necessary for file lists)
    "-i", video_list_file, # Input file list
    "-c", "copy", # Copy streams directly (no re-encoding)
    final_output_video,
]

print(f"Executing FFmpeg command: {' '.join(ffmpeg_concat_command)}")
subprocess.run(ffmpeg_concat_command, check=True)

print(f"All videos merged into: {final_output_video}")
display(Video(str(final_output_video), width=600, embed=True))

In [ ]:
import pandas as pd
from google.colab import files

if "all_dfs" not in globals() or not all_dfs:
    raise RuntimeError("No results yet. Finish chunk processing first.")

# Keep each chunk's tracks distinguishable.
df_final = pd.concat(
    [df.assign(chunk_id=i) for i, df in enumerate(all_dfs, start=1)],
    ignore_index=True,
)

output_path = "/content/seganymo_tracks.csv"
df_final.to_csv(output_path, index=False)

print(f"Exported {len(df_final):,} rows from {len(all_dfs)} chunk tables.")
files.download(output_path)